# Automatic math-paper audit maintenance notebook

> **Frontend status:** The PySide6 GUI is now the primary maintained user-facing interface.
>
> **Source of truth:** Shared backend modules (`audit_runtime.py`, `audit_policy_hooks.py`, `audit_state.py`, `audit_verification.py`, `audit_chunking.py`, `audit_prompts.py`) are canonical.
>
> **Notebook role:** This notebook is a secondary developer / maintenance / experimentation frontend for state inspection, report rebuilds, verification helpers, recovery/debugging, and one-off analysis. New user-facing features do **not** need notebook UI parity.
>
> **Important:** Historical production cells below are archived as non-executing reference material so they cannot shadow backend implementations.

Use the active bootstrap cell below, then call backend helpers directly.

In [1]:
# Uncomment if needed:
# %pip install -U openai pymupdf pypdf

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Optional

from IPython.display import Markdown, display
from openai import OpenAI

from audit_hooks import install_runtime_frontend
from audit_policy_hooks import build_final_report as policy_build_final_report, build_user_message_for_chunk
from audit_prompts import (
    SHIPPED_AUDIT_SYSTEM_PROMPT,
    effective_audit_system_prompt,
    effective_audit_system_prompt_with_source,
    load_prompt_profiles,
    save_prompt_profiles,
)
from audit_runtime import (
    ask_about_audit,
    ask_about_paper,
    build_concise_report,
    build_final_report,
    build_verification_report,
    cancel_pending_response_for_retry,
    get_audit_status,
    list_qa_threads,
    load_qa_turns,
    rebuild_qa_report,
    request_pause,
    resume_audit,
    run_verification_suite_and_build_report,
    set_active_qa_thread,
    start_fresh_audit,
    start_new_qa_thread,
)
from audit_state import (
    load_issues,
    load_json,
    load_ledger,
    load_manifest,
    load_session_from_pdf,
    load_status,
    load_usage,
    session_paths,
)
from audit_verification import (
    load_verification_state,
    rerun_failed_verification_scripts,
    run_verification_suite,
    show_verification_status,
)

# Optional live API client for notebook maintenance work. Status/report inspection does not require it.
client = OpenAI() if os.environ.get("OPENAI_API_KEY") else None

install_runtime_frontend(
    globals(),
    client=client,
    prompt_builder=build_user_message_for_chunk,
    final_report_builder=policy_build_final_report,
    default_noop_display=True,
)

print("Notebook maintenance bootstrap loaded. GUI is primary; backend modules are canonical.")

### Archived legacy notebook cell 04 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_prompts.py and audit_runtime.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
DEFAULT_MODEL = "gpt-5.4"
DEFAULT_REASONING_EFFORT = "high"
MODEL_CHOICES = ["gpt-5.5", "gpt-5.5-pro", "gpt-5.4", "gpt-5.4-mini", "gpt-5.2"]
MODEL_REASONING_EFFORTS = {
    "gpt-5.5": ["none", "low", "medium", "high", "xhigh"],
    "gpt-5.5-pro": ["high"],
}

# Standard-tier prices in USD per 1M tokens.
# Update these if OpenAI pricing changes.
PRICING_USD_PER_1M = {
    "gpt-5.4":      {"input": 2.50,  "cached_input": 0.25,  "output": 15.00},
    "gpt-5.4-pro":  {"input": 30.00, "cached_input": None,  "output": 180.00},
    "gpt-5":        {"input": 1.25,  "cached_input": 0.125, "output": 10.00},
    "gpt-5-mini":   {"input": 0.25,  "cached_input": 0.025, "output": 2.00},
    "gpt-5-nano":   {"input": 0.05,  "cached_input": 0.005, "output": 0.40},
}

WORKING_STATUSES = {"queued", "in_progress"}

from audit_prompts import SHIPPED_AUDIT_SYSTEM_PROMPT

AUDIT_SYSTEM_PROMPT = SHIPPED_AUDIT_SYSTEM_PROMPT

AUDIT_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string"},
        "boundary": {"type": "string"},
        "chunk_too_large": {"type": "boolean"},
        "chunk_split_suggestions": {
            "type": "array",
            "items": {"type": "string"}
        },
        "assumptions_and_notation": {
            "type": "array",
            "items": {"type": "string"}
        },
        "verified_steps": {
            "type": "array",
            "items": {"type": "string"}
        },
        "issues": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "severity": {
                        "type": "string",
                        "enum": ["low", "medium", "high", "critical"]
                    },
                    "location": {"type": "string"},
                    "description": {"type": "string"},
                    "evidence": {"type": "string"},
                    "proposed_fix": {"type": "string"},
                    "tags": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                "required": [
                    "title",
                    "severity",
                    "location",
                    "description",
                    "evidence",
                    "proposed_fix",
                    "tags"
                ],
                "additionalProperties": False
            }
        },
        "python_checks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "purpose": {"type": "string"},
                    "description": {"type": "string"},
                    "expected_outcome": {"type": "string"},
                    "code": {"type": "string"}
                },
                "required": ["purpose", "description", "expected_outcome", "code"],
                "additionalProperties": False
            }
        },
        "latex_patch": {"type": "string"},
        "ledger_updates": {
            "type": "object",
            "properties": {
                "assumptions": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "notes": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": ["assumptions", "notes"],
            "additionalProperties": False
        },
        "next_boundary_hint": {"type": "string"},
        "confidence": {
            "type": "string",
            "enum": ["low", "medium", "high"]
        }
    },
    "required": [
        "label",
        "boundary",
        "chunk_too_large",
        "chunk_split_suggestions",
        "assumptions_and_notation",
        "verified_steps",
        "issues",
        "python_checks",
        "latex_patch",
        "ledger_updates",
        "next_boundary_hint",
        "confidence"
    ],
    "additionalProperties": False
}
````

</details>

### Archived legacy notebook cell 05 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_runtime.py and audit_policy_hooks.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def to_jsonable(obj: Any) -> Any:
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    return json.loads(json.dumps(obj, default=str))

from audit_state import append_jsonl, load_json, save_json

def read_text_file(path: str | Path) -> str:
    path = Path(path)
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return path.read_text(encoding=enc)
        except Exception:
            pass
    return path.read_text(errors="ignore")

def normalize_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def strip_tex_comments(text: str) -> str:
    out = []
    for line in text.splitlines():
        line = re.sub(r"(?<!\\)%.*$", "", line)
        out.append(line)
    return "\n".join(out)

_UNSAFE_CONTROL_CHAR_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_JSON_ESCAPE_ARTIFACTS = {
    "\x08": r"\\b",
    "\x0c": r"\\f",
}

def _repair_json_escape_artifacts(text: str) -> str:
    text = "" if text is None else str(text)
    for bad, replacement in _JSON_ESCAPE_ARTIFACTS.items():
        text = text.replace(bad, replacement)
    return text

def _strip_unsafe_control_chars(text: str) -> str:
    text = "" if text is None else str(text)
    return _UNSAFE_CONTROL_CHAR_RE.sub("", text)

def normalize_math_delimiters(text: str) -> str:
    text = _strip_unsafe_control_chars(_repair_json_escape_artifacts(text))
    text = re.sub(
        r"\\\[(.*?)\\\]",
        lambda m: "$$\\n" + m.group(1).strip() + "\\n$$",
        text,
        flags=re.DOTALL,
    )
    text = re.sub(
        r"\\\((.*?)\\\)",
        lambda m: "$" + m.group(1).strip() + "$",
        text,
        flags=re.DOTALL,
    )
    return _strip_unsafe_control_chars(text)

def extract_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    decoder = json.JSONDecoder()
    try:
        obj, _ = decoder.raw_decode(text)
        return obj
    except Exception:
        pass

    first = text.find("{")
    if first >= 0:
        try:
            obj, _ = decoder.raw_decode(text[first:])
            return obj
        except Exception:
            pass

    raise ValueError("Could not parse JSON object from model output.")

def parse_audit_response(resp) -> dict[str, Any]:
    parsed = getattr(resp, "output_parsed", None)
    if isinstance(parsed, dict):
        return parsed

    raw_text = (getattr(resp, "output_text", None) or "").strip()
    if raw_text:
        return extract_json_object(raw_text)

    raw = to_jsonable(resp)
    # Last-resort scan of message content for any output_text segments.
    for item in raw.get("output", []) or []:
        if item.get("type") == "message":
            for content in item.get("content", []) or []:
                if content.get("type") == "output_text":
                    txt = (content.get("text") or "").strip()
                    if txt:
                        return extract_json_object(txt)
    raise ValueError("Could not locate a structured audit object in the model response.")

def sanitize_ascii_punctuation(text: str) -> str:
    repl = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "--",
        "\u2014": "---",
        "\u2212": "-",
        "\u2026": "...",
    }
    for k, v in repl.items():
        text = text.replace(k, v)
    return text

def escape_latex_text_preserve_math(text: str) -> str:
    text = sanitize_ascii_punctuation(text)
    parts = re.split(r"(\$\$.*?\$\$|\$.*?\$)", text, flags=re.DOTALL)
    out = []
    trans = {
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for part in parts:
        if not part:
            continue
        if (part.startswith("$$") and part.endswith("$$")) or (part.startswith("$") and part.endswith("$")):
            out.append(part)
        else:
            for k, v in trans.items():
                part = part.replace(k, v)
            out.append(part)
    return "".join(out)

def latex_paragraph(text: str) -> str:
    text = normalize_math_delimiters(text)
    text = escape_latex_text_preserve_math(text)
    return text

def format_list_for_markdown(items: list[str]) -> str:
    if not items:
        return "- None."
    return "\n".join(f"- {normalize_math_delimiters(str(x))}" for x in items)

def _normalize_python_check_entry(
    chk: dict[str, Any],
    chunk_label: str = "",
    chunk_boundary: str = "",
) -> dict[str, str]:
    purpose = str(chk.get("purpose", "") or "").strip() or "Python check"
    description = str(chk.get("description", "") or "").strip()
    if not description:
        lead = "This local verification script tests the following claim"
        if chunk_label:
            lead += f" from {chunk_label}"
        if chunk_boundary:
            lead += f" (boundary: {chunk_boundary})"
        description = (
            f"{lead}: {purpose}. "
            "It is intended as a runnable numerical or symbolic check of the argument discussed in this chunk."
        )
    expected_outcome = str(chk.get("expected_outcome", "") or "").strip()
    if not expected_outcome:
        expected_outcome = (
            "The script should finish without exceptions or failed assertions, and any printed output "
            "should be consistent with the claim described above."
        )
    code = str(chk.get("code", "") or "")
    return {
        "purpose": purpose,
        "description": description,
        "expected_outcome": expected_outcome,
        "code": code,
    }

def render_audit_markdown(audit: dict[str, Any]) -> str:
    issues_md = []
    issues = audit.get("issues", []) or []
    if not issues:
        issues_md.append("- No issues found.")
    else:
        for issue in issues:
            issues_md.append(
                "\n".join([
                    f"### {normalize_math_delimiters(issue.get('title','Untitled issue'))} [{issue.get('severity','low')}]",
                    f"- Location: {normalize_math_delimiters(issue.get('location',''))}",
                    f"- Description: {normalize_math_delimiters(issue.get('description',''))}",
                    f"- Evidence: {normalize_math_delimiters(issue.get('evidence',''))}",
                    f"- Proposed fix: {normalize_math_delimiters(issue.get('proposed_fix',''))}",
                    f"- Tags: {', '.join(issue.get('tags', [])) if issue.get('tags') else 'none'}",
                ])
            )

    py_checks = audit.get("python_checks", []) or []
    if py_checks:
        py_md = []
        for chk in py_checks:
            entry = _normalize_python_check_entry(
                chk,
                chunk_label=str(audit.get("label", "") or ""),
                chunk_boundary=str(audit.get("boundary", "") or ""),
            )
            py_md.append(
                "\n".join([
                    f"#### {normalize_math_delimiters(entry['purpose'])}",
                    normalize_math_delimiters(entry["description"]),
                    f"- Expected outcome: {normalize_math_delimiters(entry['expected_outcome'])}",
                    "```python",
                    entry["code"].strip(),
                    "```",
                ])
            )
        python_md = "\n\n".join(py_md)
    else:
        python_md = "- No Python checks suggested."

    latex_patch = (audit.get("latex_patch") or "").strip()
    latex_patch_md = "- No LaTeX patch suggested."
    if latex_patch:
        latex_patch_md = "\n".join(["```latex", latex_patch, "```"])

    split_md = ""
    if audit.get("chunk_too_large"):
        split_md = "\n".join([
            "## Chunk size",
            "- Chunk judged too large.",
            format_list_for_markdown(audit.get("chunk_split_suggestions", [])),
            "",
        ])

    md = "\n\n".join([
        f"# {normalize_math_delimiters(audit.get('label','Audit chunk'))}",
        f"**Boundary:** {normalize_math_delimiters(audit.get('boundary',''))}",
        split_md,
        "## Assumptions and notation",
        format_list_for_markdown(audit.get("assumptions_and_notation", [])),
        "## Verified steps",
        format_list_for_markdown(audit.get("verified_steps", [])),
        "## Issues found",
        "\n\n".join(issues_md),
        "## Suggested local Python checks",
        python_md,
        "## Minimal LaTeX patch",
        latex_patch_md,
        "## Ledger updates",
        "### Assumptions\n" + format_list_for_markdown((audit.get("ledger_updates") or {}).get("assumptions", [])),
        "### Notes\n" + format_list_for_markdown((audit.get("ledger_updates") or {}).get("notes", [])),
        "## Next boundary hint",
        normalize_math_delimiters(audit.get("next_boundary_hint", "None.")),
        f"**Confidence:** {audit.get('confidence', 'medium')}",
    ])
    return _strip_unsafe_control_chars(md)

def display_audit(audit: dict[str, Any]) -> None:
    display(Markdown(render_audit_markdown(audit)))
````

</details>

### Archived legacy notebook cell 06 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_chunking.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
from audit_chunking import (
    THEOREM_ENVS,
    _chunk_from_tex_substring,
    build_auto_chunks,
    find_matching_end,
    load_pdf_pages,
    parse_tex_chunks_full_coverage,
    split_large_text_unit,
    split_pdf_pages_into_chunks,
)
````

</details>

### Archived legacy notebook cell 07 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_runtime.py and audit_state.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
from audit_state import (
    ensure_workdir_tree,
    init_state_files,
    load_issues,
    load_json,
    load_ledger,
    load_manifest,
    load_session_from_pdf,
    load_status,
    load_usage,
    save_issues,
    save_json,
    save_ledger,
    save_manifest,
    save_session,
    save_status,
    save_usage,
    session_paths,
    workdir_from_pdf,
)
from audit_verification import load_verification_state, save_verification_state


def create_new_session(
    pdf_path: str | Path,
    model: str = DEFAULT_MODEL,
    reasoning_effort: str = DEFAULT_REASONING_EFFORT,
    tex_max_chars: int = 4500,
    pdf_max_chars: int = 3500,
    store: bool = True,
    background: bool = True,
) -> dict[str, Any]:
    pdf_path = Path(pdf_path).expanduser().resolve()
    if not pdf_path.exists():
        raise FileNotFoundError(pdf_path)
    tex_path = pdf_path.with_suffix(".tex")
    workdir = workdir_from_pdf(pdf_path)
    ensure_workdir_tree(workdir)
    init_state_files(workdir, model=model, reasoning_effort=reasoning_effort)

    conversation = client.conversations.create()
    with pdf_path.open("rb") as f:
        uploaded = client.files.create(file=f, purpose="user_data")

    session = {
        "created_at": utc_now(),
        "updated_at": utc_now(),
        "pdf_path": str(pdf_path),
        "tex_path": str(tex_path) if tex_path.exists() else None,
        "workdir": str(workdir),
        "model": model,
        "reasoning_effort": reasoning_effort,
        "store": store,
        "background": background,
        "conversation_id": conversation.id,
        "pdf_file_id": uploaded.id,
        "pdf_attached_in_conversation": False,
        "developer_prompt_seeded": False,
        "next_chunk_index": 1,
        "last_response_id": None,
        "pending": None,
    }

    manifest = build_auto_chunks(
        pdf_path=pdf_path,
        tex_path=tex_path if tex_path.exists() else None,
        tex_max_chars=tex_max_chars,
        pdf_max_chars=pdf_max_chars,
    )

    save_session(session)
    save_manifest(session, manifest)

    status = load_status(session)
    status.update({
        "status": "initialized",
        "chunks_total": len(manifest["chunks"]),
        "estimated_pages_total": manifest["pdf_page_count"],
        "progress_pct": 0.0,
        "current_chunk_id": None,
        "chunks_completed": 0,
        "estimated_pages_completed": 0,
    })
    save_status(session, status)

    return session
````

</details>

### Archived legacy notebook cell 08 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_runtime.py and audit_policy_hooks.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
from audit_state import compute_usage_cost, pricing_for_model
from audit_runtime import get_audit_status

def build_user_message_for_chunk(session: dict[str, Any], chunk: dict[str, Any]) -> list[dict[str, Any]]:
    content = []
    if not session.get("pdf_attached_in_conversation", False):
        content.append({"type": "input_file", "file_id": session["pdf_file_id"]})
    prompt_text = f"""Audit this mathematics-paper chunk rigorously.

Chunk label: {chunk['label']}
Boundary: {chunk['boundary']}
Source kind: {chunk['source_kind']}
Estimated page span: {chunk['page_start']}-{chunk['page_end']}

Use the provided structured output schema.
Keep mathematical prose human-readable, with inline math in $...$ and display math in $$...$$.
If you include python_checks, every item must contain:
- purpose: a short title
- description: a self-contained explanation of the claim being tested, the test strategy, and any sample parameters or cases used
- expected_outcome: what output or condition would count as success
- code: runnable local Python code
Write the description so it can stand on its own in the final report before the script body.

Chunk text:
{chunk['chunk_text']}
"""
    content.append({"type": "input_text", "text": prompt_text})
    return [{"role": "user", "content": content}]

def wait_for_response(response_id: str, poll_every: float = 3.0, max_wait_seconds: Optional[float] = None):
    start = time.time()
    while True:
        resp = client.responses.retrieve(response_id)
        if getattr(resp, "status", None) not in WORKING_STATUSES:
            return resp
        if max_wait_seconds is not None and (time.time() - start) > max_wait_seconds:
            raise TimeoutError(f"Polling exceeded {max_wait_seconds} seconds for response {response_id}")
        time.sleep(poll_every)

def extract_fenced_blocks_from_string(text: str, language: str) -> list[str]:
    pat = re.compile(rf"```{re.escape(language)}\s*\n(.*?)```", flags=re.DOTALL | re.IGNORECASE)
    return [m.strip() + "\n" for m in pat.findall(text)]

def save_patch_and_code_files(session: dict[str, Any], chunk_id: str, audit: dict[str, Any]) -> dict[str, list[str]]:
    root = Path(session["workdir"])
    latex_paths = []
    python_paths = []

    latex_patch = (audit.get("latex_patch") or "").strip()
    if latex_patch:
        p = root / "latex_patches" / f"{chunk_id}_patch_01.tex"
        p.write_text(latex_patch + "\n", encoding="utf-8")
        latex_paths.append(str(p))

    for idx, chk in enumerate(audit.get("python_checks", []) or [], start=1):
        code = (chk.get("code") or "").strip()
        if code:
            p = root / "python_checks" / f"{chunk_id}_check_{idx:02d}.py"
            p.write_text(code + "\n", encoding="utf-8")
            python_paths.append(str(p))

    return {"latex_paths": latex_paths, "python_paths": python_paths}

def update_ledger_from_audit(session: dict[str, Any], audit: dict[str, Any]) -> None:
    ledger = load_ledger(session)
    updates = audit.get("ledger_updates") or {}
    assumptions = ledger.get("assumptions", [])
    notes = ledger.get("notes", [])

    for item in updates.get("assumptions", []) or []:
        if item not in assumptions:
            assumptions.append(item)
    for item in updates.get("notes", []) or []:
        if item not in notes:
            notes.append(item)

    ledger["assumptions"] = assumptions
    ledger["notes"] = notes
    save_ledger(session, ledger)

def add_issues_from_audit(session: dict[str, Any], chunk_id: str, issues_payload: list[dict[str, Any]]) -> list[dict[str, Any]]:
    issues_state = load_issues(session)
    created = []
    for payload in issues_payload or []:
        issue = {
            "issue_id": f"I{issues_state['next_issue_id']:03d}",
            "chunk_id": chunk_id,
            "status": "open",
            "created_at": utc_now(),
            "updated_at": utc_now(),
            "resolved_at": None,
            "title": payload.get("title", "Untitled issue"),
            "severity": payload.get("severity", "low"),
            "location": payload.get("location", ""),
            "description": payload.get("description", ""),
            "evidence": payload.get("evidence", ""),
            "proposed_fix": payload.get("proposed_fix", ""),
            "tags": payload.get("tags", []),
        }
        issues_state["issues"].append(issue)
        created.append(issue)
        issues_state["next_issue_id"] += 1
    save_issues(session, issues_state)
    return created

def update_usage_from_response(session: dict[str, Any], chunk_id: str, resp) -> dict[str, Any]:
    usage_state = load_usage(session)
    usage_obj = to_jsonable(getattr(resp, "usage", {}) or {})
    cost = compute_usage_cost(session["model"], usage_obj)

    totals = usage_state["totals"]
    totals["input_tokens"] += int(usage_obj.get("input_tokens", 0) or 0)
    totals["cached_tokens"] += int(((usage_obj.get("input_tokens_details") or {}).get("cached_tokens", 0)) or 0)
    totals["output_tokens"] += int(usage_obj.get("output_tokens", 0) or 0)
    totals["reasoning_tokens"] += int(((usage_obj.get("output_tokens_details") or {}).get("reasoning_tokens", 0)) or 0)
    totals["total_tokens"] += int(usage_obj.get("total_tokens", 0) or 0)
    totals["cost_usd"] += float(cost["total_cost"])

    usage_state["per_chunk"].append({
        "time": utc_now(),
        "chunk_id": chunk_id,
        "usage": usage_obj,
        "cost": cost,
    })
    save_usage(session, usage_state)
    return {"usage": usage_obj, "cost": cost, "usage_state": usage_state}

def finalize_chunk(
    session: dict[str, Any],
    chunk: dict[str, Any],
    resp,
    display_output: bool = True,
    verification_mode: str = "local_python_only",
    used_code_interpreter_tool: bool = False,
) -> dict[str, Any]:
    root = Path(session["workdir"])
    chunk_id = chunk["chunk_id"]
    verification_mode = _normalize_verification_mode(verification_mode)

    raw_json_path = root / "responses" / f"{chunk_id}_{resp.id}.json"
    resp_json = to_jsonable(resp)
    save_json(raw_json_path, resp_json)

    raw_text = (resp.output_text or "").strip()
    raw_text_path = root / "responses" / f"{chunk_id}_{resp.id}.raw.txt"
    if raw_text:
        raw_text_path.write_text(raw_text, encoding="utf-8")

    try:
        audit = parse_audit_response(resp)
    except Exception as e:
        log_path = root / "logs" / "parse_failures.jsonl"
        append_jsonl(log_path, {
            "time": utc_now(),
            "chunk_id": chunk_id,
            "response_id": getattr(resp, "id", None),
            "error": repr(e),
            "raw_text_path": str(raw_text_path) if raw_text else None,
        })
        raise RuntimeError(
            f"Could not parse structured output for {chunk_id}. "
            f"See {raw_text_path.name if raw_text else raw_json_path.name} and logs/parse_failures.jsonl"
        ) from e

    # fill defaults
    audit.setdefault("label", chunk["label"])
    audit.setdefault("boundary", chunk["boundary"])
    audit.setdefault("chunk_too_large", False)
    audit.setdefault("chunk_split_suggestions", [])
    audit.setdefault("assumptions_and_notation", [])
    audit.setdefault("verified_steps", [])
    audit.setdefault("issues", [])
    audit.setdefault("python_checks", [])
    audit.setdefault("latex_patch", "")
    audit.setdefault("ledger_updates", {"assumptions": [], "notes": []})
    audit.setdefault("next_boundary_hint", "")
    audit.setdefault("confidence", "medium")

    structured_path = root / "responses" / f"{chunk_id}.structured.json"
    save_json(structured_path, audit)

    md_text = render_audit_markdown(audit)
    md_path = root / "responses" / f"{chunk_id}.md"
    md_path.write_text(md_text, encoding="utf-8")

    verification_summary = {
        "used_code_interpreter": False,
        "tool_event_count": 0,
        "file_ids": [],
        "container_ids": [],
    }
    verification_path = None
    if verification_mode == "code_interpreter" or used_code_interpreter_tool:
        verification_summary = _extract_code_interpreter_summary(resp_json)
        verification_payload = {
            "time": utc_now(),
            "chunk_id": chunk_id,
            "response_id": getattr(resp, "id", None),
            "verification_mode": verification_mode,
            "used_code_interpreter_tool": bool(used_code_interpreter_tool),
            "summary": verification_summary,
        }
        verification_path = root / "responses" / f"{chunk_id}_{resp.id}.verification.json"
        save_json(verification_path, verification_payload)
        if session.get("reuse_code_interpreter_container") and verification_summary.get("container_ids"):
            session["code_interpreter_container_id"] = verification_summary["container_ids"][0]

    extracted = save_patch_and_code_files(session, chunk_id, audit)
    created_issues = add_issues_from_audit(session, chunk_id, audit.get("issues", []))
    update_ledger_from_audit(session, audit)
    usage_update = update_usage_from_response(session, chunk_id, resp)

    record = {
        "time": utc_now(),
        "chunk_id": chunk_id,
        "chunk_index": chunk["chunk_index"],
        "label": chunk["label"],
        "boundary": chunk["boundary"],
        "source_kind": chunk["source_kind"],
        "page_start": chunk["page_start"],
        "page_end": chunk["page_end"],
        "paper_progress_end": chunk["paper_progress_end"],
        "response_id": resp.id,
        "request_path": pending.get("request_path"),
        "raw_response_path": str(raw_json_path),
        "structured_response_path": str(structured_path),
        "markdown_path": str(md_path),
        "latex_paths": extracted["latex_paths"],
        "python_paths": extracted["python_paths"],
        "issue_ids": [x["issue_id"] for x in created_issues],
        "cost_usd": usage_update["cost"]["total_cost"],
        "usage": usage_update["usage"],
    }
    append_jsonl(session_paths(session["workdir"])["chunk_records"], record)

    session["last_response_id"] = resp.id
    session["next_chunk_index"] = chunk["chunk_index"] + 1
    session["pending"] = None
    session["updated_at"] = utc_now()
    session["pdf_attached_in_conversation"] = True
    session["developer_prompt_seeded"] = True
    save_session(session)

    manifest = load_manifest(session)
    usage_state = usage_update["usage_state"]
    status = load_status(session)
    status.update({
        "status": "running" if chunk["chunk_index"] < len(manifest["chunks"]) else "completed",
        "progress_pct": round(100.0 * chunk["paper_progress_end"], 1),
        "current_chunk_id": chunk_id,
        "chunks_completed": chunk["chunk_index"],
        "chunks_total": len(manifest["chunks"]),
        "estimated_pages_completed": chunk["page_end"],
        "estimated_pages_total": manifest["pdf_page_count"],
        "cost_usd": usage_state["totals"]["cost_usd"],
    })
    if chunk["chunk_index"] >= len(manifest["chunks"]):
        status["status"] = "completed"
        status["progress_pct"] = 100.0
        status["estimated_pages_completed"] = manifest["pdf_page_count"]
    save_status(session, status)

    if display_output:
        print(
            f"[{chunk_id}] Progress: {status['progress_pct']:.1f}% | "
            f"Pages: {status['estimated_pages_completed']}/{status['estimated_pages_total']} | "
            f"Chunk cost: ${record['cost_usd']:.4f} | "
            f"Cumulative cost: ${usage_state['totals']['cost_usd']:.4f} | "
            f"Total tokens: {usage_state['totals']['total_tokens']}"
        )
        display_audit(audit)

    return {"audit": audit, "record": record}

def process_one_chunk(
    session: dict[str, Any],
    chunk: dict[str, Any],
    poll_every: float = 3.0,
    max_wait_seconds: Optional[float] = None,
    display_output: bool = True,
    verification_mode: str = "local_python_only",
    code_interpreter_file_ids: Optional[list[str]] = None,
    allow_ci_failure_fallback: bool = True,
) -> dict[str, Any]:
    verification_mode = _normalize_verification_mode(verification_mode)
    if verification_mode == "code_interpreter" and not session.get("use_code_interpreter", False):
        raise RuntimeError(
            "verification_mode='code_interpreter' requested, but this session has use_code_interpreter=False. "
            "Enable use_code_interpreter when creating the session or in audit_the_paper(...)."
        )

    user_input = build_user_message_for_chunk(session, chunk)
    if not session.get("developer_prompt_seeded", False):
        input_payload = [{"role": "developer", "content": [{"type": "input_text", "text": AUDIT_SYSTEM_PROMPT}]}] + user_input
    else:
        input_payload = user_input

    verification_instruction = ""
    if verification_mode == "code_interpreter":
        verification_instruction = (
            "Verification mode for this chunk: code_interpreter. "
            "Use the code_interpreter tool when numeric/symbolic verification is needed. "
            "Keep python_checks concise local fallbacks when useful."
        )
    elif verification_mode == "none":
        verification_instruction = (
            "Verification mode for this chunk: none. "
            "Skip optional numeric/symbolic code verification and keep python_checks empty unless strictly necessary."
        )
    if verification_instruction and input_payload:
        try:
            content = input_payload[-1].get("content")
            if isinstance(content, list):
                content.append({"type": "input_text", "text": verification_instruction})
        except Exception:
            pass

    prompt_path = Path(session["workdir"]) / "prompts" / f"{chunk['chunk_id']}_prompt.json"
    save_json(prompt_path, input_payload)

    resp = client.responses.create(
        model=session["model"],
        reasoning={"effort": session["reasoning_effort"]},
        conversation=session["conversation_id"],
        input=input_payload,
        text={
            "format": {
                "type": "json_schema",
                "name": "math_audit",
                "strict": True,
                "schema": AUDIT_RESPONSE_SCHEMA,
            }
        },
        background=session["background"],
        store=session["store"],
    )

    session["pending"] = {
        "chunk_id": chunk["chunk_id"],
        "response_id": resp.id,
        "created_at": utc_now(),
    }
    session["updated_at"] = utc_now()
    save_session(session)

    status = load_status(session)
    status.update({
        "status": "running",
        "current_chunk_id": chunk["chunk_id"],
        "chunks_total": len(load_manifest(session)["chunks"]),
        "estimated_pages_total": load_manifest(session)["pdf_page_count"],
    })
    save_status(session, status)

    if getattr(resp, "status", None) in WORKING_STATUSES:
        resp = wait_for_response(resp.id, poll_every=poll_every, max_wait_seconds=max_wait_seconds)

    if getattr(resp, "status", None) != "completed":
        raise RuntimeError(f"Chunk {chunk['chunk_id']} ended with status={getattr(resp, 'status', None)}")

    verification_mode = _normalize_verification_mode((pending or {}).get("verification_mode", "local_python_only"))
    used_code_interpreter_tool = bool((pending or {}).get("used_code_interpreter_tool", False))
    return finalize_chunk(
        session,
        chunk,
        resp,
        display_output=display_output,
        verification_mode=verification_mode,
        used_code_interpreter_tool=used_code_interpreter_tool,
    )


def recover_pending_chunk(
    session: dict[str, Any],
    poll_every: float = 3.0,
    max_wait_seconds: Optional[float] = None,
    display_output: bool = True,
) -> Optional[dict[str, Any]]:
    pending = session.get("pending")
    if not pending:
        return None

    response_id = pending.get("response_id")
    chunk_id = pending.get("chunk_id")
    if not response_id or not chunk_id:
        return None

    manifest = load_manifest(session)
    chunks = manifest.get("chunks", [])
    matches = [c for c in chunks if c["chunk_id"] == chunk_id]
    if not matches:
        return None
    chunk = matches[0]

    structured_path = Path(session["workdir"]) / "responses" / f"{chunk_id}.structured.json"
    if structured_path.exists():
        session["pending"] = None
        if session.get("next_chunk_index", 1) <= chunk["chunk_index"]:
            session["next_chunk_index"] = chunk["chunk_index"] + 1
        session["updated_at"] = utc_now()
        save_session(session)
        return {"recovered": False, "skipped": True, "chunk_id": chunk_id}

    resp = client.responses.retrieve(response_id)
    if getattr(resp, "status", None) in WORKING_STATUSES:
        resp = wait_for_response(response_id, poll_every=poll_every, max_wait_seconds=max_wait_seconds)

    if getattr(resp, "status", None) != "completed":
        raise RuntimeError(f"Pending chunk {chunk_id} ended with status={getattr(resp, 'status', None)}")

    verification_mode = _normalize_verification_mode((pending or {}).get("verification_mode", "local_python_only"))
    used_code_interpreter_tool = bool((pending or {}).get("used_code_interpreter_tool", False))
    return finalize_chunk(
        session,
        chunk,
        resp,
        display_output=display_output,
        verification_mode=verification_mode,
        used_code_interpreter_tool=used_code_interpreter_tool,
    )

def show_audit_status(pdf_path: str | Path) -> dict[str, Any]:
    info = get_audit_status(pdf_path)
    session = info["session"]
    status = info["status"]
    usage = info["usage"]

    print(f"Status: {status['status']}")
    print(f"Progress: {status['progress_pct']}%")
    print(f"Chunks: {status['chunks_completed']} / {status['chunks_total']}")
    print(f"Estimated pages: {status['estimated_pages_completed']} / {status['estimated_pages_total']}")
    print(f"Cumulative cost: ${status['cost_usd']:.4f}")
    print(f"Total tokens: {usage['totals']['total_tokens']}")
    return info

def show_open_issues(pdf_path: str | Path) -> list[dict[str, Any]]:
    session = load_session_from_pdf(pdf_path)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")
    issues = load_issues(session)["issues"]
    open_issues = [x for x in issues if x.get("status", "open") == "open"]
    for issue in open_issues:
        print(f"{issue['issue_id']} [{issue['severity']}] {issue['title']} ({issue['chunk_id']})")
    return open_issues
````

</details>

### Archived legacy notebook cell 09 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_policy_hooks.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
def latex_itemize(items: list[str], indent: str = "") -> str:
    if not items:
        return indent + r"\begin{itemize}" + "\n" + indent + r"\item None." + "\n" + indent + r"\end{itemize}"
    body = "\n".join(indent + r"\item " + latex_paragraph(x) for x in items)
    return indent + r"\begin{itemize}" + "\n" + body + "\n" + indent + r"\end{itemize}"

def read_chunk_records(session: dict[str, Any]) -> list[dict[str, Any]]:
    path = session_paths(session["workdir"])["chunk_records"]
    if not path.exists():
        return []
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def _python_check_entries_for_record(
    session: dict[str, Any],
    rec: dict[str, Any],
    audit: dict[str, Any],
) -> list[dict[str, str]]:
    checks = audit.get("python_checks", []) or []
    raw_paths = rec.get("python_paths", []) if isinstance(rec.get("python_paths"), list) else []
    workdir = Path(session["workdir"]).resolve()
    entries = []
    for idx, chk in enumerate(checks, start=1):
        display_path = ""
        raw_path = raw_paths[idx - 1] if idx - 1 < len(raw_paths) else None
        if raw_path:
            path_obj = Path(str(raw_path))
            try:
                if path_obj.is_absolute():
                    display_path = path_obj.resolve().relative_to(workdir).as_posix()
                else:
                    display_path = path_obj.as_posix()
            except Exception:
                display_path = (Path("python_checks") / path_obj.name).as_posix() if path_obj.name else str(path_obj)
        if not display_path:
            chunk_id = str(rec.get("chunk_id") or "chunk")
            display_path = f"python_checks/{chunk_id}_check_{idx:02d}.py"
        entry = _normalize_python_check_entry(
            chk,
            chunk_label=str(rec.get("label") or audit.get("label") or ""),
            chunk_boundary=str(rec.get("boundary") or audit.get("boundary") or ""),
        )
        entry["display_path"] = display_path
        entries.append(entry)
    return entries

def build_final_report_markdown(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    ledger = load_ledger(session)
    issues_state = load_issues(session)
    usage = load_usage(session)
    status = load_status(session)
    manifest = load_manifest(session)
    chunk_records = read_chunk_records(session)

    title = report_title or f"Audit report — {Path(session['pdf_path']).stem}"
    lines = [
        f"# {title}",
        "",
        f"- PDF: {session['pdf_path']}",
        f"- TeX: {session.get('tex_path') or 'not found'}",
        f"- Model: {session['model']}",
        f"- Reasoning effort: {session['reasoning_effort']}",
        f"- Chunking mode: {manifest.get('chunking_mode')}",
        f"- Chunks completed: {status['chunks_completed']} / {status['chunks_total']}",
        f"- Estimated pages audited: {status['estimated_pages_completed']} / {status['estimated_pages_total']}",
        f"- Total cost (USD): {usage['totals']['cost_usd']:.4f}",
        f"- Total tokens: {usage['totals']['total_tokens']}",
        "",
        "## Ledger assumptions",
        format_list_for_markdown(ledger.get("assumptions", [])),
        "",
        "## Ledger notes",
        format_list_for_markdown(ledger.get("notes", [])),
        "",
        "## Open issues",
    ]

    open_issues = [x for x in issues_state["issues"] if x.get("status", "open") == "open"]
    typo_entries = _collect_typographical_issue_entries(open_issues, chunk_records)
    main_open_issues = [x for x in open_issues if not _is_pure_typographical_issue(x)]
    if not main_open_issues:
        lines.append("- No open mathematical/correctness issues.")
    else:
        for issue in main_open_issues:
            lines.extend([
                f"### {issue['issue_id']} — {normalize_math_delimiters(issue['title'])} [{issue['severity']}]",
                f"- Chunk: {issue['chunk_id']}",
                f"- Location: {normalize_math_delimiters(issue['location'])}",
                f"- Description: {normalize_math_delimiters(issue['description'])}",
                f"- Evidence: {normalize_math_delimiters(issue['evidence'])}",
                f"- Proposed fix: {normalize_math_delimiters(issue['proposed_fix'])}",
                f"- Tags: {', '.join(issue.get('tags', [])) if issue.get('tags') else 'none'}",
                "",
            ])

    lines.extend([_typographical_errors_markdown(typo_entries), ""])

    lines.append("## Chunk overview")
    if not chunk_records:
        lines.append("- No chunk records found.")
    else:
        for rec in chunk_records:
            audit = _coerce_audit_payload(load_json(rec["structured_response_path"]))
            lines.extend([
                f"### {normalize_math_delimiters(rec['chunk_id'])} — {normalize_math_delimiters(rec['label'])}",
                f"- Boundary: {normalize_math_delimiters(rec['boundary'])}",
                f"- Estimated pages: {rec['page_start']}-{rec['page_end']}",
                f"- Cost: ${rec['cost_usd']:.4f}",
                f"- Confidence: {audit.get('confidence','medium')}",
                "",
                "#### Assumptions and notation",
                format_list_for_markdown(audit.get("assumptions_and_notation", [])),
                "",
                "#### Verified steps",
                format_list_for_markdown(audit.get("verified_steps", [])),
                "",
                "#### Issues found",
            ])
            verification_mode = str(rec.get("verification_mode", "local_python_only"))
            lines.append(f"- Verification mode: {verification_mode}")
            verification_summary = rec.get("verification_summary") if isinstance(rec.get("verification_summary"), dict) else {}
            if verification_summary.get("used_code_interpreter"):
                lines.append(f"- Code Interpreter tool events: {int(verification_summary.get('tool_event_count', 0) or 0)}")
                ci_files = verification_summary.get("file_ids") if isinstance(verification_summary.get("file_ids"), list) else []
                if ci_files:
                    lines.append(f"- Code Interpreter files: {', '.join(ci_files[:6])}")
            lines.append("")
            chunk_issues = [issue for issue in (audit.get("issues") or []) if not _is_pure_typographical_issue(issue)]
            if chunk_issues:
                for issue in chunk_issues:
                    lines.extend([
                        f"- {normalize_math_delimiters(issue.get('title','Untitled issue'))} [{issue.get('severity','low')}]",
                        f"  - Location: {normalize_math_delimiters(issue.get('location',''))}",
                        f"  - Description: {normalize_math_delimiters(issue.get('description',''))}",
                        f"  - Proposed fix: {normalize_math_delimiters(issue.get('proposed_fix',''))}",
                    ])
            elif audit.get("issues"):
                lines.append("- No mathematical/correctness issues found in this chunk. Typographical/copyediting items are summarized separately in the Typographical errors section.")
            else:
                lines.append("- No issues found.")
            python_entries = _python_check_entries_for_record(session, rec, audit)
            if python_entries:
                lines.extend([
                    "",
                    "#### Suggested local Python checks",
                ])
                for entry in python_entries:
                    lines.extend([
                        f"##### {normalize_math_delimiters(entry['purpose'])}",
                        normalize_math_delimiters(entry["description"]),
                        f"- Expected outcome: {normalize_math_delimiters(entry['expected_outcome'])}",
                        f"- Script file: `{entry['display_path']}`",
                        "```python",
                        entry["code"].rstrip(),
                        "```",
                        "",
                    ])
            lines.extend([
                "#### Next boundary hint",
                normalize_math_delimiters(audit.get("next_boundary_hint", "None.")),
                "",
            ])

    return "\n".join(lines).strip() + "\n"

def build_final_report_tex(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    ledger = load_ledger(session)
    issues_state = load_issues(session)
    usage = load_usage(session)
    status = load_status(session)
    manifest = load_manifest(session)
    chunk_records = read_chunk_records(session)

    title = latex_paragraph(report_title or f"Audit report -- {Path(session['pdf_path']).stem}")
    pdf_path_text = latex_paragraph(session["pdf_path"])
    tex_path_text = latex_paragraph(session.get("tex_path") or "not found")
    chunking_mode_text = latex_paragraph(str(manifest.get("chunking_mode")))
    model_text = latex_paragraph(session["model"])
    effort_text = latex_paragraph(session["reasoning_effort"])

    parts = [r"""\documentclass[11pt]{article}
\usepackage[a4paper,margin=1in]{geometry}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage{lmodern}
\usepackage{microtype}
\usepackage{amsmath,amssymb,mathtools}
\usepackage{hyperref}
\usepackage{enumitem}
\usepackage{longtable}
\usepackage{booktabs}
\usepackage{xcolor}
\usepackage{verbatim}
\setlist[itemize]{leftmargin=2em}
\setlength{\parskip}{0.5em}
\setlength{\parindent}{0pt}

\begin{document}
"""]
    parts.append(r"\section*{" + title + "}")
    parts.append(r"\begin{itemize}")
    parts.append(r"\item PDF: " + pdf_path_text)
    parts.append(r"\item TeX: " + tex_path_text)
    parts.append(r"\item Model: " + model_text)
    parts.append(r"\item Reasoning effort: " + effort_text)
    parts.append(r"\item Chunking mode: " + chunking_mode_text)
    parts.append(r"\item Chunks completed: " + str(status["chunks_completed"]) + " / " + str(status["chunks_total"]))
    parts.append(r"\item Estimated pages audited: " + str(status["estimated_pages_completed"]) + " / " + str(status["estimated_pages_total"]))
    parts.append(r"\item Total cost (USD): " + f"{usage['totals']['cost_usd']:.4f}")
    parts.append(r"\item Total tokens: " + str(usage["totals"]["total_tokens"]))
    parts.append(r"\end{itemize}")

    parts.append(r"\section*{Ledger assumptions}")
    parts.append(latex_itemize(ledger.get("assumptions", [])))

    parts.append(r"\section*{Ledger notes}")
    parts.append(latex_itemize(ledger.get("notes", [])))

    parts.append(r"\section*{Open issues}")
    open_issues = [x for x in issues_state["issues"] if x.get("status", "open") == "open"]
    if not open_issues:
        parts.append("No open issues.")
    else:
        for issue in open_issues:
            title_line = latex_paragraph(f"{issue['issue_id']} — {issue['title']} [{issue['severity']}]")
            parts.append(r"\subsection*{" + title_line + "}")
            parts.append(r"\begin{itemize}")
            parts.append(r"\item Chunk: " + latex_paragraph(issue["chunk_id"]))
            parts.append(r"\item Location: " + latex_paragraph(issue.get("location", "")))
            parts.append(r"\item Description: " + latex_paragraph(issue.get("description", "")))
            parts.append(r"\item Evidence: " + latex_paragraph(issue.get("evidence", "")))
            parts.append(r"\item Proposed fix: " + latex_paragraph(issue.get("proposed_fix", "")))
            tag_text = ", ".join(issue.get("tags", [])) if issue.get("tags") else "none"
            parts.append(r"\item Tags: " + latex_paragraph(tag_text))
            parts.append(r"\end{itemize}")

    parts.append(r"\section*{Chunk overview}")
    if not chunk_records:
        parts.append("No chunk records found.")
    else:
        for rec in chunk_records:
            audit = load_json(rec["structured_response_path"])
            heading = latex_paragraph(f"{rec['chunk_id']} — {rec['label']}")
            parts.append(r"\subsection*{" + heading + "}")
            parts.append(r"\begin{itemize}")
            parts.append(r"\item Boundary: " + latex_paragraph(rec["boundary"]))
            parts.append(r"\item Estimated pages: " + str(rec["page_start"]) + "--" + str(rec["page_end"]))
            parts.append(r"\item Cost: \$" + f"{rec['cost_usd']:.4f}")
            parts.append(r"\item Confidence: " + latex_paragraph(audit.get("confidence", "medium")))
            parts.append(r"\end{itemize}")

            parts.append(r"\paragraph{Assumptions and notation}")
            parts.append(latex_itemize(audit.get("assumptions_and_notation", [])))

            parts.append(r"\paragraph{Verified steps}")
            parts.append(latex_itemize(audit.get("verified_steps", [])))

            parts.append(r"\paragraph{Issues found}")
            if audit.get("issues"):
                for issue in audit["issues"]:
                    parts.append(r"\subparagraph*{" + latex_paragraph(f"{issue.get('title','Untitled issue')} [{issue.get('severity','low')}]") + "}")
                    parts.append(r"\begin{itemize}")
                    parts.append(r"\item Location: " + latex_paragraph(issue.get("location", "")))
                    parts.append(r"\item Description: " + latex_paragraph(issue.get("description", "")))
                    parts.append(r"\item Evidence: " + latex_paragraph(issue.get("evidence", "")))
                    parts.append(r"\item Proposed fix: " + latex_paragraph(issue.get("proposed_fix", "")))
                    tag_text = ", ".join(issue.get("tags", [])) if issue.get("tags") else "none"
                    parts.append(r"\item Tags: " + latex_paragraph(tag_text))
                    parts.append(r"\end{itemize}")
            else:
                parts.append("No issues found.")

            parts.append(r"\paragraph{Next boundary hint}")
            parts.append(latex_paragraph(audit.get("next_boundary_hint", "None.")))

            if audit.get("python_checks"):
                parts.append(r"\paragraph{Suggested local Python checks}")
                for chk in audit["python_checks"]:
                    parts.append(r"\textbf{" + latex_paragraph(chk.get("purpose", "Python check")) + "}")
                    parts.append(r"\begin{verbatim}")
                    parts.append(chk.get("code", "").rstrip())
                    parts.append(r"\end{verbatim}")

            if (audit.get("latex_patch") or "").strip():
                parts.append(r"\paragraph{Minimal LaTeX patch}")
                parts.append(r"\begin{verbatim}")
                parts.append(audit["latex_patch"].rstrip())
                parts.append(r"\end{verbatim}")

    parts.append(r"\end{document}")
    return "\n".join(parts) + "\n"

def build_final_report(session_or_pdf: dict[str, Any] | str | Path, report_title: Optional[str] = None) -> dict[str, str]:
    session = session_or_pdf if isinstance(session_or_pdf, dict) else load_session_from_pdf(session_or_pdf)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")
    root = Path(session["workdir"])
    report_stem = Path(session["pdf_path"]).stem + "_audit_report"

    md_text = build_final_report_markdown(session, report_title=report_title)
    tex_text = build_final_report_tex(session, report_title=report_title)

    reports_dir = root / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)

    md_path = reports_dir / f"{report_stem}.md"
    tex_path = reports_dir / f"{report_stem}.tex"
    json_path = reports_dir / f"{report_stem}.json"

    md_path.write_text(md_text, encoding="utf-8")
    tex_path.write_text(tex_text, encoding="utf-8")

    report_json = {
        "session": load_session_from_pdf(session["pdf_path"]),
        "status": load_status(session),
        "ledger": load_ledger(session),
        "issues": load_issues(session),
        "usage": load_usage(session),
        "manifest": load_manifest(session),
        "chunk_records": read_chunk_records(session),
        "generated_at": utc_now(),
    }
    save_json(json_path, report_json)

    return {
        "markdown": str(md_path),
        "tex": str(tex_path),
        "json": str(json_path),
    }
````

</details>

### Archived legacy notebook cell 10 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_runtime.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
def audit_the_paper(
    pdf_path: str | Path,
    model: str = DEFAULT_MODEL,
    reasoning_effort: str = DEFAULT_REASONING_EFFORT,
    tex_max_chars: int = 4500,
    pdf_max_chars: int = 3500,
    continue_existing: bool = True,
    poll_every: float = 3.0,
    max_wait_seconds: Optional[float] = None,
    stop_after_chunks: Optional[int] = None,
    verbose: bool = True,
) -> dict[str, Any]:
    pdf_path = Path(pdf_path).expanduser().resolve()

    session = load_session_from_pdf(pdf_path) if continue_existing else None
    if session is None:
        session = create_new_session(
            pdf_path,
            model=model,
            reasoning_effort=reasoning_effort,
            tex_max_chars=tex_max_chars,
            pdf_max_chars=pdf_max_chars,
        )
        if verbose:
            print(f"Created workdir: {session['workdir']}")
            print(f"PDF: {session['pdf_path']}")
            print(f"TeX: {session['tex_path'] or 'not found'}")
    else:
        session["model"] = model
        session["reasoning_effort"] = reasoning_effort
        session["updated_at"] = utc_now()
        save_session(session)
        if verbose:
            print(f"Loaded existing session from {session['workdir']}")
            print(f"Using model: {session['model']}")
            print(f"Using reasoning effort: {session['reasoning_effort']}")

    manifest = load_manifest(session)
    chunks = manifest["chunks"]

    if not chunks:
        raise RuntimeError("Chunk manifest is empty.")

    if session.get("pending"):
        if verbose:
            print(f"Recovering pending chunk {session['pending'].get('chunk_id')} ...")
        recover_pending_chunk(
            session,
            poll_every=poll_every,
            max_wait_seconds=max_wait_seconds,
            display_output=True,
        )

    start_idx = session["next_chunk_index"]
    processed = 0

    for idx in range(start_idx, len(chunks) + 1):
        chunk = chunks[idx - 1]
        process_one_chunk(
            session,
            chunk,
            poll_every=poll_every,
            max_wait_seconds=max_wait_seconds,
            display_output=True,
        )
        processed += 1
        if stop_after_chunks is not None and processed >= stop_after_chunks:
            break

    report_paths = build_final_report(session)
    if verbose:
        print(f"Final report (.md): {report_paths['markdown']}")
        print(f"Final report (.tex): {report_paths['tex']}")
        print(f"JSON report: {report_paths['json']}")

    return {
        "session": session,
        "status": load_status(session),
        "usage": load_usage(session),
        "report_paths": report_paths,
    }
````

</details>

### Archived legacy notebook cell 11 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_policy_hooks.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Patched recovery + safer final report with selective macro import from the paper preamble.

import json
import re
from pathlib import Path

_SECTION_HEADINGS = [
    "Assumptions and notation",
    "Verified steps",
    "Issues found",
    "Suggested local Python checks",
    "Minimal LaTeX patch",
    "Ledger updates",
    "Next boundary hint",
]

def extract_json_object(text: str) -> dict[str, Any]:
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty text")

    # direct JSON
    try:
        return json.loads(text)
    except Exception:
        pass

    # fenced ```json ... ```
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    # first parseable JSON object in the text
    decoder = json.JSONDecoder()
    first = text.find("{")
    while first >= 0:
        try:
            obj, _ = decoder.raw_decode(text[first:])
            return obj
        except Exception:
            first = text.find("{", first + 1)

    raise ValueError("Could not parse JSON object from model output.")

def _clean_lines_to_items(text: str) -> list[str]:
    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]
    if not lines:
        return []
    if len(lines) == 1 and lines[0].lower().startswith("no "):
        return []
    return lines

def _parse_issues_block(text: str) -> list[dict[str, Any]]:
    text = str(text).strip()
    if not text or text.lower().startswith("no issues found"):
        return []

    lines = text.splitlines()
    issues = []
    current = None
    current_field = None

    def flush():
        nonlocal current
        if current is not None:
            if isinstance(current.get("tags"), str):
                current["tags"] = [t.strip() for t in current["tags"].split(",") if t.strip()]
            current.setdefault("severity", "medium")
            current.setdefault("status", "open")
            current.setdefault("location", "")
            current.setdefault("description", "")
            current.setdefault("evidence", "")
            current.setdefault("proposed_fix", "")
            current.setdefault("tags", [])
            issues.append(current)
            current = None

    def next_nonempty(idx: int) -> str:
        for j in range(idx + 1, len(lines)):
            s = lines[j].strip()
            if s:
                return s
        return ""

    field_re = re.compile(r"^(Status|Location|Description|Evidence|Proposed fix|Tags):\s*(.*)$")

    for i, line in enumerate(lines):
        s = line.strip()
        if not s:
            continue

        m = field_re.match(s)
        if m:
            if current is None:
                current = {"title": "Issue", "severity": "medium"}
            key = m.group(1).lower().replace(" ", "_")
            val = m.group(2).strip()
            if key == "tags":
                current["tags"] = val
            else:
                current[key] = val
            current_field = key
            continue

        looks_like_title = (
            ":" not in s
            and (
                re.search(r"\[[^\]]+\]\s*$", s) is not None
                or next_nonempty(i).startswith("Status:")
            )
        )

        if looks_like_title:
            flush()
            m2 = re.match(r"^(.*?)(?:\s*\[([A-Za-z_ -]+)\])?$", s)
            title = m2.group(1).strip()
            sev = (m2.group(2) or "medium").strip().lower()
            current = {"title": title, "severity": sev}
            current_field = None
            continue

        if current is not None and current_field is not None:
            current[current_field] = (current.get(current_field, "") + " " + s).strip()

    flush()
    return issues

def _parse_ledger_block(text: str) -> dict[str, list[str]]:
    text = str(text).strip()
    if not text:
        return {}

    lines = text.splitlines()
    ledger = {}
    current = None

    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s in {"Assumptions", "Notes"}:
            current = s.lower()
            ledger.setdefault(current, [])
            continue
        if current is None:
            ledger.setdefault("notes", []).append(s)
        else:
            ledger[current].append(s)

    return ledger

def parse_legacy_markdown_audit(text: str) -> dict[str, Any]:
    lines = str(text).replace("\r\n", "\n").split("\n")

    label = ""
    boundary = ""
    confidence = ""
    buffers = {h: [] for h in _SECTION_HEADINGS}
    current = None
    first_real_seen = False

    for line in lines:
        s = line.strip()

        if not first_real_seen and s:
            if not s.startswith("Boundary:") and s not in _SECTION_HEADINGS and not s.startswith("Confidence:"):
                label = s
                first_real_seen = True
                continue

        if s.startswith("Boundary:"):
            boundary = s.split(":", 1)[1].strip()
            current = None
            first_real_seen = True
            continue

        if s.startswith("Confidence:"):
            confidence = s.split(":", 1)[1].strip()
            current = None
            continue

        if s in _SECTION_HEADINGS:
            current = s
            first_real_seen = True
            continue

        if current is not None:
            buffers[current].append(line)

    sec = {k: "\n".join(v).strip() for k, v in buffers.items()}

    pychecks_text = sec["Suggested local Python checks"].strip()
    pychecks = []
    if pychecks_text and not pychecks_text.lower().startswith("no python checks suggested"):
        pychecks = [{
            "purpose": "Suggested local Python checks",
            "description": "Legacy Python verification script extracted from the chunk audit. Review the code block together with the chunk context for the exact claim being tested.",
            "expected_outcome": "The script should run without exceptions or failed assertions, and its printed output should support the claim under review.",
            "code": pychecks_text,
        }]

    return {
        "label": label or "Audit chunk",
        "boundary": boundary,
        "assumptions_and_notation": _clean_lines_to_items(sec["Assumptions and notation"]),
        "verified_steps": _clean_lines_to_items(sec["Verified steps"]),
        "issues": _parse_issues_block(sec["Issues found"]),
        "python_checks": pychecks,
        "latex_patch": sec["Minimal LaTeX patch"].strip(),
        "ledger_updates": _parse_ledger_block(sec["Ledger updates"]),
        "next_boundary_hint": sec["Next boundary hint"].strip(),
        "confidence": confidence or "",
        "_legacy_raw_markdown": text,
    }

def parse_audit_response(resp) -> dict[str, Any]:
    parsed = getattr(resp, "output_parsed", None)
    if isinstance(parsed, dict):
        return parsed

    raw_text = (getattr(resp, "output_text", None) or "").strip()
    if raw_text:
        try:
            return extract_json_object(raw_text)
        except Exception:
            return parse_legacy_markdown_audit(raw_text)

    raw = to_jsonable(resp)
    for item in raw.get("output", []) or []:
        if item.get("type") == "message":
            for content in item.get("content", []) or []:
                if content.get("type") == "output_text":
                    txt = (content.get("text") or "").strip()
                    if txt:
                        try:
                            return extract_json_object(txt)
                        except Exception:
                            return parse_legacy_markdown_audit(txt)
    raise ValueError("Could not locate a structured audit object in the model response.")

def _coerce_audit_payload(audit: Any) -> dict[str, Any]:
    if not isinstance(audit, dict):
        audit = {}

    def _as_str(x: Any) -> str:
        return _strip_unsafe_control_chars(_repair_json_escape_artifacts("" if x is None else str(x)))

    def _as_list_of_str(x: Any) -> list[str]:
        if isinstance(x, list):
            return [_as_str(v) for v in x if _as_str(v).strip()]
        if x is None:
            return []
        s = _as_str(x).strip()
        return [s] if s else []

    def _as_issue_list(x: Any) -> list[dict[str, Any]]:
        out = []
        for it in (x if isinstance(x, list) else []):
            if not isinstance(it, dict):
                continue
            sev = _as_str(it.get("severity", "medium")).lower().strip() or "medium"
            if sev not in {"low", "medium", "high", "critical"}:
                sev = "medium"
            out.append({
                "title": _as_str(it.get("title", "Untitled issue")),
                "severity": sev,
                "location": _as_str(it.get("location", "")),
                "description": _as_str(it.get("description", "")),
                "evidence": _as_str(it.get("evidence", "")),
                "proposed_fix": _as_str(it.get("proposed_fix", "")),
                "tags": _as_list_of_str(it.get("tags", [])),
            })
        return out

    def _as_python_checks(x: Any) -> list[dict[str, str]]:
        out = []
        for it in (x if isinstance(x, list) else []):
            if not isinstance(it, dict):
                continue
            purpose = _as_str(it.get("purpose", "")).strip() or "Python check"
            description = _as_str(it.get("description", "")).strip() or purpose
            expected_outcome = _as_str(it.get("expected_outcome", "")).strip()
            code = _as_str(it.get("code", ""))
            if purpose or description or expected_outcome or code:
                out.append({
                    "purpose": purpose,
                    "description": description,
                    "expected_outcome": expected_outcome,
                    "code": code,
                })
        return out

    ledger = audit.get("ledger_updates") if isinstance(audit.get("ledger_updates"), dict) else {}
    return {
        "label": _as_str(audit.get("label", "")),
        "boundary": _as_str(audit.get("boundary", "")),
        "chunk_too_large": bool(audit.get("chunk_too_large", False)),
        "chunk_split_suggestions": _as_list_of_str(audit.get("chunk_split_suggestions", [])),
        "assumptions_and_notation": _as_list_of_str(audit.get("assumptions_and_notation", [])),
        "verified_steps": _as_list_of_str(audit.get("verified_steps", [])),
        "issues": _as_issue_list(audit.get("issues", [])),
        "python_checks": _as_python_checks(audit.get("python_checks", [])),
        "latex_patch": _as_str(audit.get("latex_patch", "")),
        "ledger_updates": {
            "assumptions": _as_list_of_str(ledger.get("assumptions", [])),
            "notes": _as_list_of_str(ledger.get("notes", [])),
        },
        "next_boundary_hint": _as_str(audit.get("next_boundary_hint", "")),
        "confidence": _as_str(audit.get("confidence", "medium")) or "medium",
    }

_TYPO_POSITIVE_TAGS = {
    "typo", "editorial", "copyedit", "copy-edit", "copyediting",
    "grammar", "spelling", "punctuation", "cross-reference", "placeholder",
}
_TYPO_NEGATIVE_TAGS = {
    "mathematical-correctness", "proof-gap", "uniformity", "domain", "convergence",
    "asymptotics", "indexing", "hidden-assumption", "well-posedness",
    "omitted-justification", "analyticity", "formal-power-series", "definition",
    "range", "parameter-range", "variance", "boundary", "quantifiers", "big-o",
    "exponents", "uniformity", "asymptotic-expansion",
}
_TYPO_KEYWORDS = (
    "typo", "typographical", "spelling", "grammar", "copyedit", "copy editing",
    "copyediting", "misprint", "misspell", "punctuation", "placeholder",
    "placeholders", "incomplete marker",
)
_TYPO_LITERAL_FRAGMENT_RE = re.compile(r"`([^`\n]{2,120})`")

def _issue_tags(issue: dict[str, Any]) -> set[str]:
    out = set()
    for tag in issue.get("tags", []) or []:
        s = normalize_whitespace(str(tag)).lower()
        if s:
            out.add(s)
    return out

def _issue_text_for_typo_classification(issue: dict[str, Any]) -> str:
    parts = [
        issue.get("title", ""),
        issue.get("location", ""),
        issue.get("description", ""),
        issue.get("evidence", ""),
        issue.get("proposed_fix", ""),
    ]
    return normalize_whitespace(" ".join(str(x) for x in parts if x)).lower()

def _is_pure_typographical_issue(issue: dict[str, Any]) -> bool:
    tags = _issue_tags(issue)
    text = _issue_text_for_typo_classification(issue)
    positive = bool(tags & _TYPO_POSITIVE_TAGS) or any(keyword in text for keyword in _TYPO_KEYWORDS)
    negative = bool(tags & _TYPO_NEGATIVE_TAGS)
    return positive and not negative

def _chunk_record_map(chunk_records: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    out = {}
    for rec in chunk_records or []:
        chunk_id = str(rec.get("chunk_id") or "").strip()
        if chunk_id:
            out[chunk_id] = rec
    return out

def _page_range_text(rec: Optional[dict[str, Any]], markdown: bool = True) -> str:
    if not isinstance(rec, dict):
        return ""
    try:
        start = int(rec.get("page_start"))
        end = int(rec.get("page_end"))
    except Exception:
        return ""
    if start <= 0 or end <= 0:
        return ""
    if start == end:
        return f"Page {start}"
    sep = "-" if markdown else "--"
    return f"Pages {start}{sep}{end}"

def _extract_searchable_phrases(issue: dict[str, Any], max_items: int = 2) -> list[str]:
    fragments = []
    for key in ["evidence", "description", "location"]:
        text = str(issue.get(key, "") or "")
        for frag in _TYPO_LITERAL_FRAGMENT_RE.findall(text):
            frag = normalize_whitespace(frag)
            if frag:
                fragments.append(frag)
    fragments = _dedupe_preserve_order(fragments)
    preferred = [frag for frag in fragments if any(ch.isspace() for ch in frag) or "?" in frag or "--" in frag]
    return (preferred or fragments)[:max_items]

def _extract_incorrect_text(issue: dict[str, Any]) -> str:
    for key in ["description", "evidence"]:
        text = str(issue.get(key, "") or "")
        m = re.search(r"currently reads\s+(\$\$.*?\$\$|`[^`]+`)", text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            return normalize_whitespace(m.group(1).strip("`"))
    return ""

def _collect_typographical_issue_entries(
    issues: list[dict[str, Any]],
    chunk_records: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    chunk_map = _chunk_record_map(chunk_records)
    entries = []
    for issue in issues or []:
        if not _is_pure_typographical_issue(issue):
            continue
        rec = chunk_map.get(str(issue.get("chunk_id") or "").strip())
        entries.append({
            "issue": issue,
            "chunk_record": rec,
            "page_text_md": _page_range_text(rec, markdown=True),
            "page_text_tex": _page_range_text(rec, markdown=False),
            "location_detail": normalize_whitespace(str(issue.get("location", "") or "")),
            "searchable_phrases": _extract_searchable_phrases(issue),
            "incorrect_text": _extract_incorrect_text(issue),
            "page_start_sort": int(rec.get("page_start") or 10**9) if isinstance(rec, dict) else 10**9,
            "page_end_sort": int(rec.get("page_end") or 10**9) if isinstance(rec, dict) else 10**9,
        })
    entries.sort(key=lambda item: (item["page_start_sort"], item["page_end_sort"], str(item["issue"].get("issue_id") or "")))
    return entries

def _typographical_errors_markdown(entries: list[dict[str, Any]]) -> str:
    lines = ["## Typographical errors", ""]
    if not entries:
        lines.append("- No typographical/copyediting issues identified.")
        lines.append("")
        return "\n".join(lines)
    for entry in entries:
        issue = entry["issue"]
        lines.append(f"### {issue.get('issue_id', 'issue')} — {normalize_math_delimiters(issue.get('title', 'Untitled issue'))} [{issue.get('severity', 'low')}]")
        if entry.get("page_text_md"):
            lines.append(f"- Approximate pages: {entry['page_text_md']}")
        if entry.get("location_detail"):
            lines.append(f"- Location detail: {normalize_math_delimiters(entry['location_detail'])}")
        phrases = entry.get("searchable_phrases") or []
        if phrases:
            label = "Searchable phrase" if len(phrases) == 1 else "Searchable phrases"
            lines.append(f"- {label}: {'; '.join(normalize_math_delimiters(x) for x in phrases)}")
        if entry.get("incorrect_text"):
            lines.append(f"- Incorrect text: {normalize_math_delimiters(entry['incorrect_text'])}")
        if issue.get("proposed_fix"):
            lines.append(f"- Suggested correction: {normalize_math_delimiters(issue.get('proposed_fix', ''))}")
        lines.append("")
    return "\n".join(lines)

def _typographical_errors_tex(entries: list[dict[str, Any]]) -> str:
    parts = [r"\section*{Typographical errors}"]
    if not entries:
        parts.append("No typographical/copyediting issues identified.")
        return "\n".join(parts) + "\n"
    for entry in entries:
        issue = entry["issue"]
        title = report_latex_paragraph(f"{issue.get('issue_id', 'issue')} -- {issue.get('title', 'Untitled issue')} [{issue.get('severity', 'low')}]")
        parts.append(r"\subsection*{" + title + "}")
        parts.append(r"\begin{itemize}")
        if entry.get("page_text_tex"):
            parts.append(r"\item Approximate pages: " + report_latex_paragraph(entry["page_text_tex"]))
        if entry.get("location_detail"):
            parts.append(r"\item Location detail: " + report_latex_paragraph(entry["location_detail"]))
        phrases = entry.get("searchable_phrases") or []
        if phrases:
            label = "Searchable phrase" if len(phrases) == 1 else "Searchable phrases"
            parts.append(r"\item " + report_latex_paragraph(label + ": " + "; ".join(str(x) for x in phrases)))
        if entry.get("incorrect_text"):
            parts.append(r"\item Incorrect text: " + report_latex_paragraph(entry["incorrect_text"]))
        if issue.get("proposed_fix"):
            parts.append(r"\item Suggested correction: " + report_latex_paragraph(issue.get("proposed_fix", "")))
        parts.append(r"\end{itemize}")
    return "\n".join(parts) + "\n"

SAFE_REPORT_PACKAGES = {
    "tikz", "xspace", "xparse", "amsmath", "amssymb", "mathtools", "amsthm",
    "bm", "bbm", "mathrsfs", "stmaryrd", "graphicx", "etoolbox", "array",
    "calc", "ifthen", "xcolor", "dsfont", "yhmath", "accents"
}
BASE_REPORT_PACKAGES = {
    "geometry", "fontenc", "inputenc", "lmodern", "microtype", "amsmath",
    "amssymb", "mathtools", "hyperref", "enumitem", "longtable", "booktabs",
    "xcolor", "fancyvrb"
}

def _dedupe_preserve_order(seq: list[str]) -> list[str]:
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _extract_safe_report_preamble(tex_path: str | Path | None) -> str:
    if not tex_path:
        return ""
    tex_path = Path(tex_path)
    if not tex_path.exists():
        return ""

    try:
        text = tex_path.read_text(encoding="utf-8")
    except Exception:
        try:
            text = tex_path.read_text(encoding="latin-1")
        except Exception:
            return ""

    pre = text.split(r"\begin{document}", 1)[0]
    pre = strip_tex_comments(pre)
    lines = pre.splitlines()

    package_lines = []
    macro_lines = []

    collecting = False
    current = []
    balance = 0

    def flush_current():
        nonlocal collecting, current, balance
        if current:
            block = "\n".join(current).strip()
            if block:
                macro_lines.append(block)
        collecting = False
        current = []
        balance = 0

    macro_start_re = re.compile(
        r"^\s*\\(?:newcommand|renewcommand|providecommand|DeclareMathOperator\*?|DeclareRobustCommand|def|newtheorem)\b"
    )

    usepkg_re = re.compile(r"^\s*\\usepackage(\[[^\]]*\])?\{([^}]*)\}")
    tikzlib_re = re.compile(r"^\s*\\usetikzlibrary(?:\[[^\]]*\])?\{([^}]*)\}")

    for line in lines:
        s = line.strip()
        if not s:
            if collecting:
                current.append(line)
            continue

        m_pkg = usepkg_re.match(s)
        if m_pkg:
            pkg_opts = m_pkg.group(1) or ""
            pkgs = [p.strip() for p in m_pkg.group(2).split(",") if p.strip()]
            keep_pkgs = [p for p in pkgs if (p in SAFE_REPORT_PACKAGES) and (p not in BASE_REPORT_PACKAGES)]
            if keep_pkgs:
                package_lines.append(r"\usepackage" + pkg_opts + "{" + ",".join(keep_pkgs) + "}")
            continue

        m_tikz = tikzlib_re.match(s)
        if m_tikz:
            package_lines.append(s)
            continue

        if not collecting and macro_start_re.match(s):
            collecting = True
            current = [line]
            balance = line.count("{") - line.count("}")
            if balance <= 0:
                flush_current()
            continue

        if collecting:
            current.append(line)
            balance += line.count("{") - line.count("}")
            if balance <= 0:
                flush_current()

    if collecting:
        flush_current()

    keep_macros = []
    for block in macro_lines:
        if any(tok in block for tok in [r"\write18", r"\openout", r"\input", r"\include", r"\catcode", r"\read"]):
            continue
        keep_macros.append(block)

    package_lines = _dedupe_preserve_order(package_lines)
    keep_macros = _dedupe_preserve_order(keep_macros)

    if not package_lines and not keep_macros:
        return ""

    return "\n".join(package_lines + [""] + keep_macros).strip() + "\n"

_DANGEROUS_MATH_COMMAND_RE = re.compile(
    r"\\(?:usepackage|documentclass|begin|end|input|include|newcommand|renewcommand|providecommand|def|write18|openout|catcode|usetikzlibrary|ref|eqref|autoref|cref|Cref|cite)\b"
)

def _normalize_report_latex_unicode_math(s: str) -> str:
    s = "" if s is None else str(s)
    repl = {
        "∇": r"\\nabla",
        "ρ": r"\\rho",
        "λ": r"\\lambda",
        "Λ": r"\\Lambda",
        "≤": r" \\le ",
        "≥": r" \\ge ",
        "≪": r" \\ll ",
        "≫": r" \\gg ",
        "≍": r" \\asymp ",
        "≈": r" \\approx ",
        "→": r" \\to ",
        "∞": r"\\infty",
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    return s

def _report_escape_text(s: str) -> str:
    s = sanitize_ascii_punctuation("" if s is None else str(s))
    s = _normalize_report_latex_unicode_math(s)
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(repl.get(ch, ch) for ch in s)

def report_latex_paragraph(text: str) -> str:
    text = normalize_math_delimiters("" if text is None else str(text))
    text = _strip_unsafe_control_chars(_repair_json_escape_artifacts(text))
    parts = re.split(r"(\$\$.*?\$\$|\$.*?\$)", text, flags=re.DOTALL)
    out = []
    for part in parts:
        if not part:
            continue
        if (part.startswith("$$") and part.endswith("$$")) or (part.startswith("$") and part.endswith("$")):
            delim = "$$" if part.startswith("$$") else "$"
            body = part[len(delim):-len(delim)]
            body = sanitize_ascii_punctuation(body)
            body = _normalize_report_latex_unicode_math(body)
            if _DANGEROUS_MATH_COMMAND_RE.search(body):
                out.append(r"\texttt{" + _report_escape_text(part) + "}")
            else:
                out.append(delim + body + delim)
        else:
            out.append(_report_escape_text(part))
    return "".join(out)

def report_latex_itemize(items: list[str], indent: str = "") -> str:
    items = list(items or [])
    if not items:
        return indent + r"\begin{itemize}" + "\n" + indent + r"\item None." + "\n" + indent + r"\end{itemize}"
    body = "\n".join(indent + r"\item " + report_latex_paragraph(x) for x in items)
    return indent + r"\begin{itemize}" + "\n" + body + "\n" + indent + r"\end{itemize}"

def _verbatim_block(text: str) -> str:
    text = _strip_unsafe_control_chars(_repair_json_escape_artifacts("" if text is None else str(text))).rstrip()
    if not text:
        return ""
    text = text.replace(r"\end{Verbatim}", r"\string\end{Verbatim}")
    return "\\begin{Verbatim}[fontsize=\\small]\n" + text + "\n\\end{Verbatim}\n"

def build_final_report_tex(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    ledger = load_ledger(session)
    issues_state = load_issues(session)
    usage = load_usage(session)
    status = load_status(session)
    manifest = load_manifest(session)
    chunk_records = read_chunk_records(session)

    title = report_latex_paragraph(report_title or f"Audit report -- {Path(session['pdf_path']).stem}")
    pdf_path_text = report_latex_paragraph(session["pdf_path"])
    tex_path_text = report_latex_paragraph(session.get("tex_path") or "not found")
    chunking_mode_text = report_latex_paragraph(str(manifest.get("chunking_mode")))
    model_text = report_latex_paragraph(session["model"])
    effort_text = report_latex_paragraph(session["reasoning_effort"])
    imported_preamble = _extract_safe_report_preamble(session.get("tex_path"))

    parts = [r"""\documentclass[11pt]{article}
\usepackage[a4paper,margin=1in]{geometry}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage{lmodern}
\usepackage{microtype}
\usepackage{amsmath,amssymb,mathtools}
\usepackage{hyperref}
\usepackage{enumitem}
\usepackage{longtable}
\usepackage{booktabs}
\usepackage{xcolor}
\usepackage{fancyvrb}
\setlist[itemize]{leftmargin=2em}
\setlength{\parskip}{0.5em}
\setlength{\parindent}{0pt}

"""]
    if imported_preamble:
        parts.append("% Imported selectively from the paper preamble for macro compatibility\n")
        parts.append(imported_preamble)
        parts.append("\n")

    parts.append(r"\begin{document}" + "\n")
    parts.append(r"\section*{" + title + "}" + "\n")
    parts.append(r"\begin{itemize}" + "\n")
    parts.append(r"\item PDF: " + pdf_path_text + "\n")
    parts.append(r"\item TeX: " + tex_path_text + "\n")
    parts.append(r"\item Model: " + model_text + "\n")
    parts.append(r"\item Reasoning effort: " + effort_text + "\n")
    parts.append(r"\item Chunking mode: " + chunking_mode_text + "\n")
    parts.append(r"\item Chunks completed: " + str(status["chunks_completed"]) + " / " + str(status["chunks_total"]) + "\n")
    parts.append(r"\item Estimated pages audited: " + str(status["estimated_pages_completed"]) + " / " + str(status["estimated_pages_total"]) + "\n")
    parts.append(r"\item Total cost (USD): " + f"{usage['totals']['cost_usd']:.4f}" + "\n")
    parts.append(r"\item Total tokens: " + str(usage["totals"]["total_tokens"]) + "\n")
    parts.append(r"\end{itemize}" + "\n")

    parts.append(r"\section*{Ledger assumptions}" + "\n")
    parts.append(report_latex_itemize(ledger.get("assumptions", [])) + "\n")

    parts.append(r"\section*{Ledger notes}" + "\n")
    parts.append(report_latex_itemize(ledger.get("notes", [])) + "\n")

    parts.append(r"\section*{Open issues}" + "\n")
    open_issues = [x for x in issues_state["issues"] if x.get("status", "open") == "open"]
    typo_entries = _collect_typographical_issue_entries(open_issues, chunk_records)
    main_open_issues = [x for x in open_issues if not _is_pure_typographical_issue(x)]
    if not main_open_issues:
        parts.append("No open mathematical/correctness issues.\n")
    else:
        for issue in main_open_issues:
            title_line = report_latex_paragraph(f"{issue['issue_id']} -- {issue['title']} [{issue['severity']}]")
            parts.append(r"\subsection*{" + title_line + "}" + "\n")
            parts.append(r"\begin{itemize}" + "\n")
            parts.append(r"\item Chunk: " + report_latex_paragraph(issue["chunk_id"]) + "\n")
            parts.append(r"\item Location: " + report_latex_paragraph(issue.get("location", "")) + "\n")
            parts.append(r"\item Description: " + report_latex_paragraph(issue.get("description", "")) + "\n")
            parts.append(r"\item Evidence: " + report_latex_paragraph(issue.get("evidence", "")) + "\n")
            parts.append(r"\item Proposed fix: " + report_latex_paragraph(issue.get("proposed_fix", "")) + "\n")
            tag_text = ", ".join(issue.get("tags", [])) if issue.get("tags") else "none"
            parts.append(r"\item Tags: " + report_latex_paragraph(tag_text) + "\n")
            parts.append(r"\end{itemize}" + "\n")

    parts.append(_typographical_errors_tex(typo_entries) + "\n")

    parts.append(r"\section*{Chunk overview}" + "\n")
    if not chunk_records:
        parts.append("No chunk records found.\n")
    else:
        for rec in chunk_records:
            try:
                audit = _coerce_audit_payload(load_json(rec["structured_response_path"]))
            except Exception as e:
                heading = report_latex_paragraph(f"{rec.get('chunk_id','chunk')} -- {rec.get('label','(unavailable)')}")
                parts.append(r"\subsection*{" + heading + "}" + "\n")
                warn = f"Could not load structured response at {rec.get('structured_response_path','(missing)')}: {e}"
                parts.append(r"\textbf{Warning:} " + report_latex_paragraph(warn) + "\n\n")
                continue
            heading = report_latex_paragraph(f"{rec['chunk_id']} -- {rec['label']}")
            parts.append(r"\subsection*{" + heading + "}" + "\n")
            parts.append(r"\begin{itemize}" + "\n")
            parts.append(r"\item Boundary: " + report_latex_paragraph(rec["boundary"]) + "\n")
            parts.append(r"\item Estimated pages: " + str(rec["page_start"]) + "--" + str(rec["page_end"]) + "\n")
            parts.append(r"\item Cost: \$" + f"{rec['cost_usd']:.4f}" + "\n")
            parts.append(r"\item Confidence: " + report_latex_paragraph(audit.get("confidence", "medium")) + "\n")
            verification_mode = str(rec.get("verification_mode", "local_python_only"))
            parts.append(r"\item Verification mode: " + report_latex_paragraph(verification_mode) + "\n")
            verification_summary = rec.get("verification_summary") if isinstance(rec.get("verification_summary"), dict) else {}
            if verification_summary.get("used_code_interpreter"):
                tool_events = int(verification_summary.get("tool_event_count", 0) or 0)
                parts.append(r"\item Code Interpreter tool events: " + str(tool_events) + "\n")
                ci_files = verification_summary.get("file_ids") if isinstance(verification_summary.get("file_ids"), list) else []
                if ci_files:
                    parts.append(r"\item Code Interpreter files: " + report_latex_paragraph(", ".join(ci_files[:6])) + "\n")
            parts.append(r"\end{itemize}" + "\n")

            parts.append(r"\paragraph{Assumptions and notation}" + "\n")
            parts.append(report_latex_itemize(audit.get("assumptions_and_notation", [])) + "\n")

            parts.append(r"\paragraph{Verified steps}" + "\n")
            parts.append(report_latex_itemize(audit.get("verified_steps", [])) + "\n")

            parts.append(r"\paragraph{Issues found}" + "\n")
            chunk_issues = [issue for issue in (audit.get("issues") or []) if not _is_pure_typographical_issue(issue)]
            if chunk_issues:
                for issue in chunk_issues:
                    parts.append(r"\subparagraph*{" + report_latex_paragraph(f"{issue.get('title','Untitled issue')} [{issue.get('severity','low')}]") + "}" + "\n")
                    parts.append(r"\begin{itemize}" + "\n")
                    parts.append(r"\item Location: " + report_latex_paragraph(issue.get("location", "")) + "\n")
                    parts.append(r"\item Description: " + report_latex_paragraph(issue.get("description", "")) + "\n")
                    parts.append(r"\item Evidence: " + report_latex_paragraph(issue.get("evidence", "")) + "\n")
                    parts.append(r"\item Proposed fix: " + report_latex_paragraph(issue.get("proposed_fix", "")) + "\n")
                    tag_text = ", ".join(issue.get("tags", [])) if issue.get("tags") else "none"
                    parts.append(r"\item Tags: " + report_latex_paragraph(tag_text) + "\n")
                    parts.append(r"\end{itemize}" + "\n")
            elif audit.get("issues"):
                parts.append("No mathematical/correctness issues found in this chunk. Typographical/copyediting items are summarized separately in the Typographical errors section.\n")
            else:
                parts.append("No issues found.\n")

            parts.append(r"\paragraph{Next boundary hint}" + "\n")
            parts.append(report_latex_paragraph(audit.get("next_boundary_hint", "None.")) + "\n\n")

            python_entries = _python_check_entries_for_record(session, rec, audit)
            if python_entries:
                parts.append(r"\paragraph{Suggested local Python checks}" + "\n")
                for entry in python_entries:
                    parts.append(r"\textbf{" + report_latex_paragraph(entry.get("purpose", "Python check")) + "}" + "\n")
                    parts.append(report_latex_paragraph(entry.get("description", entry.get("purpose", "Python check"))) + "\n")
                    parts.append(r"\textit{Expected outcome: }" + report_latex_paragraph(entry.get("expected_outcome", "")) + "\n")
                    parts.append(r"\textit{Script file: }\texttt{" + report_latex_paragraph(entry.get("display_path", "")) + "}" + "\n")
                    parts.append(_verbatim_block(entry.get("code", "")) + "\n")

            if (audit.get("latex_patch") or "").strip():
                parts.append(r"\paragraph{Minimal LaTeX patch}" + "\n")
                parts.append(_verbatim_block(audit["latex_patch"]) + "\n")

    parts.append(r"\end{document}" + "\n")
    return _strip_unsafe_control_chars("".join(parts))

def build_final_report(session_or_pdf: dict[str, Any] | str | Path, report_title: Optional[str] = None) -> dict[str, str]:
    session = session_or_pdf if isinstance(session_or_pdf, dict) else load_session_from_pdf(session_or_pdf)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")
    root = Path(session["workdir"])
    report_stem = Path(session["pdf_path"]).stem + "_audit_report"

    md_text = build_final_report_markdown(session, report_title=report_title)
    tex_text = build_final_report_tex(session, report_title=report_title)

    reports_dir = root / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)

    md_path = reports_dir / f"{report_stem}.md"
    tex_path = reports_dir / f"{report_stem}.tex"
    json_path = reports_dir / f"{report_stem}.json"

    md_path.write_text(md_text, encoding="utf-8")
    tex_path.write_text(tex_text, encoding="utf-8")

    report_json = {
        "session": load_session_from_pdf(session["pdf_path"]),
        "status": load_status(session),
        "ledger": load_ledger(session),
        "issues": load_issues(session),
        "usage": load_usage(session),
        "manifest": load_manifest(session),
        "chunk_records": read_chunk_records(session),
        "generated_at": utc_now(),
    }
    save_json(json_path, report_json)

    return {
        "markdown": str(md_path),
        "tex": str(tex_path),
        "json": str(json_path),
    }
````

</details>

### Archived legacy notebook cell 12 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `backend examples / documentation`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original markdown cell</summary>

````markdown
## Run the automatic audit

Replace the path below with your own PDF path.

If `my_paper.tex` exists in the same folder, the notebook will use it automatically.

The notebook writes the final report both as:

- Markdown report: `..._audit_report.md`
- LaTeX report: `..._audit_report.tex`
- JSON report: `..._audit_report.json`

The `.tex` report is generated directly and is intended to compile with **TeXShop** or command-line tools such as `pdflatex`.
````

</details>

### Archived legacy notebook cell 13 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_runtime.py and audit_state.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python

# Timing patch: per-chunk elapsed time and total active audit time.

from datetime import datetime, timezone
import ast
import re
import subprocess
import sys
from audit_state import (
    _ensure_timing_state as _audit_state_ensure_timing_state,
    format_duration,
    update_usage_from_usage_obj,
)
from audit_runtime import get_audit_status

_old_create_new_session = create_new_session
_old_load_session_from_pdf = load_session_from_pdf
_old_build_final_report_markdown = build_final_report_markdown
_old_build_final_report_tex = build_final_report_tex

def _parse_iso_utc(ts: str | None):
    if not ts:
        return None
    try:
        return datetime.fromisoformat(ts)
    except Exception:
        return None

def _seconds_since(ts: str | None) -> float:
    dt = _parse_iso_utc(ts)
    if dt is None:
        return 0.0
    return max(0.0, (datetime.now(timezone.utc) - dt).total_seconds())

def _ensure_timing_state(session: dict[str, Any]) -> dict[str, Any]:
    return _audit_state_ensure_timing_state(session, default_model=DEFAULT_MODEL)

def create_new_session(
    pdf_path: str | Path,
    model: str = DEFAULT_MODEL,
    reasoning_effort: str = DEFAULT_REASONING_EFFORT,
    tex_max_chars: int = 4500,
    pdf_max_chars: int = 3500,
    store: bool = True,
    background: bool = True,
) -> dict[str, Any]:
    session = _old_create_new_session(
        pdf_path,
        model=model,
        reasoning_effort=reasoning_effort,
        tex_max_chars=tex_max_chars,
        pdf_max_chars=pdf_max_chars,
        store=store,
        background=background,
    )
    return _ensure_timing_state(session)

def load_session_from_pdf(pdf_path: str | Path) -> Optional[dict[str, Any]]:
    session = _old_load_session_from_pdf(pdf_path)
    if session is None:
        return None
    return _ensure_timing_state(session)

VERIFICATION_MODES = {"local_python_only", "code_interpreter", "none"}
CI_FAILURE_FALLBACK_MODES = {"off", "retry_local_python_only_once"}

def _dedupe_strings(seq: list[str]) -> list[str]:
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _normalize_verification_mode(mode: Optional[str]) -> str:
    aliases = {
        "local": "local_python_only",
        "python": "local_python_only",
        "local_python": "local_python_only",
        "ci": "code_interpreter",
        "code": "code_interpreter",
        "off": "none",
    }
    normalized = "local_python_only" if mode is None else str(mode).strip().lower()
    normalized = aliases.get(normalized, normalized)
    if normalized not in VERIFICATION_MODES:
        allowed = ", ".join(sorted(VERIFICATION_MODES))
        raise ValueError(f"verification_mode must be one of: {allowed}")
    return normalized


def _normalize_ci_failure_fallback_mode(mode: Optional[str]) -> str:
    aliases = {
        "none": "off",
        "false": "off",
        "disabled": "off",
        "retry": "retry_local_python_only_once",
        "local": "retry_local_python_only_once",
        "retry_local": "retry_local_python_only_once",
        "local_python_only": "retry_local_python_only_once",
    }
    normalized = "off" if mode is None else str(mode).strip().lower()
    normalized = aliases.get(normalized, normalized)
    if normalized not in CI_FAILURE_FALLBACK_MODES:
        allowed = ", ".join(sorted(CI_FAILURE_FALLBACK_MODES))
        raise ValueError(f"ci_failure_fallback_mode must be one of: {allowed}")
    return normalized


def _artifact_timestamp_token() -> str:
    return re.sub(r"[^0-9A-Za-z]+", "", utc_now())


def _save_request_metadata(
    session: dict[str, Any],
    chunk: dict[str, Any],
    request_kwargs: dict[str, Any],
    verification_mode: str,
    used_code_interpreter_tool: bool,
    code_interpreter_file_ids: Optional[list[str]] = None,
    attempt_label: Optional[str] = None,
) -> str:
    root = Path(session["workdir"])
    suffix = f"_{attempt_label}" if attempt_label else ""
    request_path = root / "requests" / f"{chunk['chunk_id']}_{_artifact_timestamp_token()}{suffix}.request.json"
    payload = {
        "time": utc_now(),
        "chunk_id": chunk.get("chunk_id"),
        "chunk_index": chunk.get("chunk_index"),
        "label": chunk.get("label"),
        "boundary": chunk.get("boundary"),
        "verification_mode": verification_mode,
        "used_code_interpreter_tool": bool(used_code_interpreter_tool),
        "code_interpreter_file_ids": list(code_interpreter_file_ids or []),
        "request": to_jsonable(request_kwargs),
    }
    save_json(request_path, payload)
    return str(request_path)


def _pause_audit_after_chunk_failure(
    session: dict[str, Any],
    chunk: dict[str, Any],
    resp,
    verification_mode: str,
    used_code_interpreter_tool: bool = False,
    request_path: Optional[str] = None,
    note: Optional[str] = None,
    discovered_during_recovery: bool = False,
) -> dict[str, Any]:
    root = Path(session["workdir"])
    chunk_id = chunk["chunk_id"]
    response_id = getattr(resp, "id", None) or f"failed_{_artifact_timestamp_token()}"
    resp_json = to_jsonable(resp)
    raw_json_path = root / "responses" / f"{chunk_id}_{response_id}.json"
    save_json(raw_json_path, resp_json)

    raw_text = (getattr(resp, "output_text", None) or "").strip()
    raw_text_path = root / "responses" / f"{chunk_id}_{response_id}.raw.txt"
    if raw_text:
        raw_text_path.write_text(raw_text, encoding="utf-8")

    tool_summary = {
        "used_code_interpreter": False,
        "tool_event_count": 0,
        "file_ids": [],
        "container_ids": [],
    }
    if verification_mode == "code_interpreter" or used_code_interpreter_tool:
        tool_summary = _extract_code_interpreter_summary(resp_json)

    failure_summary = {
        "time": utc_now(),
        "chunk_id": chunk_id,
        "chunk_index": chunk.get("chunk_index"),
        "label": chunk.get("label"),
        "boundary": chunk.get("boundary"),
        "response_id": response_id,
        "status": getattr(resp, "status", None),
        "error": resp_json.get("error"),
        "incomplete_details": resp_json.get("incomplete_details"),
        "last_error": resp_json.get("last_error"),
        "verification_mode": verification_mode,
        "used_code_interpreter_tool": bool(used_code_interpreter_tool),
        "tool_summary": tool_summary,
        "request_path": request_path,
        "raw_response_path": str(raw_json_path),
        "raw_text_path": str(raw_text_path) if raw_text else None,
        "discovered_during_recovery": bool(discovered_during_recovery),
        "note": note,
    }
    failure_path = root / "responses" / f"{chunk_id}_{response_id}.failure.json"
    save_json(failure_path, failure_summary)
    failure_summary["failure_summary_path"] = str(failure_path)
    append_jsonl(root / "logs" / "failed_chunks.jsonl", failure_summary)

    session["last_response_id"] = response_id
    session["pending"] = None
    session["next_chunk_index"] = int(chunk.get("chunk_index") or session.get("next_chunk_index") or 1)
    session["updated_at"] = utc_now()
    save_session(session)

    manifest = load_manifest(session)
    status = load_status(session)
    status.update({
        "status": "paused",
        "current_chunk_id": chunk_id,
        "chunks_total": len(manifest.get("chunks", [])),
        "estimated_pages_total": manifest.get("pdf_page_count", 0),
        "current_chunk_elapsed_seconds": 0.0,
        "audit_started_at": session.get("audit_started_at", session.get("created_at")),
        "audit_finished_at": None,
        "updated_at": utc_now(),
    })
    save_status(session, status)
    return failure_summary

def _is_ci_invalid_prompt_failure(failure_summary: Optional[dict[str, Any]]) -> bool:
    if not isinstance(failure_summary, dict):
        return False
    err = failure_summary.get("error")
    if not isinstance(err, dict):
        return False
    return str(err.get("code") or "").strip().lower() == "invalid_prompt"

def _record_ci_invalid_prompt_local_fallback(
    session: dict[str, Any],
    chunk: dict[str, Any],
    failure_summary: dict[str, Any],
    source: str,
) -> dict[str, Any]:
    root = Path(session["workdir"])
    event = {
        "time": utc_now(),
        "source": source,
        "reason": "code_interpreter_invalid_prompt",
        "chunk_id": chunk.get("chunk_id"),
        "chunk_index": chunk.get("chunk_index"),
        "label": chunk.get("label"),
        "original_response_id": failure_summary.get("response_id"),
        "original_status": failure_summary.get("status"),
        "error": failure_summary.get("error"),
        "failure_summary_path": failure_summary.get("failure_summary_path"),
        "request_path": failure_summary.get("request_path"),
        "retry_verification_mode": "local_python_only",
    }
    fallback_log_path = root / "logs" / "ci_invalid_prompt_fallbacks.jsonl"
    append_jsonl(fallback_log_path, event)
    failure_summary["auto_local_fallback_triggered"] = True
    failure_summary["auto_local_fallback_reason"] = "code_interpreter_invalid_prompt"
    failure_summary["auto_local_fallback_log_path"] = str(fallback_log_path)
    failure_path = failure_summary.get("failure_summary_path")
    if isinstance(failure_path, str) and failure_path:
        try:
            save_json(Path(failure_path), failure_summary)
        except Exception:
            pass
    session["last_ci_invalid_prompt_fallback"] = event
    session["updated_at"] = utc_now()
    save_session(session)
    return event

def _build_code_interpreter_tools(
    session: dict[str, Any],
    extra_file_ids: Optional[list[str]] = None,
    include_memory_limit: bool = True,
) -> list[dict[str, Any]]:
    container = {"type": "auto"}
    file_ids = []
    for fid in [session.get("pdf_file_id")] + list(extra_file_ids or []):
        if isinstance(fid, str) and fid and fid.startswith("file-"):
            file_ids.append(fid)
    file_ids = _dedupe_strings(file_ids)
    if file_ids:
        container["file_ids"] = file_ids
    memory_limit = session.get("code_interpreter_memory_limit")
    if include_memory_limit and memory_limit:
        container["memory_limit"] = str(memory_limit)
    return [{"type": "code_interpreter", "container": container}]

def _extract_code_interpreter_summary(resp_json: dict[str, Any]) -> dict[str, Any]:
    stack = [resp_json]
    tool_event_count = 0
    file_ids: list[str] = []
    container_ids: list[str] = []
    while stack:
        cur = stack.pop()
        if isinstance(cur, dict):
            typ = cur.get("type")
            if isinstance(typ, str) and "code_interpreter" in typ:
                tool_event_count += 1
            fid = cur.get("file_id")
            if isinstance(fid, str) and fid.startswith("file-"):
                file_ids.append(fid)
            fids = cur.get("file_ids")
            if isinstance(fids, list):
                for item in fids:
                    if isinstance(item, str) and item.startswith("file-"):
                        file_ids.append(item)
            cid = cur.get("container_id")
            if isinstance(cid, str) and cid:
                container_ids.append(cid)
            for v in cur.values():
                if isinstance(v, (dict, list)):
                    stack.append(v)
        elif isinstance(cur, list):
            for item in cur:
                if isinstance(item, (dict, list)):
                    stack.append(item)
    return {
        "used_code_interpreter": tool_event_count > 0,
        "tool_event_count": tool_event_count,
        "file_ids": _dedupe_strings(file_ids)[:20],
        "container_ids": _dedupe_strings(container_ids)[:5],
    }

def update_usage_from_response(session: dict[str, Any], chunk_id: str, resp, elapsed_seconds: float = 0.0) -> dict[str, Any]:
    usage_obj = to_jsonable(getattr(resp, "usage", {}) or {})
    return update_usage_from_usage_obj(
        session,
        chunk_id,
        usage_obj,
        model=session.get("model") or DEFAULT_MODEL,
        elapsed_seconds=elapsed_seconds,
    )

def finalize_chunk(
    session: dict[str, Any],
    chunk: dict[str, Any],
    resp,
    display_output: bool = True,
    verification_mode: str = "local_python_only",
    used_code_interpreter_tool: bool = False,
) -> dict[str, Any]:
    root = Path(session["workdir"])
    chunk_id = chunk["chunk_id"]
    verification_mode = _normalize_verification_mode(verification_mode)

    raw_json_path = root / "responses" / f"{chunk_id}_{resp.id}.json"
    resp_json = to_jsonable(resp)
    save_json(raw_json_path, resp_json)

    raw_text = (resp.output_text or "").strip()
    raw_text_path = root / "responses" / f"{chunk_id}_{resp.id}.raw.txt"
    if raw_text:
        raw_text_path.write_text(raw_text, encoding="utf-8")

    try:
        audit = _coerce_audit_payload(parse_audit_response(resp))
    except Exception as e:
        log_path = root / "logs" / "parse_failures.jsonl"
        append_jsonl(log_path, {
            "time": utc_now(),
            "chunk_id": chunk_id,
            "response_id": getattr(resp, "id", None),
            "error": repr(e),
            "raw_text_path": str(raw_text_path) if raw_text else None,
        })
        raise RuntimeError(
            f"Could not parse structured output for {chunk_id}. "
            f"See {raw_text_path.name if raw_text else raw_json_path.name} and logs/parse_failures.jsonl"
        ) from e

    audit.setdefault("label", chunk["label"])
    audit.setdefault("boundary", chunk["boundary"])
    audit.setdefault("chunk_too_large", False)
    audit.setdefault("chunk_split_suggestions", [])
    audit.setdefault("assumptions_and_notation", [])
    audit.setdefault("verified_steps", [])
    audit.setdefault("issues", [])
    audit.setdefault("python_checks", [])
    audit.setdefault("latex_patch", "")
    audit.setdefault("ledger_updates", {"assumptions": [], "notes": []})
    audit.setdefault("next_boundary_hint", "")
    audit.setdefault("confidence", "medium")

    structured_path = root / "responses" / f"{chunk_id}.structured.json"
    save_json(structured_path, audit)

    md_text = render_audit_markdown(audit)
    md_path = root / "responses" / f"{chunk_id}.md"
    md_path.write_text(md_text, encoding="utf-8")

    verification_summary = {
        "used_code_interpreter": False,
        "tool_event_count": 0,
        "file_ids": [],
        "container_ids": [],
    }
    verification_path = None
    if verification_mode == "code_interpreter" or used_code_interpreter_tool:
        verification_summary = _extract_code_interpreter_summary(resp_json)
        verification_payload = {
            "time": utc_now(),
            "chunk_id": chunk_id,
            "response_id": getattr(resp, "id", None),
            "verification_mode": verification_mode,
            "used_code_interpreter_tool": bool(used_code_interpreter_tool),
            "summary": verification_summary,
        }
        verification_path = root / "responses" / f"{chunk_id}_{resp.id}.verification.json"
        save_json(verification_path, verification_payload)
        if session.get("reuse_code_interpreter_container") and verification_summary.get("container_ids"):
            session["code_interpreter_container_id"] = verification_summary["container_ids"][0]

    extracted = save_patch_and_code_files(session, chunk_id, audit)
    created_issues = add_issues_from_audit(session, chunk_id, audit.get("issues", []))
    update_ledger_from_audit(session, audit)

    pending = session.get("pending") or {}
    elapsed_seconds = _seconds_since(pending.get("started_at") or pending.get("created_at"))
    usage_update = update_usage_from_response(session, chunk_id, resp, elapsed_seconds=elapsed_seconds)

    record = {
        "time": utc_now(),
        "chunk_id": chunk_id,
        "chunk_index": chunk["chunk_index"],
        "label": chunk["label"],
        "boundary": chunk["boundary"],
        "source_kind": chunk["source_kind"],
        "page_start": chunk["page_start"],
        "page_end": chunk["page_end"],
        "paper_progress_end": chunk["paper_progress_end"],
        "response_id": resp.id,
        "request_path": pending.get("request_path"),
        "raw_response_path": str(raw_json_path),
        "structured_response_path": str(structured_path),
        "markdown_path": str(md_path),
        "latex_paths": extracted["latex_paths"],
        "python_paths": extracted["python_paths"],
        "issue_ids": [x["issue_id"] for x in created_issues],
        "cost_usd": usage_update["cost"]["total_cost"],
        "usage": usage_update["usage"],
        "elapsed_seconds": float(elapsed_seconds or 0.0),
        "verification_mode": verification_mode,
        "verification_summary": verification_summary,
        "verification_path": str(verification_path) if verification_path else None,
    }
    append_jsonl(session_paths(session["workdir"])["chunk_records"], record)

    latest_session = load_session_from_pdf(session["pdf_path"]) or session
    if session.get("code_interpreter_container_id"):
        latest_session["code_interpreter_container_id"] = session.get("code_interpreter_container_id")
    latest_session["last_response_id"] = resp.id
    latest_session["next_chunk_index"] = chunk["chunk_index"] + 1
    latest_session["pending"] = None
    latest_session["updated_at"] = utc_now()
    latest_session["pdf_attached_in_conversation"] = True
    latest_session["developer_prompt_seeded"] = True
    if chunk["chunk_index"] >= len(load_manifest(latest_session)["chunks"]):
        latest_session["audit_finished_at"] = utc_now()
    save_session(latest_session)
    session = latest_session

    manifest = load_manifest(session)
    usage_state = usage_update["usage_state"]
    status = load_status(session)
    status.update({
        "status": "running" if chunk["chunk_index"] < len(manifest["chunks"]) else "completed",
        "progress_pct": round(100.0 * chunk["paper_progress_end"], 1),
        "current_chunk_id": chunk_id,
        "chunks_completed": chunk["chunk_index"],
        "chunks_total": len(manifest["chunks"]),
        "estimated_pages_completed": chunk["page_end"],
        "estimated_pages_total": manifest["pdf_page_count"],
        "cost_usd": usage_state["totals"]["cost_usd"],
        "total_audit_seconds": float(usage_state["totals"].get("audit_seconds", 0.0) or 0.0),
        "current_chunk_elapsed_seconds": 0.0,
        "audit_started_at": session.get("audit_started_at", session.get("created_at")),
        "audit_finished_at": session.get("audit_finished_at"),
    })
    if chunk["chunk_index"] >= len(manifest["chunks"]):
        status["status"] = "completed"
        status["progress_pct"] = 100.0
        status["estimated_pages_completed"] = manifest["pdf_page_count"]
    save_status(session, status)

    if display_output:
        print(
            f"[{chunk_id}] Progress: {status['progress_pct']:.1f}% | "
            f"Pages: {status['estimated_pages_completed']}/{status['estimated_pages_total']} | "
            f"Chunk cost: ${record['cost_usd']:.4f} | "
            f"Chunk time: {format_duration(record['elapsed_seconds'])} | "
            f"Cumulative cost: ${usage_state['totals']['cost_usd']:.4f} | "
            f"Total audit time: {format_duration(usage_state['totals'].get('audit_seconds', 0.0))} | "
            f"Total tokens: {usage_state['totals']['total_tokens']}"
        )
        if verification_mode != "local_python_only" or verification_summary.get("used_code_interpreter"):
            print(
                f"[{chunk_id}] Verification mode: {verification_mode} | "
                f"CI tool events: {verification_summary.get('tool_event_count', 0)}"
            )
        display_audit(audit)

    return {"audit": audit, "record": record}

def process_one_chunk(
    session: dict[str, Any],
    chunk: dict[str, Any],
    poll_every: float = 3.0,
    max_wait_seconds: Optional[float] = None,
    display_output: bool = True,
    verification_mode: str = "local_python_only",
    code_interpreter_file_ids: Optional[list[str]] = None,
    allow_ci_failure_fallback: bool = True,
) -> dict[str, Any]:
    verification_mode = _normalize_verification_mode(verification_mode)
    if verification_mode == "code_interpreter" and not session.get("use_code_interpreter", False):
        raise RuntimeError(
            "verification_mode='code_interpreter' requested, but this session has use_code_interpreter=False. "
            "Enable use_code_interpreter when creating the session or in audit_the_paper(...)."
        )

    user_input = build_user_message_for_chunk(session, chunk)
    if not session.get("developer_prompt_seeded", False):
        input_payload = [{"role": "developer", "content": [{"type": "input_text", "text": AUDIT_SYSTEM_PROMPT}]}] + user_input
    else:
        input_payload = user_input

    verification_instruction = ""
    if verification_mode == "code_interpreter":
        verification_instruction = (
            "Verification mode for this chunk: code_interpreter. "
            "Use the code_interpreter tool when numeric/symbolic verification is needed. "
            "Keep python_checks concise local fallbacks when useful."
        )
    elif verification_mode == "none":
        verification_instruction = (
            "Verification mode for this chunk: none. "
            "Skip optional numeric/symbolic code verification and keep python_checks empty unless strictly necessary."
        )
    if verification_instruction and input_payload:
        try:
            content = input_payload[-1].get("content")
            if isinstance(content, list):
                content.append({"type": "input_text", "text": verification_instruction})
        except Exception:
            pass

    prompt_path = Path(session["workdir"]) / "prompts" / f"{chunk['chunk_id']}_prompt.json"
    save_json(prompt_path, input_payload)

    chunk_started_at = utc_now()
    request_kwargs = {
        "model": session["model"],
        "reasoning": {"effort": session["reasoning_effort"]},
        "conversation": session["conversation_id"],
        "input": input_payload,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "math_audit",
                "strict": True,
                "schema": AUDIT_RESPONSE_SCHEMA,
            }
        },
        "background": session["background"],
        "store": session["store"],
    }
    used_code_interpreter_tool = False
    if verification_mode == "code_interpreter":
        request_kwargs["tools"] = _build_code_interpreter_tools(
            session,
            extra_file_ids=code_interpreter_file_ids,
            include_memory_limit=True,
        )
        used_code_interpreter_tool = True
    request_path = _save_request_metadata(
        session,
        chunk,
        request_kwargs,
        verification_mode=verification_mode,
        used_code_interpreter_tool=used_code_interpreter_tool,
        code_interpreter_file_ids=code_interpreter_file_ids,
        attempt_label="initial",
    )
    try:
        resp = client.responses.create(**request_kwargs)
    except Exception as e:
        if used_code_interpreter_tool and "memory" in str(e).lower():
            request_kwargs["tools"] = _build_code_interpreter_tools(
                session,
                extra_file_ids=code_interpreter_file_ids,
                include_memory_limit=False,
            )
            request_path = _save_request_metadata(
                session,
                chunk,
                request_kwargs,
                verification_mode=verification_mode,
                used_code_interpreter_tool=used_code_interpreter_tool,
                code_interpreter_file_ids=code_interpreter_file_ids,
                attempt_label="memory_retry",
            )
            resp = client.responses.create(**request_kwargs)
        else:
            raise

    session["pending"] = {
        "chunk_id": chunk["chunk_id"],
        "response_id": resp.id,
        "created_at": chunk_started_at,
        "started_at": chunk_started_at,
        "verification_mode": verification_mode,
        "used_code_interpreter_tool": used_code_interpreter_tool,
        "request_path": request_path,
    }
    session["updated_at"] = utc_now()
    save_session(session)

    manifest = load_manifest(session)
    status = load_status(session)
    status.update({
        "status": "running",
        "current_chunk_id": chunk["chunk_id"],
        "chunks_total": len(manifest["chunks"]),
        "estimated_pages_total": manifest["pdf_page_count"],
        "current_chunk_elapsed_seconds": 0.0,
        "total_audit_seconds": float(load_usage(session)["totals"].get("audit_seconds", 0.0) or 0.0),
        "audit_started_at": session.get("audit_started_at", session.get("created_at")),
    })
    save_status(session, status)

    if getattr(resp, "status", None) in WORKING_STATUSES:
        resp = wait_for_response(resp.id, poll_every=poll_every, max_wait_seconds=max_wait_seconds)

    if getattr(resp, "status", None) != "completed":
        failure_summary = _pause_audit_after_chunk_failure(
            session,
            chunk,
            resp,
            verification_mode=verification_mode,
            used_code_interpreter_tool=used_code_interpreter_tool,
            request_path=request_path,
            note="Responses API returned a terminal non-completed status during process_one_chunk.",
        )
        if (
            allow_ci_failure_fallback
            and verification_mode == "code_interpreter"
            and used_code_interpreter_tool
            and _is_ci_invalid_prompt_failure(failure_summary)
        ):
            _record_ci_invalid_prompt_local_fallback(
                session,
                chunk,
                failure_summary,
                source="process_one_chunk",
            )
            if display_output:
                print(
                    f"[{chunk['chunk_id']}] Code Interpreter chunk failed with error.code=invalid_prompt. "
                    "Retrying once with verification_mode='local_python_only'."
                )
            return process_one_chunk(
                session,
                chunk,
                poll_every=poll_every,
                max_wait_seconds=max_wait_seconds,
                display_output=display_output,
                verification_mode="local_python_only",
                code_interpreter_file_ids=code_interpreter_file_ids,
                allow_ci_failure_fallback=False,
            )
        fallback_mode = _normalize_ci_failure_fallback_mode(session.get("ci_failure_fallback_mode", "off"))
        if (
            allow_ci_failure_fallback
            and verification_mode == "code_interpreter"
            and used_code_interpreter_tool
            and fallback_mode == "retry_local_python_only_once"
        ):
            if display_output:
                print(
                    f"[{chunk['chunk_id']}] Code Interpreter chunk failed with status={getattr(resp, 'status', None)}. "
                    "Retrying once with verification_mode='local_python_only'."
                )
            return process_one_chunk(
                session,
                chunk,
                poll_every=poll_every,
                max_wait_seconds=max_wait_seconds,
                display_output=display_output,
                verification_mode="local_python_only",
                code_interpreter_file_ids=code_interpreter_file_ids,
                allow_ci_failure_fallback=False,
            )
        raise RuntimeError(
            f"Chunk {chunk['chunk_id']} ended with status={getattr(resp, 'status', None)}. "
            f"See {failure_summary.get('failure_summary_path')} and logs/failed_chunks.jsonl"
        )

    return finalize_chunk(
        session,
        chunk,
        resp,
        display_output=display_output,
        verification_mode=verification_mode,
        used_code_interpreter_tool=used_code_interpreter_tool,
    )

def recover_pending_chunk(
    session: dict[str, Any],
    poll_every: float = 3.0,
    max_wait_seconds: Optional[float] = None,
    display_output: bool = True,
    allow_ci_failure_fallback: bool = True,
) -> Optional[dict[str, Any]]:
    pending = session.get("pending")
    if not pending:
        return None

    response_id = pending.get("response_id")
    chunk_id = pending.get("chunk_id")
    if not response_id or not chunk_id:
        return None

    if "started_at" not in pending:
        pending["started_at"] = pending.get("created_at", utc_now())
        session["pending"] = pending
        save_session(session)

    manifest = load_manifest(session)
    chunks = manifest.get("chunks", [])
    matches = [c for c in chunks if c["chunk_id"] == chunk_id]
    if not matches:
        return None
    chunk = matches[0]

    structured_path = Path(session["workdir"]) / "responses" / f"{chunk_id}.structured.json"
    if structured_path.exists():
        session["pending"] = None
        if session.get("next_chunk_index", 1) <= chunk["chunk_index"]:
            session["next_chunk_index"] = chunk["chunk_index"] + 1
        session["updated_at"] = utc_now()
        save_session(session)
        return {"recovered": False, "skipped": True, "chunk_id": chunk_id}

    status = load_status(session)
    status["current_chunk_id"] = chunk_id
    status["current_chunk_elapsed_seconds"] = _seconds_since(pending.get("started_at"))
    save_status(session, status)

    try:
        resp = client.responses.retrieve(response_id)
    except Exception as e:
        session["pending"] = None
        session["updated_at"] = utc_now()
        save_session(session)
        status = load_status(session)
        status["status"] = "paused"
        status["current_chunk_id"] = None
        status["current_chunk_elapsed_seconds"] = 0.0
        save_status(session, status)
        return {"recovered": False, "skipped": True, "chunk_id": chunk_id, "error": repr(e)}

    if getattr(resp, "status", None) in WORKING_STATUSES:
        resp = wait_for_response(response_id, poll_every=poll_every, max_wait_seconds=max_wait_seconds)

    if getattr(resp, "status", None) != "completed":
        verification_mode = _normalize_verification_mode((pending or {}).get("verification_mode", "local_python_only"))
        used_code_interpreter_tool = bool((pending or {}).get("used_code_interpreter_tool", False))
        failure_summary = _pause_audit_after_chunk_failure(
            session,
            chunk,
            resp,
            verification_mode=verification_mode,
            used_code_interpreter_tool=used_code_interpreter_tool,
            request_path=(pending or {}).get("request_path"),
            note="Responses API returned a terminal non-completed status during recovery of an earlier submitted request.",
            discovered_during_recovery=True,
        )
        if (
            allow_ci_failure_fallback
            and verification_mode == "code_interpreter"
            and used_code_interpreter_tool
            and _is_ci_invalid_prompt_failure(failure_summary)
        ):
            _record_ci_invalid_prompt_local_fallback(
                session,
                chunk,
                failure_summary,
                source="recover_pending_chunk",
            )
            if display_output:
                print(
                    f"[{chunk_id}] Recovered Code Interpreter chunk failed with error.code=invalid_prompt. "
                    "Retrying once with verification_mode='local_python_only'."
                )
            return process_one_chunk(
                session,
                chunk,
                poll_every=poll_every,
                max_wait_seconds=max_wait_seconds,
                display_output=display_output,
                verification_mode="local_python_only",
                allow_ci_failure_fallback=False,
            )
        fallback_mode = _normalize_ci_failure_fallback_mode(session.get("ci_failure_fallback_mode", "off"))
        if (
            allow_ci_failure_fallback
            and verification_mode == "code_interpreter"
            and used_code_interpreter_tool
            and fallback_mode == "retry_local_python_only_once"
        ):
            if display_output:
                print(
                    f"[{chunk_id}] Recovered Code Interpreter chunk failed with status={getattr(resp, 'status', None)}. "
                    "Retrying once with verification_mode='local_python_only'."
                )
            return process_one_chunk(
                session,
                chunk,
                poll_every=poll_every,
                max_wait_seconds=max_wait_seconds,
                display_output=display_output,
                verification_mode="local_python_only",
                allow_ci_failure_fallback=False,
            )
        if display_output:
            print(
                f"[{chunk_id}] A previously submitted request ended with status={getattr(resp, 'status', None)} during recovery. "
                f"Saved failure details to {failure_summary.get('failure_summary_path')}."
            )
        return {
            "recovered": False,
            "paused": True,
            "chunk_id": chunk_id,
            "status": getattr(resp, "status", None),
            "failure_summary_path": failure_summary.get("failure_summary_path"),
            "failure_summary": failure_summary,
        }

    verification_mode = _normalize_verification_mode((pending or {}).get("verification_mode", "local_python_only"))
    used_code_interpreter_tool = bool((pending or {}).get("used_code_interpreter_tool", False))
    return finalize_chunk(
        session,
        chunk,
        resp,
        display_output=display_output,
        verification_mode=verification_mode,
        used_code_interpreter_tool=used_code_interpreter_tool,
    )

def show_audit_status(pdf_path: str | Path) -> dict[str, Any]:
    info = get_audit_status(pdf_path)
    session = info["session"]
    status = info["status"]
    usage = info["usage"]

    current_chunk_elapsed = 0.0
    if session.get("pending"):
        current_chunk_elapsed = _seconds_since((session["pending"] or {}).get("started_at") or (session["pending"] or {}).get("created_at"))
        status["current_chunk_elapsed_seconds"] = current_chunk_elapsed
        save_status(session, status)

    print(f"Status: {status['status']}")
    print(f"Progress: {status['progress_pct']}%")
    print(f"Chunks: {status['chunks_completed']} / {status['chunks_total']}")
    print(f"Estimated pages: {status['estimated_pages_completed']} / {status['estimated_pages_total']}")
    print(f"Cumulative cost: ${status['cost_usd']:.4f}")
    print(f"Total tokens: {usage['totals']['total_tokens']}")
    print(f"Total audit time: {format_duration(usage['totals'].get('audit_seconds', 0.0))}")
    if status.get("current_chunk_id") and status["status"] == "running":
        print(f"Current chunk: {status['current_chunk_id']}")
        print(f"Current chunk elapsed: {format_duration(current_chunk_elapsed)}")
    return info

def _timing_summary_markdown(session: dict[str, Any]) -> str:
    usage = load_usage(session)
    lines = [
        "## Timing summary",
        "",
        f"- Total active audit time: {format_duration(usage['totals'].get('audit_seconds', 0.0))}",
        "",
    ]
    per_chunk = usage.get("per_chunk", [])
    if per_chunk:
        for entry in per_chunk:
            lines.append(f"- {entry.get('chunk_id','chunk')}: {format_duration(entry.get('elapsed_seconds', 0.0))}")
    else:
        lines.append("- No completed chunks yet.")
    lines.append("")
    return "\n".join(lines)

def build_final_report_markdown(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    _ensure_timing_state(session)
    base = _old_build_final_report_markdown(session, report_title=report_title)
    total_line = f"- Total active audit time: {format_duration(load_usage(session)['totals'].get('audit_seconds', 0.0))}"
    if "- Total tokens:" in base and total_line not in base:
        base = base.replace("- Total tokens: " + str(load_usage(session)["totals"]["total_tokens"]),
                            "- Total tokens: " + str(load_usage(session)["totals"]["total_tokens"]) + "\n" + total_line,
                            1)
    return base.rstrip() + "\n\n" + _timing_summary_markdown(session)

def _timing_summary_tex(session: dict[str, Any]) -> str:
    usage = load_usage(session)
    parts = [
        r"\section*{Timing summary}",
        r"\begin{itemize}",
        r"\item Total active audit time: " + report_latex_paragraph(format_duration(usage["totals"].get("audit_seconds", 0.0))),
    ]
    per_chunk = usage.get("per_chunk", [])
    if per_chunk:
        for entry in per_chunk:
            parts.append(r"\item " + report_latex_paragraph(f"{entry.get('chunk_id','chunk')}: {format_duration(entry.get('elapsed_seconds', 0.0))}"))
    else:
        parts.append(r"\item No completed chunks yet.")
    parts.append(r"\end{itemize}")
    return "\n".join(parts) + "\n"

def build_final_report_tex(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    _ensure_timing_state(session)
    base = _old_build_final_report_tex(session, report_title=report_title)
    usage = load_usage(session)
    total_line = r"\item Total active audit time: " + report_latex_paragraph(format_duration(usage["totals"].get("audit_seconds", 0.0)))
    if total_line not in base:
        m = re.search(r"(\\item Total tokens: .*?\n)", base)
        if m:
            base = base[:m.end()] + total_line + "\n" + base[m.end():]
        else:
            base = base.replace(r"\end{document}", total_line + "\n" + r"\end{document}", 1)
    timing_section = _timing_summary_tex(session)
    if r"\end{document}" in base:
        base = base.replace(r"\end{document}", timing_section + r"\end{document}", 1)
    else:
        base = base + "\n" + timing_section
    return base

def build_final_report(session_or_pdf: dict[str, Any] | str | Path, report_title: Optional[str] = None) -> dict[str, str]:
    session = session_or_pdf if isinstance(session_or_pdf, dict) else load_session_from_pdf(session_or_pdf)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")
    _ensure_timing_state(session)
    root = Path(session["workdir"])
    report_stem = Path(session["pdf_path"]).stem + "_audit_report"

    md_text = build_final_report_markdown(session, report_title=report_title)
    tex_text = build_final_report_tex(session, report_title=report_title)

    reports_dir = root / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)

    md_path = reports_dir / f"{report_stem}.md"
    tex_path = reports_dir / f"{report_stem}.tex"
    json_path = reports_dir / f"{report_stem}.json"

    md_path.write_text(md_text, encoding="utf-8")
    tex_path.write_text(tex_text, encoding="utf-8")

    report_json = {
        "session": load_session_from_pdf(session["pdf_path"]),
        "status": load_status(session),
        "ledger": load_ledger(session),
        "issues": load_issues(session),
        "usage": load_usage(session),
        "manifest": load_manifest(session),
        "chunk_records": read_chunk_records(session),
        "generated_at": utc_now(),
    }
    save_json(json_path, report_json)

    return {
        "markdown": str(md_path),
        "tex": str(tex_path),
        "json": str(json_path),
    }
````

</details>

### Archived legacy notebook cell 14 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `backend examples / documentation`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Example:
# result = audit_the_paper(
#     "/absolute/path/to/my_paper.pdf",
#     model="gpt-5.5",
#     reasoning_effort="medium",
#     continue_existing=False,
# )
# result
````

</details>

### Archived legacy notebook cell 15 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `backend status/report helpers`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original markdown cell</summary>

````markdown
## Inspect progress or build the report again
````

</details>

### Archived legacy notebook cell 16 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `backend examples / documentation`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Example:
# show_audit_status("/absolute/path/to/my_paper.pdf")
# show_open_issues("/absolute/path/to/my_paper.pdf")
# paths = build_final_report("/absolute/path/to/my_paper.pdf")
# print(paths["tex"])
````

</details>

### Archived legacy notebook cell 17 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_policy_hooks.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Strong numbering-enforcement patch: compiled PDF numbering must win over source-local numbers.
# Important: for already-saved chunk audits, rebuilding the final report alone cannot invent new numbers
# if the older chunk text already hard-coded source-local references like "(1.1)".
# Use reset_audit_from_chunk(...) and rerun the affected chunks after this patch is active.

import json
import re
from pathlib import Path

_OLD_BUILD_USER_MESSAGE_FOR_NUMBERING_STRICT = build_user_message_for_chunk

REFERENCE_MENTION_STYLES = {"auto", "compiled_pdf_numbers", "source_labels"}
_PDF_REFERENCE_PAGE_CACHE: dict[str, list[str]] = {}
_VISIBLE_NUMBERED_OBJECT_RE = re.compile(
    r"\b(Theorem|Lemma|Proposition|Corollary|Definition|Remark|Section|Figure|Table)\s+(\d+(?:\.\d+)*)\b"
)
_EQUATION_NUMBER_ONLY_RE = re.compile(r"^\((\d+(?:\.\d+)*)\)$")
_EQUATION_NUMBER_LINE_END_RE = re.compile(r"\((\d+(?:\.\d+)*)\)\s*$")


def _normalize_reference_mention_style(style: Optional[str]) -> str:
    aliases = {
        "compiled": "compiled_pdf_numbers",
        "numbers": "compiled_pdf_numbers",
        "pdf": "compiled_pdf_numbers",
        "pdf_numbers": "compiled_pdf_numbers",
        "label": "source_labels",
        "labels": "source_labels",
        "source": "source_labels",
    }
    normalized = "auto" if style is None else str(style).strip().lower()
    normalized = aliases.get(normalized, normalized)
    if normalized not in REFERENCE_MENTION_STYLES:
        allowed = ", ".join(sorted(REFERENCE_MENTION_STYLES))
        raise ValueError(f"reference_mention_style must be one of: {allowed}")
    return normalized




def _normalize_report_reference_style(style: Optional[str]) -> str:
    aliases = {
        "match": "match_audit",
        "audit": "match_audit",
        "compiled": "compiled_pdf_numbers",
        "numbers": "compiled_pdf_numbers",
        "pdf": "compiled_pdf_numbers",
        "pdf_numbers": "compiled_pdf_numbers",
        "label": "source_labels",
        "labels": "source_labels",
        "source": "source_labels",
    }
    normalized = "match_audit" if style is None else str(style).strip().lower()
    normalized = aliases.get(normalized, normalized)
    allowed = {"match_audit", "compiled_pdf_numbers", "source_labels"}
    if normalized not in allowed:
        choices = ", ".join(sorted(allowed))
        raise ValueError(f"report_reference_style must be one of: {choices}")
    return normalized

def _extract_chunk_labels(text: str) -> list[str]:
    text = text or ""
    patterns = [
        r"\\label\{([^}]+)\}",
        r"\\(?:eqref|ref|autoref|cref|Cref)\{([^}]+)\}",
    ]
    labels: list[str] = []
    for pat in patterns:
        labels.extend(re.findall(pat, text))
    return _dedupe_preserve_order([x.strip() for x in labels if x and x.strip()])


def _infer_kind_from_label(label: str) -> str:
    lab = (label or "").lower()
    if lab.startswith(("eq:", "equation:", "equ:")):
        return "equation"
    if lab.startswith(("thm:", "theorem:")):
        return "theorem"
    if lab.startswith(("lem:", "lemma:")):
        return "lemma"
    if lab.startswith(("prop:", "proposition:")):
        return "proposition"
    if lab.startswith(("cor:", "corollary:")):
        return "corollary"
    if lab.startswith(("sec:", "section:")):
        return "section"
    if lab.startswith(("fig:", "figure:")):
        return "figure"
    if lab.startswith(("tab:", "table:")):
        return "table"
    return "item"


def _display_for_kind(kind: str, num: str, label: str) -> str:
    k = (kind or "").strip().lower() or _infer_kind_from_label(label)
    n = (num or "").strip()
    if not n:
        return ""
    if k == "equation":
        return f"equation ({n})"
    if k in {"theorem", "lemma", "proposition", "corollary", "section", "figure", "table"}:
        return f"{k.title()} {n}"
    return n


def _aux_read_braced_group(text: str, start: int) -> tuple[str, int]:
    if start >= len(text) or text[start] != "{":
        raise ValueError("Expected '{' while parsing AUX.")
    depth = 0
    chars = []
    i = start
    while i < len(text):
        ch = text[i]
        if ch == "{":
            if depth > 0:
                chars.append(ch)
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth < 0:
                raise ValueError("Unbalanced braces while parsing AUX.")
            if depth == 0:
                return "".join(chars), i + 1
            chars.append(ch)
        else:
            chars.append(ch)
        i += 1
    raise ValueError("Unterminated brace group while parsing AUX.")


def _aux_top_level_groups(text: str) -> list[str]:
    groups = []
    i = 0
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text) or text[i] != "{":
            break
        group, i = _aux_read_braced_group(text, i)
        groups.append(group)
    return groups


def _infer_kind_from_aux_anchor(anchor: str, label: str) -> str:
    prefix = (anchor or "").strip().split(".", 1)[0].lower()
    aliases = {
        "equation": "equation",
        "section": "section",
        "subsection": "section",
        "subsubsection": "section",
        "chapter": "section",
        "appendix": "section",
        "theorem": "theorem",
        "thm": "theorem",
        "lemma": "lemma",
        "lem": "lemma",
        "proposition": "proposition",
        "prop": "proposition",
        "corollary": "corollary",
        "cor": "corollary",
        "figure": "figure",
        "fig": "figure",
        "table": "table",
        "tab": "table",
    }
    return aliases.get(prefix, _infer_kind_from_label(label))


def _load_aux_label_map(aux_path: str | Path) -> dict[str, dict[str, str]]:
    aux_path = Path(aux_path)
    try:
        text = aux_path.read_text(encoding="utf-8")
    except Exception:
        text = aux_path.read_text(encoding="latin-1")

    label_map: dict[str, dict[str, str]] = {}
    needle = r"\newlabel"
    i = 0
    while True:
        idx = text.find(needle, i)
        if idx < 0:
            break
        j = idx + len(needle)
        while j < len(text) and text[j].isspace():
            j += 1
        if j >= len(text) or text[j] != "{":
            i = j
            continue
        label, j = _aux_read_braced_group(text, j)
        while j < len(text) and text[j].isspace():
            j += 1
        if j >= len(text) or text[j] != "{":
            i = j
            continue
        payload, j = _aux_read_braced_group(text, j)
        i = j

        label = (label or "").strip()
        if not label or "@cref" in label.lower():
            continue

        groups = _aux_top_level_groups(payload)
        if not groups:
            continue
        number = (groups[0] or "").strip()
        page = (groups[1] or "").strip() if len(groups) > 1 else ""
        anchor = (groups[3] or "").strip() if len(groups) > 3 else ""
        kind = _infer_kind_from_aux_anchor(anchor, label)
        entry = {
            "number": number,
            "kind": kind,
            "display": _display_for_kind(kind, number, label),
        }
        if page:
            entry["page"] = page
        if anchor:
            entry["anchor"] = anchor
        label_map[label] = entry

    return label_map


def _load_pdf_pages_for_reference_hints(pdf_path: str | Path) -> list[str]:
    key = str(Path(pdf_path).expanduser().resolve())
    pages = _PDF_REFERENCE_PAGE_CACHE.get(key)
    if pages is None:
        pages = load_pdf_pages(key)
        _PDF_REFERENCE_PAGE_CACHE[key] = pages
    return pages


def _extract_visible_pdf_reference_hints_from_pages(
    page_texts: list[str],
    start_page: int,
    max_items: int = 24,
) -> list[str]:
    rows: list[str] = []
    seen: set[tuple[str, str, int]] = set()
    for offset, raw_page in enumerate(page_texts, start=0):
        page_no = start_page + offset
        page_text = raw_page or ""
        for m in _VISIBLE_NUMBERED_OBJECT_RE.finditer(page_text):
            kind = m.group(1).strip().lower()
            number = m.group(2).strip()
            display = _display_for_kind(kind, number, f"{kind}:{number}") or f"{m.group(1)} {number}"
            key = (kind, number, page_no)
            if key in seen:
                continue
            seen.add(key)
            rows.append(f"- page {page_no}: visible {display}")
            if len(rows) >= max_items:
                return rows
        for line in page_text.splitlines():
            line = (line or "").strip()
            if not line:
                continue
            m = _EQUATION_NUMBER_ONLY_RE.fullmatch(line)
            if m:
                number = m.group(1)
                key = ("equation", number, page_no)
                if key in seen:
                    continue
                seen.add(key)
                rows.append(f"- page {page_no}: standalone displayed equation number equation ({number})")
                if len(rows) >= max_items:
                    return rows
                continue
            m = _EQUATION_NUMBER_LINE_END_RE.search(line)
            if not m:
                continue
            number = m.group(1)
            key = ("equation", number, page_no)
            if key in seen:
                continue
            seen.add(key)
            snippet = normalize_whitespace(line[:m.start()].strip())
            if snippet:
                rows.append(
                    f"- page {page_no}: line ending with equation ({number}) near '{_truncate_text(snippet, limit=90)}'"
                )
            else:
                rows.append(f"- page {page_no}: line ending with equation ({number})")
            if len(rows) >= max_items:
                return rows
    return rows


def _pdf_reference_context_for_chunk(session: dict[str, Any], chunk: dict[str, Any], max_items: int = 24) -> str:
    try:
        pages = _load_pdf_pages_for_reference_hints(session["pdf_path"])
    except Exception:
        return "No reliable visible numbering hints were extracted from the PDF for this chunk."
    page_start = max(1, int(chunk.get("page_start") or 1))
    page_end = max(page_start, int(chunk.get("page_end") or page_start))
    page_texts = pages[page_start - 1:page_end]
    rows = _extract_visible_pdf_reference_hints_from_pages(page_texts, start_page=page_start, max_items=max_items)
    if not rows:
        return "No reliable visible numbering hints were extracted from the PDF pages for this chunk."
    return "\n".join(rows)


def _reference_label_rows(labels: list[str], label_map: dict[str, dict[str, str]]) -> list[str]:
    rows = []
    for lab in labels:
        info = label_map.get(lab) or {}
        kind = (info.get("kind") or _infer_kind_from_label(lab)).strip() or "item"
        rows.append(f"- {lab} -> cite as 'the {kind} labeled {lab}'")
    return rows


def _reference_number_rows(labels: list[str], label_map: dict[str, dict[str, str]]) -> list[str]:
    rows = []
    for lab in labels:
        info = label_map.get(lab) or {}
        disp = (info.get("display") or "").strip()
        kind = (info.get("kind") or _infer_kind_from_label(lab)).strip() or "item"
        num = (info.get("number") or "").strip()
        if disp and num:
            rows.append(f"- {lab} -> {disp}")
        elif disp:
            rows.append(f"- {lab} -> {disp} (compiled number unavailable; cite by label if needed)")
        else:
            rows.append(f"- {lab} -> compiled number unavailable; cite as 'the {kind} labeled {lab}'")
    return rows


def ensure_reference_map(session: dict[str, Any]) -> dict[str, Any]:
    root = Path(session["workdir"])
    ref_path = root / "state" / "reference_map.json"
    tex_path = session.get("tex_path")
    aux_path = Path(tex_path).with_suffix(".aux") if tex_path else None
    aux_exists = bool(aux_path and aux_path.exists())

    cached: dict[str, Any] = {}
    if ref_path.exists():
        try:
            cached_obj = load_json(ref_path)
            if isinstance(cached_obj, dict):
                cached = dict(cached_obj)
        except Exception as e:
            cached = {"warning": f"Could not read cached reference_map.json: {e}"}

    if aux_exists and aux_path is not None:
        try:
            label_map = _load_aux_label_map(aux_path)
            ref_state = {
                "label_map": label_map,
                "source_aux_path": str(aux_path),
                "map_source": "aux" if label_map else "aux_empty",
                "updated_at": utc_now(),
            }
            if not label_map:
                ref_state["warning"] = f"AUX file {aux_path.name} was parsed but no labels were recovered."
            save_json(ref_path, ref_state)
            return ref_state
        except Exception as e:
            cached_map = cached.get("label_map") if isinstance(cached.get("label_map"), dict) else {}
            ref_state = {
                "label_map": cached_map or {},
                "source_aux_path": str(aux_path),
                "map_source": "cached_after_aux_error" if cached_map else "aux_error",
                "updated_at": utc_now(),
                "warning": (
                    f"Failed to parse AUX reference map from {aux_path.name}: {e}. "
                    + ("Using cached reference map fallback." if cached_map else "No cached reference map fallback is available.")
                ),
            }
            save_json(ref_path, ref_state)
            return ref_state

    if isinstance(cached.get("label_map"), dict):
        cached.setdefault("source_aux_path", str(aux_path) if aux_path and aux_path.exists() else cached.get("source_aux_path"))
        cached.setdefault("map_source", "cached" if cached.get("label_map") else "none")
        cached["updated_at"] = utc_now()
        save_json(ref_path, cached)
        return cached

    ref_state = {
        "label_map": {},
        "source_aux_path": str(aux_path) if aux_path and aux_path.exists() else None,
        "map_source": "none",
        "updated_at": utc_now(),
    }
    save_json(ref_path, ref_state)
    return ref_state


def _reference_map_has_valid_aux_numbers(ref_state: Any) -> bool:
    if not isinstance(ref_state, dict):
        return False
    label_map = ref_state.get("label_map")
    if not isinstance(label_map, dict) or not label_map:
        return False
    map_source = str(ref_state.get("map_source") or "").strip().lower()
    source_aux_path = ref_state.get("source_aux_path")
    return map_source == "aux" or (not map_source and bool(source_aux_path))


def _effective_reference_mention_style(
    session: dict[str, Any],
    ref_state: Optional[dict[str, Any]] = None,
) -> str:
    style = _normalize_reference_mention_style(session.get("reference_mention_style", "auto"))
    if style != "auto":
        return style
    ref_state = ref_state if isinstance(ref_state, dict) else ensure_reference_map(session)
    if _reference_map_has_valid_aux_numbers(ref_state):
        return "compiled_pdf_numbers"
    return "auto"


def _reference_prompt_status_note(ref_state: dict[str, Any]) -> str:
    label_map = ref_state.get("label_map", {}) if isinstance(ref_state, dict) else {}
    map_source = str(ref_state.get("map_source") or "none") if isinstance(ref_state, dict) else "none"
    warning = normalize_whitespace(ref_state.get("warning", "")) if isinstance(ref_state, dict) else ""
    if _reference_map_has_valid_aux_numbers(ref_state):
        count = len(label_map)
        return (
            "Authoritative compiled numbering is available from the paper's AUX file "
            f"({count} labels recovered). When the guidance below maps a label to a compiled number, "
            "use that exact compiled PDF number and ignore any different local number printed in the pasted TeX."
        )
    if label_map:
        note = (
            "A freshly parsed AUX numbering map is not available for this run. "
            f"Reference map source: {map_source}. Use compiled numbers only when the guidance below explicitly provides them; "
            "otherwise cite by label or use descriptive page-local wording."
        )
    else:
        note = (
            "No valid AUX-derived compiled numbering map is available for this run. "
            f"Reference map source: {map_source}. Do not copy source-local equation/theorem/section numbers from the pasted TeX; "
            "cite by label when available, otherwise use descriptive page-local wording."
        )
    if warning:
        note += f" Note: {warning}"
    return note


def _reference_context_for_chunk_strict(
    session: dict[str, Any],
    chunk: dict[str, Any],
    max_items: int = 80,
    ref_state: Optional[dict[str, Any]] = None,
) -> str:
    ref_state = ref_state if isinstance(ref_state, dict) else ensure_reference_map(session)
    style = _effective_reference_mention_style(session, ref_state=ref_state)
    label_map = ref_state.get("label_map", {}) if isinstance(ref_state, dict) else {}
    labels = _extract_chunk_labels(chunk.get("chunk_text", ""))
    label_rows = _reference_label_rows(labels, label_map) if labels else []
    numbered_rows = _reference_number_rows(labels, label_map) if labels else []
    pdf_context = _pdf_reference_context_for_chunk(session, chunk, max_items=max(8, min(24, max_items // 3)))
    pdf_available = not pdf_context.startswith("No reliable visible numbering hints")

    if style == "source_labels":
        if label_rows:
            rows = label_rows[:max_items]
            if pdf_available:
                rows.extend([
                    "",
                    "Visible PDF numbering hints for fallback only (use only when a source label is unavailable for the object):",
                    pdf_context,
                ])
            return "\n".join(rows).strip()
        if pdf_available:
            return (
                "No source labels were recovered for this chunk. Use the visible PDF numbering hints below when they clearly match the object; "
                "otherwise use descriptive page-local wording without inventing labels.\n" + pdf_context
            )
        return "No source labels or reliable visible PDF numbering hints were recovered for this chunk."

    if label_map and labels:
        rows = numbered_rows[:max_items]
        return "\n".join(rows).strip()

    if style == "compiled_pdf_numbers":
        sections = []
        if pdf_available:
            sections.append(
                "Visible PDF numbering hints for this chunk (heuristic; use only when they clearly match the current object):\n" + pdf_context
            )
        if label_rows:
            sections.append(
                "Source labels recovered for this chunk. Use them only if the compiled/PDF-visible number remains unclear:\n" +
                "\n".join(label_rows[:max_items])
            )
        if sections:
            return "\n\n".join(sections).strip()
        return (
            "No AUX-derived numbering map, source labels, or reliable visible PDF numbering hints were recovered for this chunk. "
            "Do not invent numbers; use descriptive page-local wording instead."
        )

    if label_rows and pdf_available:
        return (
            "No AUX-derived compiled numbering was recovered for this chunk.\n\n"
            "Source labels recovered for this chunk:\n" + "\n".join(label_rows[:max_items]) +
            "\n\nVisible PDF numbering hints (use them when they clearly identify the same object):\n" + pdf_context
        )
    if label_rows:
        return "No AUX-derived compiled numbering was recovered for this chunk.\n" + "\n".join(label_rows[:max_items])
    if pdf_available:
        return (
            "No explicit source labels were recovered for this chunk. Use the visible PDF numbering hints below when they clearly match the object; "
            "otherwise use descriptive page-local wording.\n" + pdf_context
        )
    return "No explicit source labels or reliable visible PDF numbering hints were recovered for this chunk."


def _reference_prompt_rule_for_style(style: str) -> str:
    style = _normalize_reference_mention_style(style)
    if style == "compiled_pdf_numbers":
        return (
            "REFERENCE STYLE FOR THIS RUN: compiled_pdf_numbers.\n"
            "When the guidance below maps a label to a compiled number, write that exact compiled PDF number in prose.\n"
            "Never copy a different source-local number from the pasted TeX. If the compiled/PDF guidance is ambiguous, do not invent a number; prefer descriptive page-local wording, and use a source label only when that is the only reliable identifier available."
        )
    if style == "source_labels":
        return (
            "REFERENCE STYLE FOR THIS RUN: source_labels.\n"
            "When a source label is available for an object in this chunk, cite it by label, for example 'the equation labeled eq:foo'.\n"
            "Do not replace a recovered label with a guessed local number from the pasted TeX. If no source label is available for the object (for example in PDF-only chunking), use visible PDF numbering when clear; otherwise use descriptive page-local wording."
        )
    return (
        "REFERENCE STYLE FOR THIS RUN: auto.\n"
        "Prefer compiled PDF numbering whenever the reference guidance below confirms it, and treat AUX-derived mappings as authoritative when present.\n"
        "If only source labels are available for an object in this chunk, cite by label.\n"
        "If neither a reliable number nor a source label is available, use descriptive page-local wording rather than inventing a reference."
    )


def build_user_message_for_chunk(session: dict[str, Any], chunk: dict[str, Any]) -> list[dict[str, Any]]:
    content = []
    if not session.get("pdf_attached_in_conversation", False):
        content.append({"type": "input_file", "file_id": session["pdf_file_id"]})

    style = _normalize_reference_mention_style(session.get("reference_mention_style", "auto"))
    try:
        ref_state = ensure_reference_map(session)
        style = _effective_reference_mention_style(session, ref_state=ref_state)
        ref_status_note = _reference_prompt_status_note(ref_state)
        ref_context = _reference_context_for_chunk_strict(session, chunk, ref_state=ref_state)
    except Exception as e:
        ref_status_note = (
            f"Reference numbering state could not be fully loaded ({type(e).__name__}: {e}). "
            "Check state/reference_map.json for AUX parsing status. Do not invent numbering."
        )
        ref_context = (
            f"Reference guidance unavailable for this chunk ({type(e).__name__}: {e}). "
            "Check state/reference_map.json for AUX parsing status. Use compiled numbering when known; otherwise cite by label if a source label exists, "
            "and do not invent numbering."
        )
    prompt_text = f"""Audit this mathematics-paper chunk rigorously.

Chunk label: {chunk['label']}
Boundary: {chunk['boundary']}
Source kind: {chunk['source_kind']}
Estimated page span: {chunk['page_start']}-{chunk['page_end']}

Use the provided structured output schema.
Keep mathematical prose human-readable, with inline math in $...$ and display math in $$...$$.
If you include python_checks, every item must contain:
- purpose: a short title
- description: a self-contained explanation of the claim being tested, the test strategy, and any sample parameters or cases used
- expected_outcome: what output or condition would count as success
- code: runnable local Python code
Write the description so it can stand on its own in the final report before the script body.

Reference numbering status for this run:
{ref_status_note}

CRITICAL REFERENCE RULE
{_reference_prompt_rule_for_style(style)}
When the reference guidance below maps a label to a compiled number, write that exact compiled number.
Do NOT reuse source-local numbering from pasted TeX unless the reference guidance below explicitly confirms that it matches the compiled PDF.

Reference guidance for this chunk:
{ref_context}

Chunk text:
{chunk['chunk_text']}
"""
    content.append({"type": "input_text", "text": prompt_text})
    return [{"role": "user", "content": content}]

def reset_audit_from_chunk(pdf_path: str | Path, start_chunk_index: int = 1) -> dict[str, Any]:
    """Delete saved outputs from start_chunk_index onward and reset the session to rerun those chunks.
    This is useful after numbering/report patches, because old saved chunk outputs keep whatever wording the
    model originally produced.
    """
    session = load_session_from_pdf(pdf_path)
    if session is None:
        raise RuntimeError("No existing audit session found for that PDF.")
    workdir = Path(session["workdir"])
    paths = session_paths(workdir)

    # Load manifest and validate
    manifest = load_manifest(session)
    chunks = manifest.get("chunks", [])
    if start_chunk_index < 1 or start_chunk_index > max(1, len(chunks)):
        raise ValueError(f"start_chunk_index must be between 1 and {max(1, len(chunks))}")

    # Filter chunk records
    kept_records = []
    removed_chunk_ids = set()
    if paths["chunk_records"].exists():
        for line in paths["chunk_records"].read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if int(rec.get("chunk_index", 10**9)) < start_chunk_index:
                kept_records.append(rec)
            else:
                removed_chunk_ids.add(rec.get("chunk_id"))
        with paths["chunk_records"].open("w", encoding="utf-8") as f:
            for rec in kept_records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    # Remove response/prompt/request/patch/code files for rerun chunks
    chunk_pat = re.compile(r"^chunk_(\d{3})")
    for sub in ["responses", "requests", "prompts", "latex_patches", "python_checks"]:
        d = workdir / sub
        if d.exists():
            for p in d.iterdir():
                m = chunk_pat.match(p.name)
                if m and int(m.group(1)) >= start_chunk_index:
                    try:
                        p.unlink()
                    except Exception:
                        pass

    # Remove issues attached to rerun chunks
    issues_state = load_issues(session)
    issues_state["issues"] = [it for it in issues_state.get("issues", []) if it.get("chunk_id") not in removed_chunk_ids]
    issues_state["updated_at"] = utc_now()
    save_issues(session, issues_state)

    # Recompute usage from kept records only when possible
    usage = load_usage(session)
    kept_ids = {rec.get("chunk_id") for rec in kept_records}
    usage["per_chunk"] = [it for it in usage.get("per_chunk", []) if it.get("chunk_id") in kept_ids]
    totals = {
        "input_tokens": 0,
        "cached_tokens": 0,
        "output_tokens": 0,
        "reasoning_tokens": 0,
        "total_tokens": 0,
        "cost_usd": 0.0,
        "audit_seconds": 0.0,
    }
    for it in usage.get("per_chunk", []):
        u = it.get("usage", {})
        c = it.get("cost", {})
        totals["input_tokens"] += int(u.get("input_tokens", 0))
        totals["cached_tokens"] += int((u.get("input_tokens_details") or {}).get("cached_tokens", 0))
        totals["output_tokens"] += int(u.get("output_tokens", 0))
        totals["reasoning_tokens"] += int((u.get("output_tokens_details") or {}).get("reasoning_tokens", 0))
        totals["total_tokens"] += int(u.get("total_tokens", 0))
        totals["cost_usd"] += float(c.get("total_cost", 0.0))
        totals["audit_seconds"] += float(it.get("elapsed_seconds", 0.0))
    usage["totals"].update(totals)
    usage["updated_at"] = utc_now()
    save_usage(session, usage)

    # Reset session and status
    session["next_chunk_index"] = start_chunk_index
    session["pending"] = None
    session["audit_finished_at"] = None
    session["updated_at"] = utc_now()
    save_session(session)

    status = load_status(session)
    completed = start_chunk_index - 1
    status["status"] = "initialized" if completed == 0 else "paused"
    status["chunks_completed"] = completed
    status["chunks_total"] = len(chunks)
    status["current_chunk_id"] = None
    status["progress_pct"] = 0.0 if len(chunks) == 0 else round(100.0 * completed / len(chunks), 1)
    if completed > 0:
        completed_pages = max(rec.get("page_end", 0) for rec in kept_records) if kept_records else 0
    else:
        completed_pages = 0
    status["estimated_pages_completed"] = completed_pages
    status["cost_usd"] = usage["totals"]["cost_usd"]
    status["updated_at"] = utc_now()
    save_status(session, status)

    print(f"Reset audit from chunk {start_chunk_index:03d}.")
    print("You should now rerun audit_the_paper(..., continue_existing=True).")
    return session
````

</details>

### Archived legacy notebook cell 18 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `backend examples / documentation`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original markdown cell</summary>

````markdown
## Recommended entry point for new papers

This final patch adds `start_fresh_audit(...)`, which archives any existing `_audit` folder for the same PDF and starts a clean new session. For a fresh rerun of a paper, use this helper rather than reusing older state.
````

</details>

### Archived legacy notebook cell 19 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_runtime.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Consolidated fresh-session patch:
# - recommended general-purpose entry point for new papers
# - adds archival of an existing _audit workdir before starting a new session
# - keeps all earlier fixes (structured output, recovery, numbering, timing, safer report)

import shutil
from pathlib import Path

def archive_existing_audit_workdir(pdf_path: str | Path) -> Path | None:
    pdf_path = Path(pdf_path).expanduser().resolve()
    workdir = workdir_from_pdf(pdf_path)
    if not workdir.exists():
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archived = workdir.with_name(workdir.name + "_archived_" + timestamp)
    counter = 1
    while archived.exists():
        archived = workdir.with_name(workdir.name + f"_archived_{timestamp}_{counter}")
        counter += 1
    shutil.move(str(workdir), str(archived))
    print(f"Archived existing workdir to: {archived}")
    return archived

def create_new_session(
    pdf_path: str | Path,
    model: str = DEFAULT_MODEL,
    reasoning_effort: str = DEFAULT_REASONING_EFFORT,
    tex_max_chars: int = 4500,
    pdf_max_chars: int = 3500,
    store: bool = True,
    background: bool = True,
    archive_existing_workdir: bool = False,
    use_code_interpreter: bool = False,
    code_interpreter_memory_limit: Optional[str] = "4g",
    reuse_code_interpreter_container: bool = False,
    ci_failure_fallback_mode: str = "off",
    reference_mention_style: str = "auto",
    report_reference_style: str = "match_audit",
) -> dict[str, Any]:
    pdf_path = Path(pdf_path).expanduser().resolve()
    if not pdf_path.exists():
        raise FileNotFoundError(pdf_path)
    tex_path = pdf_path.with_suffix(".tex")
    workdir = workdir_from_pdf(pdf_path)

    if archive_existing_workdir and workdir.exists():
        archive_existing_audit_workdir(pdf_path)

    ensure_workdir_tree(workdir)
    init_state_files(workdir, model=model, reasoning_effort=reasoning_effort)

    conversation = client.conversations.create()
    with pdf_path.open("rb") as f:
        uploaded = client.files.create(file=f, purpose="user_data")

    session = {
        "created_at": utc_now(),
        "updated_at": utc_now(),
        "pdf_path": str(pdf_path),
        "tex_path": str(tex_path) if tex_path.exists() else None,
        "workdir": str(workdir),
        "model": model,
        "reasoning_effort": reasoning_effort,
        "store": store,
        "background": background,
        "conversation_id": conversation.id,
        "pdf_file_id": uploaded.id,
        "pdf_attached_in_conversation": False,
        "developer_prompt_seeded": False,
        "next_chunk_index": 1,
        "last_response_id": None,
        "pending": None,
        "use_code_interpreter": bool(use_code_interpreter),
        "code_interpreter_memory_limit": str(code_interpreter_memory_limit) if code_interpreter_memory_limit else None,
        "reuse_code_interpreter_container": bool(reuse_code_interpreter_container),
        "code_interpreter_container_id": None,
        "ci_failure_fallback_mode": _normalize_ci_failure_fallback_mode(ci_failure_fallback_mode),
        "reference_mention_style": _normalize_reference_mention_style(reference_mention_style),
        "report_reference_style": _normalize_report_reference_style(report_reference_style),
    }

    manifest = build_auto_chunks(
        pdf_path=pdf_path,
        tex_path=tex_path if tex_path.exists() else None,
        tex_max_chars=tex_max_chars,
        pdf_max_chars=pdf_max_chars,
    )

    save_session(session)
    save_manifest(session, manifest)

    status = load_status(session)
    status.update({
        "status": "initialized",
        "progress_pct": 0.0,
        "current_chunk_id": None,
        "chunks_completed": 0,
        "chunks_total": len(manifest["chunks"]),
        "estimated_pages_completed": 0,
        "estimated_pages_total": manifest.get("pdf_page_count", 0),
        "cost_usd": 0.0,
    })
    save_status(session, status)
    return _ensure_timing_state(session)

def _is_obviously_noncomputational_chunk(chunk: dict[str, Any], short_text_limit: int = 350) -> bool:
    if (chunk.get("source_kind") or "").strip().lower() != "tex-gap":
        return False
    text = (chunk.get("chunk_text") or "").strip()
    if not text or len(text) > int(short_text_limit):
        return False
    if re.search(r"\\(?:label|eqref|ref|cref|Cref|autoref)\{", text):
        return False
    if re.search(r"\\begin\{(?:align\*?|equation\*?|gather\*?|multline\*?|split)\}", text):
        return False
    if "\\[" in text or "$$" in text:
        return False
    if re.search(
        r"\\(?:begin|end)\{(?:theorem|lemma|proposition|corollary|claim|fact|definition|remark|example|conjecture|algorithm|thm|lem|prop|cor|defn|proof)\}",
        text,
        flags=re.IGNORECASE,
    ):
        return False
    return True

def _pause_audit_if_requested(session: dict[str, Any], verbose: bool = True) -> Optional[dict[str, Any]]:
    pause_requested_at = str(session.get("pause_requested_at") or "").strip()
    if not pause_requested_at:
        return None

    session["last_pause_requested_at"] = pause_requested_at
    session.pop("pause_requested_at", None)
    session["updated_at"] = utc_now()
    save_session(session)

    usage = load_usage(session)
    status = load_status(session)
    status["status"] = "paused"
    status["current_chunk_id"] = None
    status["current_chunk_elapsed_seconds"] = 0.0
    status["total_audit_seconds"] = float(usage["totals"].get("audit_seconds", 0.0) or 0.0)
    status["paused_at"] = utc_now()
    status["pause_reason"] = "requested"
    save_status(session, status)

    if verbose:
        print(
            f"Audit paused on request after completing the current chunk. "
            f"Resume from chunk index {session.get('next_chunk_index', 1)} when ready."
        )

    return {
        "paused": True,
        "reason": "requested",
        "pause_requested_at": pause_requested_at,
        "next_chunk_index": int(session.get("next_chunk_index", 1) or 1),
    }

def audit_the_paper(
    pdf_path: str | Path,
    model: str = DEFAULT_MODEL,
    reasoning_effort: str = DEFAULT_REASONING_EFFORT,
    tex_max_chars: int = 4500,
    pdf_max_chars: int = 3500,
    continue_existing: bool = True,
    archive_existing_workdir: bool = False,
    poll_every: float = 3.0,
    max_wait_seconds: Optional[float] = None,
    stop_after_chunks: Optional[int] = None,
    run_generated_verification: bool = False,
    verification_timeout_seconds: int = 120,
    include_verification_summary_in_final_report: bool = True,
    write_separate_verification_report: bool = True,
    use_code_interpreter: Optional[bool] = None,
    code_interpreter_memory_limit: Optional[str] = None,
    reuse_code_interpreter_container: Optional[bool] = None,
    ci_failure_fallback_mode: Optional[str] = None,
    reference_mention_style: Optional[str] = None,
    report_reference_style: Optional[str] = None,
    verification_mode: str = "local_python_only",
    verification_chunk_indices: Optional[list[int]] = None,
    verbose: bool = True,
) -> dict[str, Any]:
    pdf_path = Path(pdf_path).expanduser().resolve()
    verification_mode = _normalize_verification_mode(verification_mode)
    explicit_verification_chunk_selection = verification_chunk_indices is not None
    selected_chunk_indices = set()
    for raw_idx in verification_chunk_indices or []:
        try:
            selected_chunk_indices.add(int(raw_idx))
        except Exception:
            continue

    if continue_existing:
        session = load_session_from_pdf(pdf_path)
    else:
        session = None

    if session is None:
        session = create_new_session(
            pdf_path,
            model=model,
            reasoning_effort=reasoning_effort,
            tex_max_chars=tex_max_chars,
            pdf_max_chars=pdf_max_chars,
            archive_existing_workdir=archive_existing_workdir,
            use_code_interpreter=bool(use_code_interpreter) if use_code_interpreter is not None else False,
            code_interpreter_memory_limit=code_interpreter_memory_limit if code_interpreter_memory_limit is not None else "4g",
            reuse_code_interpreter_container=bool(reuse_code_interpreter_container) if reuse_code_interpreter_container is not None else False,
            ci_failure_fallback_mode=_normalize_ci_failure_fallback_mode(ci_failure_fallback_mode) if ci_failure_fallback_mode is not None else "off",
            reference_mention_style=_normalize_reference_mention_style(reference_mention_style) if reference_mention_style is not None else "auto",
            report_reference_style=_normalize_report_reference_style(report_reference_style) if report_reference_style is not None else "match_audit",
        )
        if verbose:
            print(f"Created workdir: {session['workdir']}")
            print(f"PDF: {session['pdf_path']}")
            print(f"TeX: {session['tex_path'] or 'not found'}")
    else:
        session["model"] = model
        session["reasoning_effort"] = reasoning_effort
        if use_code_interpreter is not None:
            session["use_code_interpreter"] = bool(use_code_interpreter)
        if code_interpreter_memory_limit is not None:
            session["code_interpreter_memory_limit"] = str(code_interpreter_memory_limit) if code_interpreter_memory_limit else None
        if reuse_code_interpreter_container is not None:
            session["reuse_code_interpreter_container"] = bool(reuse_code_interpreter_container)
        if ci_failure_fallback_mode is not None:
            session["ci_failure_fallback_mode"] = _normalize_ci_failure_fallback_mode(ci_failure_fallback_mode)
        if reference_mention_style is not None:
            session["reference_mention_style"] = _normalize_reference_mention_style(reference_mention_style)
        if report_reference_style is not None:
            session["report_reference_style"] = _normalize_report_reference_style(report_reference_style)
        session["updated_at"] = utc_now()
        save_session(session)
        if verbose:
            print(f"Loaded existing session from {session['workdir']}")
            print(f"Using model: {session['model']}")
            print(f"Using reasoning effort: {session['reasoning_effort']}")

    session["verification_timeout_seconds"] = int(verification_timeout_seconds)
    session["include_verification_summary_in_final_report"] = bool(include_verification_summary_in_final_report)
    session["write_separate_verification_report"] = bool(write_separate_verification_report)
    session["updated_at"] = utc_now()
    save_session(session)

    manifest = load_manifest(session)
    chunks = manifest["chunks"]

    if not chunks:
        raise RuntimeError("Chunk manifest is empty.")

    recovery_result = None
    if session.get("pending"):
        if verbose:
            print(f"Recovering pending chunk {session['pending'].get('chunk_id')} ...")
        recovery_result = recover_pending_chunk(
            session,
            poll_every=poll_every,
            max_wait_seconds=max_wait_seconds,
            display_output=True,
        )
        paused_status = load_status(session)
        if paused_status.get("status") == "paused":
            if verbose:
                failure_path = None
                if isinstance(recovery_result, dict):
                    failure_path = recovery_result.get("failure_summary_path")
                if failure_path:
                    print(f"Audit paused after recovery detected a previously submitted request failed remotely. See: {failure_path}")
                else:
                    print("Audit paused during recovery of a previously submitted request. Inspect logs/ and responses/ for details.")
            return {
                "session": session,
                "status": paused_status,
                "usage": load_usage(session),
                "report_paths": None,
                "recovery_result": recovery_result,
            }

    session = load_session_from_pdf(pdf_path) or session
    pause_result = _pause_audit_if_requested(session, verbose=verbose)
    if pause_result is not None:
        return {
            "session": session,
            "status": load_status(session),
            "usage": load_usage(session),
            "report_paths": None,
            "pause_result": pause_result,
        }

    start_idx = session["next_chunk_index"]
    processed = 0
    for idx in range(start_idx, len(chunks) + 1):
        chunk = chunks[idx - 1]
        chunk_verification_mode = verification_mode if (not selected_chunk_indices or idx in selected_chunk_indices) else "local_python_only"
        if (
            session.get("use_code_interpreter", False)
            and verification_mode == "code_interpreter"
            and not explicit_verification_chunk_selection
            and chunk_verification_mode == "code_interpreter"
            and _is_obviously_noncomputational_chunk(chunk)
        ):
            chunk_verification_mode = "local_python_only"
        process_one_chunk(
            session,
            chunk,
            poll_every=poll_every,
            max_wait_seconds=max_wait_seconds,
            display_output=True,
            verification_mode=chunk_verification_mode,
        )
        processed += 1

        session = load_session_from_pdf(pdf_path) or session
        pause_result = _pause_audit_if_requested(session, verbose=verbose)
        if pause_result is not None:
            return {
                "session": session,
                "status": load_status(session),
                "usage": load_usage(session),
                "report_paths": None,
                "pause_result": pause_result,
            }

        if stop_after_chunks is not None and processed >= stop_after_chunks:
            break

    if run_generated_verification:
        verification_run = run_verification_suite(
            pdf_path,
            timeout=int(verification_timeout_seconds),
            safe_only=True,
        )
        if verbose:
            summary = verification_run.get("summary", {})
            print(
                "Verification suite:" 
                f"{summary.get('passed', 0)} passed, "
                f"{summary.get('failed', 0)} failed, "
                f"{summary.get('timeout', 0)} timed out, "
                f"{summary.get('skipped', 0)} skipped"
            )

    session = load_session_from_pdf(pdf_path)
    paths = build_final_report(
        session,
        include_verification_summary_in_final_report=include_verification_summary_in_final_report,
        write_separate_verification_report=write_separate_verification_report,
    )
    if verbose:
        print("Final report:", paths.get("markdown"))
        print("JSON report:", paths.get("json"))
        print("TeX report:", paths.get("tex"))
    return {"session": session, "report_paths": paths}

def start_fresh_audit(
    pdf_path: str | Path,
    model: str = DEFAULT_MODEL,
    reasoning_effort: str = DEFAULT_REASONING_EFFORT,
    tex_max_chars: int = 4500,
    pdf_max_chars: int = 3500,
    poll_every: float = 3.0,
    max_wait_seconds: Optional[float] = None,
    stop_after_chunks: Optional[int] = None,
    run_generated_verification: bool = False,
    verification_timeout_seconds: int = 120,
    include_verification_summary_in_final_report: bool = True,
    write_separate_verification_report: bool = True,
    use_code_interpreter: bool = False,
    code_interpreter_memory_limit: Optional[str] = "4g",
    reuse_code_interpreter_container: bool = False,
    ci_failure_fallback_mode: str = "off",
    reference_mention_style: str = "auto",
    report_reference_style: str = "match_audit",
    verification_mode: str = "local_python_only",
    verification_chunk_indices: Optional[list[int]] = None,
    verbose: bool = True,
) -> dict[str, Any]:
    return audit_the_paper(
        pdf_path=pdf_path,
        model=model,
        reasoning_effort=reasoning_effort,
        tex_max_chars=tex_max_chars,
        pdf_max_chars=pdf_max_chars,
        continue_existing=False,
        archive_existing_workdir=True,
        poll_every=poll_every,
        max_wait_seconds=max_wait_seconds,
        stop_after_chunks=stop_after_chunks,
        run_generated_verification=run_generated_verification,
        verification_timeout_seconds=verification_timeout_seconds,
        include_verification_summary_in_final_report=include_verification_summary_in_final_report,
        write_separate_verification_report=write_separate_verification_report,
        use_code_interpreter=use_code_interpreter,
        code_interpreter_memory_limit=code_interpreter_memory_limit,
        reuse_code_interpreter_container=reuse_code_interpreter_container,
        ci_failure_fallback_mode=ci_failure_fallback_mode,
        reference_mention_style=reference_mention_style,
        report_reference_style=report_reference_style,
        verification_mode=verification_mode,
        verification_chunk_indices=verification_chunk_indices,
        verbose=verbose,
    )
````

</details>

### Archived legacy notebook cell 20 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_verification.py, audit_runtime.py, and audit_policy_hooks.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Post-audit verification patch: run generated python_checks locally and report them safely.

import ast
import subprocess
import sys
from pathlib import Path
from audit_verification import (
    _ast_call_name,
    _check_verification_script_safety,
    _chunk_id_from_script_name,
    _chunk_index_from_chunk_id,
    _collect_verification_scripts,
    _ensure_verification_results_dir,
    _first_nonempty_line,
    _infer_verification_conclusion,
    _load_verification_results,
    _open_call_looks_dangerous,
    _resolve_verification_script_path,
    _run_verification_scripts,
    _truncate_text,
    _verification_result_path,
    _verification_results_path,
    _verification_summary_counts,
    rerun_failed_verification_scripts,
    run_verification_suite,
    show_verification_status,
)
from audit_runtime import build_verification_report as runtime_build_verification_report

_old_build_final_report_markdown_with_local_verification = build_final_report_markdown
_old_build_final_report_tex_with_local_verification = build_final_report_tex
_old_build_final_report_with_local_verification = build_final_report

def _compact_verification_summary_markdown(session: dict[str, Any]) -> str:
    state = load_verification_state(session)
    last_run = state.get("last_run") or {}
    results = _load_verification_results(session, state=state)
    if not last_run and not results:
        return ""
    counts = _verification_summary_counts(results) if results else {
        "scripts_total": int(last_run.get("scripts_total", 0) or 0),
        "passed": int(last_run.get("passed", 0) or 0),
        "failed": int(last_run.get("failed", 0) or 0),
        "timeout": int(last_run.get("timeout", 0) or 0),
        "skipped": int(last_run.get("skipped", 0) or 0),
    }
    lines = [
        "## Verification summary",
        "",
        f"- Scripts run: {counts['scripts_total']}",
        f"- Passed: {counts['passed']}",
        f"- Failed: {counts['failed']}",
        f"- Timed out: {counts['timeout']}",
        f"- Skipped: {counts['skipped']}",
    ]
    failing = [item.get("script_name") for item in results if item.get("status") in {"failed", "timeout"}]
    if failing:
        lines.append(f"- Failing scripts: {', '.join(failing[:10])}")
    skipped = [item.get("script_name") for item in results if item.get("status") == "skipped"]
    if skipped:
        lines.append(f"- Skipped scripts: {', '.join(skipped[:10])}")
    lines.append("")
    return "\n".join(lines)

def _compact_verification_summary_tex(session: dict[str, Any]) -> str:
    state = load_verification_state(session)
    last_run = state.get("last_run") or {}
    results = _load_verification_results(session, state=state)
    if not last_run and not results:
        return ""
    counts = _verification_summary_counts(results) if results else {
        "scripts_total": int(last_run.get("scripts_total", 0) or 0),
        "passed": int(last_run.get("passed", 0) or 0),
        "failed": int(last_run.get("failed", 0) or 0),
        "timeout": int(last_run.get("timeout", 0) or 0),
        "skipped": int(last_run.get("skipped", 0) or 0),
    }
    parts = [
        r"\section*{Verification summary}",
        r"\begin{itemize}",
        r"\item Scripts run: " + str(counts["scripts_total"]),
        r"\item Passed: " + str(counts["passed"]),
        r"\item Failed: " + str(counts["failed"]),
        r"\item Timed out: " + str(counts["timeout"]),
        r"\item Skipped: " + str(counts["skipped"]),
    ]
    failing = [item.get("script_name") for item in results if item.get("status") in {"failed", "timeout"}]
    if failing:
        parts.append(r"\item Failing scripts: " + report_latex_paragraph(", ".join(failing[:10])))
    skipped = [item.get("script_name") for item in results if item.get("status") == "skipped"]
    if skipped:
        parts.append(r"\item Skipped scripts: " + report_latex_paragraph(", ".join(skipped[:10])))
    parts.append(r"\end{itemize}")
    return "\n".join(parts) + "\n"

def _display_verification_script_path(session: dict[str, Any], result: dict[str, Any]) -> str:
    script_path_text = str(result.get("script_path") or "").strip()
    if not script_path_text:
        return str(result.get("script_name") or "")
    script_path = Path(script_path_text)
    root = Path(session["workdir"]).resolve()
    try:
        return str(script_path.resolve().relative_to(root))
    except Exception:
        return script_path.name or script_path_text

def _verification_conclusion_for_display(result: dict[str, Any], limit: int = 160) -> str:
    text = _strip_unsafe_control_chars(_repair_json_escape_artifacts(result.get("conclusion") or "No conclusion available."))
    return _truncate_text(text, limit=limit).replace("\\n", " ")

def _verification_report_markdown(session: dict[str, Any], state: dict[str, Any], results: list[dict[str, Any]]) -> str:
    counts = _verification_summary_counts(results)
    last_run = state.get("last_run") or {}
    title = Path(session["pdf_path"]).stem
    lines = [
        f"# Verification report -- {title}",
        "",
        f"- PDF: {session['pdf_path']}",
        f"- Workdir: {session['workdir']}",
        f"- Python interpreter: {last_run.get('python_executable', sys.executable)}",
        f"- Safe only mode: {last_run.get('safe_only', True)}",
        f"- Timeout per script: {last_run.get('timeout_seconds', 0)}s",
        f"- Scripts run: {counts['scripts_total']}",
        f"- Passed: {counts['passed']}",
        f"- Failed: {counts['failed']}",
        f"- Timed out: {counts['timeout']}",
        f"- Skipped: {counts['skipped']}",
        "",
    ]
    if not results:
        lines.append("No verification results found.")
        return "\\n".join(lines).strip() + "\\n"
    for result in results:
        lines.extend([
            f"## {result.get('script_name','script')} [{result.get('status','unknown')}]",
            f"- Chunk: {result.get('chunk_id') or 'unknown'}",
            f"- Script: {_display_verification_script_path(session, result)}",
            f"- Return code: {result.get('returncode')}",
            f"- Elapsed time: {format_duration(result.get('elapsed_seconds', 0.0))}",
            f"- Conclusion: {_verification_conclusion_for_display(result)}",
        ])
        if result.get("skip_reason"):
            lines.append(f"- Skip reason: {result.get('skip_reason')}")
        stdout_excerpt = _truncate_text(result.get("stdout", ""))
        if stdout_excerpt:
            lines.extend(["- Stdout excerpt:", "```text", stdout_excerpt, "```"])
        stderr_excerpt = _truncate_text(result.get("stderr", ""))
        if stderr_excerpt:
            lines.extend(["- Stderr excerpt:", "```text", stderr_excerpt, "```"])
        lines.append("")
    return _strip_unsafe_control_chars("\\n".join(lines).strip() + "\\n")

def _verification_report_tex(session: dict[str, Any], state: dict[str, Any], results: list[dict[str, Any]]) -> str:
    counts = _verification_summary_counts(results)
    last_run = state.get("last_run") or {}
    title = report_latex_paragraph(f"Verification report -- {Path(session['pdf_path']).stem}")
    parts = [r"""\documentclass[11pt]{article}
\usepackage[a4paper,margin=1in]{geometry}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage{lmodern}
\usepackage{microtype}
\usepackage{hyperref}
\usepackage{enumitem}
\usepackage{xcolor}
\usepackage{fancyvrb}
\setlist[itemize]{leftmargin=2em}
\setlength{\parskip}{0.5em}
\setlength{\parindent}{0pt}
\begin{document}
"""]
    parts.append(r"\section*{" + title + "}" + "\n")
    parts.append(r"\begin{itemize}" + "\n")
    parts.append(r"\item PDF: " + report_latex_paragraph(session["pdf_path"]) + "\n")
    parts.append(r"\item Workdir: " + report_latex_paragraph(session["workdir"]) + "\n")
    parts.append(r"\item Python interpreter: " + report_latex_paragraph(str(last_run.get("python_executable", sys.executable))) + "\n")
    parts.append(r"\item Safe only mode: " + report_latex_paragraph(str(last_run.get("safe_only", True))) + "\n")
    parts.append(r"\item Timeout per script: " + str(last_run.get("timeout_seconds", 0)) + "s\n")
    parts.append(r"\item Scripts run: " + str(counts["scripts_total"]) + "\n")
    parts.append(r"\item Passed: " + str(counts["passed"]) + "\n")
    parts.append(r"\item Failed: " + str(counts["failed"]) + "\n")
    parts.append(r"\item Timed out: " + str(counts["timeout"]) + "\n")
    parts.append(r"\item Skipped: " + str(counts["skipped"]) + "\n")
    parts.append(r"\end{itemize}" + "\n")
    if not results:
        parts.append("No verification results found.\n")
    else:
        for result in results:
            heading = report_latex_paragraph(f"{result.get('script_name','script')} [{result.get('status','unknown')}]" )
            parts.append(r"\subsection*{" + heading + "}" + "\n")
            parts.append(r"\begin{itemize}" + "\n")
            parts.append(r"\item Chunk: " + report_latex_paragraph(result.get("chunk_id") or "unknown") + "\n")
            parts.append(r"\item Script: " + report_latex_paragraph(_display_verification_script_path(session, result)) + "\n")
            parts.append(r"\item Return code: " + report_latex_paragraph(str(result.get("returncode"))) + "\n")
            parts.append(r"\item Elapsed time: " + report_latex_paragraph(format_duration(result.get("elapsed_seconds", 0.0))) + "\n")
            parts.append(r"\item Conclusion: " + report_latex_paragraph(_verification_conclusion_for_display(result)) + "\n")
            if result.get("skip_reason"):
                parts.append(r"\item Skip reason: " + report_latex_paragraph(result.get("skip_reason")) + "\n")
            parts.append(r"\end{itemize}" + "\n")
            stdout_excerpt = _truncate_text(result.get("stdout", ""))
            if stdout_excerpt:
                parts.append(r"\paragraph{Stdout excerpt}" + "\n")
                parts.append(_verbatim_block(stdout_excerpt) + "\n")
            stderr_excerpt = _truncate_text(result.get("stderr", ""))
            if stderr_excerpt:
                parts.append(r"\paragraph{Stderr excerpt}" + "\n")
                parts.append(_verbatim_block(stderr_excerpt) + "\n")
    parts.append(r"\end{document}" + "\n")
    return _strip_unsafe_control_chars("".join(parts))

def build_verification_report(session_or_pdf: dict[str, Any] | str | Path) -> dict[str, str]:
    return runtime_build_verification_report(session_or_pdf)

def build_final_report_markdown(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    text = _old_build_final_report_markdown_with_local_verification(session, report_title=report_title)
    if not session.get("include_verification_summary_in_final_report", True):
        return text
    summary = _compact_verification_summary_markdown(session)
    if not summary:
        return text
    return text.rstrip() + "\n\n" + summary.strip() + "\n"

def build_final_report_tex(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    text = _old_build_final_report_tex_with_local_verification(session, report_title=report_title)
    if not session.get("include_verification_summary_in_final_report", True):
        return text
    summary = _compact_verification_summary_tex(session)
    if not summary:
        return text
    if r"\end{document}" in text:
        return text.replace(r"\end{document}", summary + r"\end{document}", 1)
    return text + "\n" + summary

def build_final_report(
    session_or_pdf: dict[str, Any] | str | Path,
    report_title: Optional[str] = None,
    include_verification_summary_in_final_report: Optional[bool] = None,
    write_separate_verification_report: Optional[bool] = None,
) -> dict[str, str]:
    session = session_or_pdf if isinstance(session_or_pdf, dict) else load_session_from_pdf(session_or_pdf)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")
    changed = False
    if include_verification_summary_in_final_report is not None:
        session["include_verification_summary_in_final_report"] = bool(include_verification_summary_in_final_report)
        changed = True
    if write_separate_verification_report is not None:
        session["write_separate_verification_report"] = bool(write_separate_verification_report)
        changed = True
    if changed:
        session["updated_at"] = utc_now()
        save_session(session)
    paths = _old_build_final_report_with_local_verification(session, report_title=report_title)
    if session.get("write_separate_verification_report", True):
        verification_paths = build_verification_report(session)
        if verification_paths:
            paths = dict(paths)
            paths["verification_markdown"] = verification_paths.get("markdown", "")
            paths["verification_tex"] = verification_paths.get("tex", "")
            paths["verification_json"] = verification_paths.get("json", "")
    return paths
````

</details>

### Archived legacy notebook cell 21 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `maintenance-only archived reference; promote to backend before reuse`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Backfill helper for older audits that lack python_checks descriptions or expected outcomes.

def _backfill_markdown_path_for_record(rec: dict[str, Any], structured_path: Path) -> Path:
    raw_path = rec.get("markdown_path")
    if raw_path:
        candidate = Path(str(raw_path))
        if candidate.name:
            return structured_path.parent / candidate.name
    return structured_path.with_suffix(".md")


def backfill_python_check_descriptions(
    pdf_path: str | Path,
    rebuild_reports: bool = True,
    rewrite_chunk_markdown: bool = True,
) -> dict[str, Any]:
    session = load_session_from_pdf(pdf_path)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")

    updated_chunks = []
    for rec in read_chunk_records(session):
        structured_path = Path(str(rec.get("structured_response_path") or ""))
        if not structured_path.exists():
            continue
        audit = load_json(structured_path)
        checks = audit.get("python_checks") if isinstance(audit.get("python_checks"), list) else []
        if not checks:
            continue

        changed = False
        new_checks = []
        for chk in checks:
            if not isinstance(chk, dict):
                new_checks.append(chk)
                continue
            normalized = _normalize_python_check_entry(
                chk,
                chunk_label=str(rec.get("label") or audit.get("label") or ""),
                chunk_boundary=str(rec.get("boundary") or audit.get("boundary") or ""),
            )
            merged = dict(chk)
            if not str(merged.get("purpose", "") or "").strip():
                merged["purpose"] = normalized["purpose"]
                changed = True
            if not str(merged.get("description", "") or "").strip():
                merged["description"] = normalized["description"]
                changed = True
            if not str(merged.get("expected_outcome", "") or "").strip():
                merged["expected_outcome"] = normalized["expected_outcome"]
                changed = True
            if "code" not in merged:
                merged["code"] = normalized["code"]
                changed = True
            new_checks.append(merged)

        if not changed:
            continue

        audit["python_checks"] = new_checks
        audit = _coerce_audit_payload(audit)
        save_json(structured_path, audit)
        markdown_path = _backfill_markdown_path_for_record(rec, structured_path)
        if rewrite_chunk_markdown:
            markdown_path.write_text(render_audit_markdown(audit), encoding="utf-8")
        updated_chunks.append({
            "chunk_id": rec.get("chunk_id"),
            "structured_response_path": str(structured_path),
            "markdown_path": str(markdown_path),
        })

    report_paths = {}
    if rebuild_reports and updated_chunks:
        report_paths = build_final_report(session)

    return {
        "session": session,
        "updated_count": len(updated_chunks),
        "updated_chunks": updated_chunks,
        "report_paths": report_paths,
    }
````

</details>

### Archived legacy notebook cell 22 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_runtime.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Post-audit Q&A patch: reuse saved conversation and audit artifacts safely.

from IPython.display import Markdown, display
from audit_runtime import (
    _load_qa_turns as _runtime_load_qa_turns,
    _resolve_qa_session as _runtime_resolve_qa_session,
    ask_about_audit as runtime_ask_about_audit,
    ask_about_paper as runtime_ask_about_paper,
    rebuild_qa_report as runtime_rebuild_qa_report,
    set_openai_client as _runtime_set_openai_client,
)
_runtime_set_openai_client(client)

_QA_TURN_RE = re.compile(r"^qa_(\d+)\.json$")
_QA_STOPWORDS = {
    "a", "about", "all", "an", "and", "are", "as", "at", "be", "by", "can", "draft", "explain",
    "find", "findings", "for", "from", "give", "how", "i", "in", "into", "is", "it", "its", "me",
    "most", "of", "on", "or", "paper", "please", "referee", "related", "show", "so", "summarize",
    "summary", "tell", "that", "the", "their", "them", "there", "these", "this", "those", "to",
    "what", "which", "with", "would", "write", "you",
}


def _resolve_qa_session(session_or_pdf_path: dict[str, Any] | str | Path) -> dict[str, Any]:
    if isinstance(session_or_pdf_path, dict):
        session = session_or_pdf_path
    else:
        session = load_session_from_pdf(session_or_pdf_path)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")
    return session


def _assert_qa_ready(session: dict[str, Any]) -> None:
    if session.get("pending"):
        raise RuntimeError(
            "This audit session still has a pending chunk. Recover or finish the audit before using post-audit Q&A."
        )
    status = load_status(session)
    if str(status.get("status") or "").lower() != "completed":
        raise RuntimeError(
            f"Post-audit Q&A is only enabled after the main audit is completed. Current status={status.get('status')!r}."
        )


def _ensure_qa_conversation(session: dict[str, Any]) -> dict[str, Any]:
    changed = False
    if not session.get("conversation_id"):
        conversation = client.conversations.create()
        session["conversation_id"] = conversation.id
        changed = True
    if not session.get("pdf_file_id"):
        pdf_path = Path(session["pdf_path"]).expanduser().resolve()
        if pdf_path.exists():
            with pdf_path.open("rb") as f:
                uploaded = client.files.create(file=f, purpose="user_data")
            session["pdf_file_id"] = uploaded.id
            session["pdf_attached_in_conversation"] = False
            changed = True
    if changed:
        session["updated_at"] = utc_now()
        save_session(session)
    return session


def _qa_dir(session: dict[str, Any]) -> Path:
    path = Path(session["workdir"]) / "qa"
    path.mkdir(parents=True, exist_ok=True)
    return path


def _qa_turn_paths(session: dict[str, Any], idx: int) -> dict[str, Path]:
    root = _qa_dir(session)
    stem = f"qa_{idx:03d}"
    return {
        "json": root / f"{stem}.json",
        "md": root / f"{stem}.md",
    }


def _next_qa_index(session: dict[str, Any]) -> int:
    max_idx = 0
    for path in _qa_dir(session).glob("qa_*.json"):
        m = _QA_TURN_RE.match(path.name)
        if m:
            max_idx = max(max_idx, int(m.group(1)))
    return max_idx + 1


def _load_qa_turns(session: dict[str, Any]) -> list[dict[str, Any]]:
    turns = []
    for path in sorted(_qa_dir(session).glob("qa_*.json")):
        try:
            data = load_json(path)
        except Exception:
            continue
        if isinstance(data, dict):
            data.setdefault("turn_path", str(path))
            turns.append(data)
    turns.sort(key=lambda item: (str(item.get("time") or ""), str(item.get("turn_id") or "")))
    return turns


def _save_qa_turn(
    session: dict[str, Any],
    turn: dict[str, Any],
    answer_markdown: str,
    idx: int,
) -> dict[str, str]:
    paths = _qa_turn_paths(session, idx)
    save_json(paths["json"], turn)
    lines = [
        f"# Q&A turn {idx:03d} -- {Path(session['pdf_path']).stem}",
        "",
        f"- Time: {turn.get('time', '')}",
        f"- Mode: {turn.get('mode', '')}",
        f"- Model: {turn.get('model', '')}",
        f"- Reasoning effort: {turn.get('reasoning_effort', '')}",
        f"- Response ID: {turn.get('response_id', '') or 'n/a'}",
        "",
        "## Question",
        "",
        _strip_unsafe_control_chars(turn.get("question", "")),
        "",
    ]
    grounding_summary = str(turn.get("grounding_summary") or "").strip()
    if grounding_summary:
        lines.extend([
            "## Audit grounding summary",
            "",
            "```text",
            _truncate_text(_strip_unsafe_control_chars(grounding_summary), limit=5000),
            "```",
            "",
        ])
    lines.extend([
        "## Answer",
        "",
        answer_markdown.strip() or "_No answer returned._",
        "",
    ])
    paths["md"].write_text(_strip_unsafe_control_chars("\n".join(lines).strip() + "\n"), encoding="utf-8")
    return {k: str(v) for k, v in paths.items()}


def _extract_qa_answer_text(resp) -> str:
    raw_text = (getattr(resp, "output_text", None) or "").strip()
    if raw_text:
        return _strip_unsafe_control_chars(_repair_json_escape_artifacts(raw_text))

    raw = to_jsonable(resp)
    parts = []
    for item in raw.get("output", []) or []:
        if item.get("type") != "message":
            continue
        for content in item.get("content", []) or []:
            if content.get("type") == "output_text":
                txt = (content.get("text") or "").strip()
                if txt:
                    parts.append(txt)
    answer = "\n\n".join(parts).strip()
    if answer:
        return _strip_unsafe_control_chars(_repair_json_escape_artifacts(answer))
    raise ValueError("Could not locate answer text in the Q&A response.")


def _display_qa_answer(question: str, answer: str, mode: str) -> None:
    heading = "Answer about the paper" if mode == "paper" else "Answer about the audit"
    question_text = _strip_unsafe_control_chars(_repair_json_escape_artifacts(question))
    answer_text = _strip_unsafe_control_chars(_repair_json_escape_artifacts(answer))
    display(Markdown(f"### {heading}\n\n**Question:** {question_text}\n\n{answer_text}"))


def _qa_developer_prompt(mode: str) -> str:
    if mode == "audit":
        return (
            "You are answering follow-up questions about a completed mathematical paper audit. "
            "Use the ongoing conversation plus the supplied audit artifacts. "
            "Ground the answer in the saved audit state when possible, distinguish paper facts from audit findings, "
            "cite chunk ids or issue ids when they materially help, and say explicitly when the artifacts are insufficient."
        )
    return (
        "You are answering follow-up questions about a mathematical paper after a completed audit. "
        "Use the ongoing conversation context conservatively, answer clearly, preserve mathematical notation, "
        "and say when a claim cannot be supported from the paper context you have."
    )


def _qa_tokens(text: str) -> list[str]:
    tokens = []
    seen = set()
    for token in re.findall(r"[A-Za-z0-9_.:-]+", str(text).lower()):
        if token in _QA_STOPWORDS:
            continue
        if len(token) < 3 and not re.fullmatch(r"\d+(?:\.\d+)*", token):
            continue
        if token not in seen:
            seen.add(token)
            tokens.append(token)
    return tokens


def _qa_relevance_score(question: str, *parts: Any) -> int:
    haystack = " ".join(str(part or "") for part in parts).lower()
    if not haystack.strip():
        return 0
    score = 0
    for token in _qa_tokens(question):
        if token in haystack:
            score += 1 + min(haystack.count(token), 3)
    return score


def _resolve_qa_artifact_path(session: dict[str, Any], raw_path: Any, subdir: str) -> Optional[Path]:
    if not raw_path:
        return None
    candidate = Path(str(raw_path))
    if candidate.exists():
        return candidate
    fallback = Path(session["workdir"]) / subdir / candidate.name
    if fallback.exists():
        return fallback
    return None


def _load_chunk_audit_for_qa(session: dict[str, Any], rec: dict[str, Any]) -> dict[str, Any]:
    path = _resolve_qa_artifact_path(session, rec.get("structured_response_path"), "responses")
    if path is None:
        return {}
    try:
        return _coerce_audit_payload(load_json(path))
    except Exception:
        return {}


def _select_relevant_strings(question: str, items: list[str], limit: int = 6) -> list[str]:
    prepared = []
    for idx, item in enumerate(items or []):
        text = _strip_unsafe_control_chars(_repair_json_escape_artifacts(str(item or "").strip()))
        if not text:
            continue
        prepared.append((_qa_relevance_score(question, text), idx, text))
    prepared.sort(key=lambda item: (-item[0], item[1]))
    chosen = [text for score, _, text in prepared if score > 0][:limit]
    if chosen:
        return chosen
    return [text for _, _, text in prepared[:limit]]


def _select_relevant_issues(question: str, issues: list[dict[str, Any]], limit: int = 6) -> list[dict[str, Any]]:
    severity_rank = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    prepared = []
    for idx, issue in enumerate(issues or []):
        score = _qa_relevance_score(
            question,
            issue.get("issue_id"),
            issue.get("chunk_id"),
            issue.get("title"),
            issue.get("location"),
            issue.get("description"),
            issue.get("evidence"),
            issue.get("proposed_fix"),
        )
        prepared.append((
            score,
            severity_rank.get(str(issue.get("severity") or "medium").lower(), 2),
            idx,
            issue,
        ))
    prepared.sort(key=lambda item: (-item[0], -item[1], item[2]))
    chosen = [issue for score, _, _, issue in prepared if score > 0][:limit]
    if chosen:
        return chosen
    return [issue for _, _, _, issue in prepared[:limit]]


def _select_relevant_chunk_records(
    session: dict[str, Any],
    question: str,
    records: list[dict[str, Any]],
    issues_by_chunk: dict[str, list[dict[str, Any]]],
    limit: int = 4,
) -> list[dict[str, Any]]:
    prepared = []
    for idx, rec in enumerate(records or []):
        chunk_id = str(rec.get("chunk_id") or "")
        related_issues = issues_by_chunk.get(chunk_id, [])
        score = _qa_relevance_score(
            question,
            chunk_id,
            rec.get("label"),
            rec.get("boundary"),
            rec.get("verification_summary"),
            " ".join(issue.get("title", "") for issue in related_issues),
            " ".join(issue.get("location", "") for issue in related_issues),
        )
        prepared.append((score, idx, rec))
    prepared.sort(key=lambda item: (-item[0], item[1]))
    chosen = [rec for score, _, rec in prepared if score > 0][:limit]
    if chosen:
        return chosen
    return [rec for _, _, rec in prepared[:limit]]


def _build_audit_qa_context(session: dict[str, Any], question: str, max_chars: int = 12000) -> str:
    ledger = load_ledger(session)
    issues_state = load_issues(session)
    status = load_status(session)
    records = read_chunk_records(session)
    open_issues = [
        issue for issue in (issues_state.get("issues") or [])
        if str(issue.get("status") or "open").lower() != "resolved"
    ]
    issues_by_chunk: dict[str, list[dict[str, Any]]] = {}
    for issue in open_issues:
        issues_by_chunk.setdefault(str(issue.get("chunk_id") or ""), []).append(issue)

    relevant_assumptions = _select_relevant_strings(question, ledger.get("assumptions") or [], limit=5)
    relevant_notes = _select_relevant_strings(question, ledger.get("notes") or [], limit=5)
    relevant_issues = _select_relevant_issues(question, open_issues, limit=6)
    relevant_records = _select_relevant_chunk_records(session, question, records, issues_by_chunk, limit=4)

    report_files = []
    reports_dir = Path(session["workdir"]) / "reports"
    if reports_dir.exists():
        for path in sorted(reports_dir.glob("*")):
            if path.is_file():
                report_files.append(path.name)

    lines = [
        "Completed audit context:",
        f"- PDF path: {session.get('pdf_path', '')}",
        f"- Workdir: {session.get('workdir', '')}",
        f"- Audit status: {status.get('status', '')}",
        f"- Chunks completed: {status.get('chunks_completed', 0)}/{status.get('chunks_total', 0)}",
        f"- Open issues tracked: {len(open_issues)}",
        "",
        "Ledger assumptions:",
    ]
    if relevant_assumptions:
        lines.extend(f"- {item}" for item in relevant_assumptions)
    else:
        lines.append("- None recorded.")
    lines.extend(["", "Ledger notes:"])
    if relevant_notes:
        lines.extend(f"- {item}" for item in relevant_notes)
    else:
        lines.append("- None recorded.")

    lines.extend(["", "Open issues most relevant to the question:"])
    if relevant_issues:
        for issue in relevant_issues:
            lines.extend([
                f"- {issue.get('issue_id', 'issue')} | {issue.get('severity', 'medium')} | {issue.get('title', '')}",
                f"  chunk: {issue.get('chunk_id', '')}",
                f"  location: {issue.get('location', '')}",
                f"  description: {_truncate_text(issue.get('description', ''), limit=350)}",
                f"  proposed fix: {_truncate_text(issue.get('proposed_fix', ''), limit=220)}",
            ])
    else:
        lines.append("- No open issues were found in the saved audit state.")

    lines.extend(["", "Relevant chunk summaries:"])
    if relevant_records:
        for rec in relevant_records:
            chunk_id = str(rec.get("chunk_id") or "")
            audit = _load_chunk_audit_for_qa(session, rec)
            lines.extend([
                f"- {chunk_id} | {rec.get('label', '') or rec.get('boundary', '')}",
                f"  boundary: {rec.get('boundary', '')}",
                f"  verification mode: {rec.get('verification_mode', '') or 'local_python_only'}",
            ])
            if rec.get("issue_ids"):
                lines.append(f"  issue ids: {', '.join(str(x) for x in rec.get('issue_ids') or [])}")
            if audit.get("verified_steps"):
                lines.append(
                    "  verified steps: " + "; ".join(
                        _truncate_text(step, limit=140) for step in (audit.get("verified_steps") or [])[:3]
                    )
                )
            if audit.get("assumptions_notation"):
                lines.append(
                    "  assumptions: " + "; ".join(
                        _truncate_text(item, limit=140) for item in (audit.get("assumptions_notation") or [])[:2]
                    )
                )
            chunk_issues = audit.get("issues_found") or []
            if chunk_issues:
                lines.append(
                    "  chunk issues: " + "; ".join(
                        _truncate_text(f"{issue.get('severity', 'medium')}: {issue.get('title', '')}", limit=140)
                        for issue in chunk_issues[:3]
                    )
                )
            if rec.get("verification_summary"):
                lines.append(f"  verification summary: {_truncate_text(rec.get('verification_summary', ''), limit=220)}")
    else:
        lines.append("- No chunk summaries were available.")

    lines.extend(["", "Available report files:"])
    if report_files:
        lines.extend(f"- {name}" for name in report_files)
    else:
        lines.append("- No reports directory found yet.")

    context = _strip_unsafe_control_chars(_repair_json_escape_artifacts("\n".join(lines).strip()))
    return _truncate_text(context, limit=max_chars)


def _qa_request_input(
    session: dict[str, Any],
    question: str,
    mode: str,
    context_text: str = "",
) -> tuple[list[dict[str, Any]], bool]:
    content = []
    attached_pdf = False
    if not session.get("pdf_attached_in_conversation", False) and session.get("pdf_file_id"):
        content.append({"type": "input_file", "file_id": session["pdf_file_id"]})
        attached_pdf = True

    clean_question = _strip_unsafe_control_chars(_repair_json_escape_artifacts(str(question or "").strip()))
    if mode == "audit":
        prompt_text = "\n\n".join([
            "Question about the completed audit:",
            clean_question,
            "Audit artifact context:",
            context_text or "(no saved audit context available)",
            "Answer using the audit artifacts above when they are relevant. Distinguish audit findings from direct paper claims.",
        ])
    else:
        prompt_text = clean_question

    content.append({"type": "input_text", "text": prompt_text})
    return [
        {"role": "developer", "content": [{"type": "input_text", "text": _qa_developer_prompt(mode)}]},
        {"role": "user", "content": content},
    ], attached_pdf


def _run_qa_turn(
    session_or_pdf_path: dict[str, Any] | str | Path,
    question: str,
    mode: str,
    model: Optional[str] = None,
    reasoning_effort: Optional[str] = None,
    save: bool = True,
) -> dict[str, Any]:
    session = _resolve_qa_session(session_or_pdf_path)
    _assert_qa_ready(session)
    session = _ensure_qa_conversation(session)

    clean_question = _strip_unsafe_control_chars(_repair_json_escape_artifacts(str(question or "").strip()))
    if not clean_question:
        raise ValueError("question must be a non-empty string")

    qa_model = model or session.get("model") or DEFAULT_MODEL
    qa_effort = reasoning_effort or session.get("reasoning_effort") or DEFAULT_REASONING_EFFORT
    grounding_summary = ""
    if mode == "audit":
        grounding_summary = _build_audit_qa_context(session, clean_question)
    input_payload, attached_pdf = _qa_request_input(session, clean_question, mode, grounding_summary)

    resp = client.responses.create(
        model=qa_model,
        reasoning={"effort": qa_effort},
        conversation=session["conversation_id"],
        input=input_payload,
        background=False,
        store=bool(session.get("store", True)),
    )
    if getattr(resp, "status", None) in WORKING_STATUSES:
        resp = wait_for_response(resp.id, poll_every=2.0, max_wait_seconds=None)
    if getattr(resp, "status", None) != "completed":
        raise RuntimeError(f"Q&A response ended with status={getattr(resp, 'status', None)}")

    answer = _extract_qa_answer_text(resp)
    _display_qa_answer(clean_question, answer, mode)

    turn = {
        "time": utc_now(),
        "turn_id": None,
        "mode": mode,
        "question": clean_question,
        "answer": answer,
        "model": qa_model,
        "reasoning_effort": qa_effort,
        "response_id": getattr(resp, "id", None),
        "conversation_id": session.get("conversation_id"),
        "grounding_summary": grounding_summary if mode == "audit" else "",
    }

    path_info = {}
    if save:
        idx = _next_qa_index(session)
        turn["turn_id"] = f"qa_{idx:03d}"
        path_info = _save_qa_turn(session, turn, answer, idx)
    if attached_pdf:
        session["pdf_attached_in_conversation"] = True
    session["updated_at"] = utc_now()
    save_session(session)

    result = dict(turn)
    if path_info:
        result["paths"] = path_info
    return result


def ask_about_paper(
    session_or_pdf_path: dict[str, Any] | str | Path,
    question: str,
    model: Optional[str] = None,
    reasoning_effort: Optional[str] = None,
    save: bool = True,
) -> dict[str, Any]:
    result = runtime_ask_about_paper(
        session_or_pdf_path,
        question,
        model=model,
        reasoning_effort=reasoning_effort,
        save=save,
    )
    _display_qa_answer(result.get("question", question), result.get("answer", ""), "paper")
    return result


def ask_about_audit(
    session_or_pdf_path: dict[str, Any] | str | Path,
    question: str,
    model: Optional[str] = None,
    reasoning_effort: Optional[str] = None,
    save: bool = True,
) -> dict[str, Any]:
    result = runtime_ask_about_audit(
        session_or_pdf_path,
        question,
        model=model,
        reasoning_effort=reasoning_effort,
        save=save,
    )
    _display_qa_answer(result.get("question", question), result.get("answer", ""), "audit")
    return result


def show_qa_history(session_or_pdf_path: dict[str, Any] | str | Path) -> list[dict[str, Any]]:
    session = _runtime_resolve_qa_session(session_or_pdf_path)
    turns = _runtime_load_qa_turns(session)
    if not turns:
        display(Markdown(f"### Saved Q&A history\n\n_No saved Q&A turns found for_ `{session['pdf_path']}`."))
        return []

    lines = [f"### Saved Q&A history -- {Path(session['pdf_path']).stem}", ""]
    for turn in turns:
        lines.extend([
            f"#### {turn.get('turn_id') or Path(str(turn.get('turn_path',''))).stem}",
            f"- Time: {turn.get('time', '')}",
            f"- Mode: {turn.get('mode', '')}",
            f"- Model: {turn.get('model', '')}",
            f"- Question: {_truncate_text(turn.get('question', ''), limit=220)}",
            f"- Answer preview: {_truncate_text(turn.get('answer', ''), limit=260)}",
            "",
        ])
    display(Markdown(_strip_unsafe_control_chars("\n".join(lines).strip() + "\n")))
    return turns


def _qa_report_markdown(session: dict[str, Any], turns: list[dict[str, Any]]) -> str:
    lines = [
        f"# Q&A appendix -- {Path(session['pdf_path']).stem}",
        "",
        f"- PDF: {session['pdf_path']}",
        f"- Workdir: {session['workdir']}",
        f"- Saved turns: {len(turns)}",
        "",
    ]
    if not turns:
        lines.append("No saved Q&A turns found.")
        return _strip_unsafe_control_chars("\n".join(lines).strip() + "\n")

    for turn in turns:
        lines.extend([
            f"## {turn.get('turn_id', 'qa_turn')} [{turn.get('mode', '')}]",
            f"- Time: {turn.get('time', '')}",
            f"- Model: {turn.get('model', '')}",
            f"- Reasoning effort: {turn.get('reasoning_effort', '')}",
            f"- Response ID: {turn.get('response_id', '') or 'n/a'}",
            "",
            "### Question",
            "",
            _strip_unsafe_control_chars(turn.get("question", "")),
            "",
        ])
        grounding_summary = str(turn.get("grounding_summary") or "").strip()
        if grounding_summary:
            lines.extend([
                "### Audit grounding summary",
                "",
                "```text",
                _truncate_text(_strip_unsafe_control_chars(grounding_summary), limit=5000),
                "```",
                "",
            ])
        lines.extend([
            "### Answer",
            "",
            _strip_unsafe_control_chars(turn.get("answer", "")) or "_No answer returned._",
            "",
        ])
    return _strip_unsafe_control_chars("\n".join(lines).strip() + "\n")


def _qa_text_to_tex_blocks(text: str) -> str:
    cleaned = _strip_unsafe_control_chars(_repair_json_escape_artifacts(str(text or "").strip()))
    if not cleaned:
        return report_latex_paragraph("(empty)") + "\n"
    parts = []
    for para in re.split(r"\n\s*\n", cleaned):
        para = para.strip()
        if not para:
            continue
        parts.append(report_latex_paragraph(para) + "\n\n")
    return "".join(parts) or (report_latex_paragraph(cleaned) + "\n")


def _qa_report_tex(session: dict[str, Any], turns: list[dict[str, Any]]) -> str:
    title = report_latex_paragraph(f"Q&A appendix -- {Path(session['pdf_path']).stem}")
    parts = [r"""\documentclass[11pt]{article}
\usepackage[a4paper,margin=1in]{geometry}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage{lmodern}
\usepackage{microtype}
\usepackage{hyperref}
\usepackage{enumitem}
\usepackage{xcolor}
\usepackage{fancyvrb}
\setlist[itemize]{leftmargin=2em}
\setlength{\parskip}{0.5em}
\setlength{\parindent}{0pt}
\begin{document}
"""]
    parts.append(r"\section*{" + title + "}" + "\n")
    parts.append(r"\begin{itemize}" + "\n")
    parts.append(r"\item PDF: " + report_latex_paragraph(session["pdf_path"]) + "\n")
    parts.append(r"\item Workdir: " + report_latex_paragraph(session["workdir"]) + "\n")
    parts.append(r"\item Saved turns: " + str(len(turns)) + "\n")
    parts.append(r"\end{itemize}" + "\n")
    if not turns:
        parts.append(report_latex_paragraph("No saved Q&A turns found.") + "\n")
    else:
        for turn in turns:
            heading = report_latex_paragraph(f"{turn.get('turn_id', 'qa_turn')} [{turn.get('mode', '')}]")
            parts.append(r"\subsection*{" + heading + "}" + "\n")
            parts.append(r"\begin{itemize}" + "\n")
            parts.append(r"\item Time: " + report_latex_paragraph(str(turn.get("time", ""))) + "\n")
            parts.append(r"\item Model: " + report_latex_paragraph(str(turn.get("model", ""))) + "\n")
            parts.append(r"\item Reasoning effort: " + report_latex_paragraph(str(turn.get("reasoning_effort", ""))) + "\n")
            parts.append(r"\item Response ID: " + report_latex_paragraph(str(turn.get("response_id", "") or "n/a")) + "\n")
            parts.append(r"\end{itemize}" + "\n")
            parts.append(r"\paragraph{Question}" + "\n")
            parts.append(_qa_text_to_tex_blocks(turn.get("question", "")))
            grounding_summary = str(turn.get("grounding_summary") or "").strip()
            if grounding_summary:
                parts.append(r"\paragraph{Audit grounding summary}" + "\n")
                parts.append(_verbatim_block(_truncate_text(grounding_summary, limit=5000)) + "\n")
            parts.append(r"\paragraph{Answer}" + "\n")
            parts.append(_qa_text_to_tex_blocks(turn.get("answer", "")))
    parts.append(r"\end{document}" + "\n")
    return _strip_unsafe_control_chars("".join(parts))


def rebuild_qa_report(session_or_pdf_path: dict[str, Any] | str | Path) -> dict[str, str]:
    return runtime_rebuild_qa_report(session_or_pdf_path)
````

</details>

### Archived legacy notebook cell 23 (non-executing)

This historical production/frontend cell has been intentionally disabled so the notebook cannot shadow canonical backend behavior.

Canonical owner now: `audit_policy_hooks.py`.

The notebook is now a maintenance/debug frontend. If this logic is needed again, update or call the backend module rather than reactivating notebook-local copies.

<details>
<summary>Archived original code cell</summary>

````python
# Post-audit reference-style report patch: best-effort prose-only rewriting for final reports.

_REPORT_REFERENCE_STYLES = {"match_audit", "compiled_pdf_numbers", "source_labels"}
_OLD_BUILD_FINAL_REPORT_MARKDOWN_WITH_REFERENCE_STYLE = build_final_report_markdown
_OLD_BUILD_FINAL_REPORT_TEX_WITH_REFERENCE_STYLE = build_final_report_tex
_OLD_BUILD_FINAL_REPORT_WITH_REFERENCE_STYLE = build_final_report


def _normalize_report_reference_style(style: Optional[str]) -> str:
    aliases = {
        "match": "match_audit",
        "audit": "match_audit",
        "compiled": "compiled_pdf_numbers",
        "numbers": "compiled_pdf_numbers",
        "pdf": "compiled_pdf_numbers",
        "pdf_numbers": "compiled_pdf_numbers",
        "label": "source_labels",
        "labels": "source_labels",
        "source": "source_labels",
    }
    normalized = "match_audit" if style is None else str(style).strip().lower()
    normalized = aliases.get(normalized, normalized)
    if normalized not in _REPORT_REFERENCE_STYLES:
        allowed = ", ".join(sorted(_REPORT_REFERENCE_STYLES))
        raise ValueError(f"report_reference_style must be one of: {allowed}")
    return normalized


def _effective_report_reference_style(session: dict[str, Any], style: Optional[str] = None) -> str:
    requested = _normalize_report_reference_style(style or session.get("report_reference_style", "match_audit"))
    if requested != "match_audit":
        return requested
    ref_state = ensure_reference_map(session)
    if _reference_map_has_valid_aux_numbers(ref_state):
        return "compiled_pdf_numbers"
    return "match_audit"


def _reference_report_status(session: dict[str, Any], style: Optional[str] = None) -> dict[str, Any]:
    requested = _normalize_report_reference_style(style or session.get("report_reference_style", "match_audit"))
    effective = _effective_report_reference_style(session, style=requested)
    ref_state = ensure_reference_map(session)
    label_map = ref_state.get("label_map", {}) if isinstance(ref_state, dict) else {}
    map_source = str(ref_state.get("map_source") or "none") if isinstance(ref_state, dict) else "none"
    warning = normalize_whitespace(ref_state.get("warning", "")) if isinstance(ref_state, dict) else ""
    valid_aux = _reference_map_has_valid_aux_numbers(ref_state)

    lines: list[str] = []
    if requested != "source_labels":
        if label_map and effective == "compiled_pdf_numbers" and not valid_aux:
            lines = [
                "Compiled-style references in this report are using a cached or non-authoritative reference map rather than a freshly parsed AUX map.",
                f"Reference map source: {map_source}.",
                f"Recovered labels available: {len(label_map)}.",
                "Verify numbering against the current compiled PDF if the paper was recompiled after the cached map was created.",
            ]
        elif not label_map:
            lines = [
                "Compiled PDF numbering could not be applied for this report.",
                f"Reference map source: {map_source}.",
                "Fallback mode: label-based references are preserved when available; otherwise the original audit wording is kept.",
            ]
        elif effective != "compiled_pdf_numbers":
            lines = [
                "A valid AUX-derived compiled numbering map was not available, so compiled numbering was not applied automatically in this report.",
                f"Reference map source: {map_source}.",
                "Fallback mode: label-based references are preserved when available; otherwise the original audit wording is kept.",
            ]
    if warning:
        lines.append(f"Warning: {warning}")
    return {
        "requested_style": requested,
        "effective_style": effective,
        "map_source": map_source,
        "label_count": len(label_map) if isinstance(label_map, dict) else 0,
        "lines": lines,
    }


def _reference_report_status_markdown(session: dict[str, Any], style: Optional[str] = None) -> str:
    lines = _reference_report_status(session, style=style).get("lines") or []
    if not lines:
        return ""
    return "## Reference numbering status\n\n" + "\n".join(f"- {line}" for line in lines) + "\n\n"


def _reference_report_status_tex(session: dict[str, Any], style: Optional[str] = None) -> str:
    lines = _reference_report_status(session, style=style).get("lines") or []
    if not lines:
        return ""
    parts = [r"\section*{Reference numbering status}", r"\begin{itemize}"]
    for line in lines:
        parts.append(r"\item " + report_latex_paragraph(line))
    parts.append(r"\end{itemize}")
    return "\n".join(parts) + "\n"


def _inject_markdown_reference_status(text: str, block: str) -> str:
    if not block:
        return text
    m = re.search(r"^##\s", text, flags=re.MULTILINE)
    if m:
        return text[:m.start()] + block + text[m.start():]
    return text.rstrip() + "\n\n" + block


def _inject_tex_reference_status(text: str, block: str) -> str:
    if not block:
        return text
    m = re.search(r"\\end\{itemize\}\n", text)
    if m:
        return text[:m.end()] + block + text[m.end():]
    marker = r"\section*{Ledger assumptions}"
    if marker in text:
        return text.replace(marker, block + marker, 1)
    if r"\end{document}" in text:
        return text.replace(r"\end{document}", block + r"\end{document}", 1)
    return text + "\n" + block


def _reference_rewrite_maps(session: dict[str, Any]) -> tuple[dict[str, dict[str, str]], dict[tuple[str, str], str]]:
    ref_state = ensure_reference_map(session)
    label_map = ref_state.get("label_map", {}) if isinstance(ref_state, dict) else {}
    reverse: dict[tuple[str, str], list[str]] = {}
    for label, info in label_map.items():
        kind = (info.get("kind") or _infer_kind_from_label(label)).strip().lower()
        number = (info.get("number") or "").strip()
        if not kind or not number:
            continue
        reverse.setdefault((kind, number), []).append(label)
    unique_reverse = {
        key: labels[0]
        for key, labels in reverse.items()
        if len(labels) == 1
    }
    return label_map, unique_reverse


def _reference_phrase_from_match(match_text: str, style: str, kind: str, label: str, display: str) -> str:
    style = _normalize_report_reference_style(style)
    starts_upper = bool(match_text[:1]) and match_text[:1].isupper()
    has_the = match_text.lower().startswith("the ")
    if style == "compiled_pdf_numbers":
        phrase = display.strip() or match_text
        if has_the and phrase[:1].islower():
            phrase = "the " + phrase
        if starts_upper and phrase[:1].islower():
            phrase = phrase[0].upper() + phrase[1:]
        return phrase
    base_kind = (kind or _infer_kind_from_label(label)).strip().lower() or "item"
    phrase = f"{base_kind} labeled {label}"
    if has_the:
        phrase = "the " + phrase
    if starts_upper:
        phrase = phrase[0].upper() + phrase[1:]
    return phrase


def _rewrite_reference_mentions_in_prose(text: str, session: dict[str, Any], style: Optional[str] = None) -> str:
    style = _effective_report_reference_style(session, style=style)
    if style == "match_audit":
        return text
    label_map, reverse_map = _reference_rewrite_maps(session)
    if not label_map:
        return text

    rewritten = str(text)
    if style == "compiled_pdf_numbers":
        items = sorted(label_map.items(), key=lambda item: len(item[0]), reverse=True)
        for label, info in items:
            display = (info.get("display") or "").strip()
            kind = (info.get("kind") or _infer_kind_from_label(label)).strip().lower() or "item"
            if not display:
                continue
            pattern = re.compile(
                rf"(?<![\w:])((?:the\s+)?{re.escape(kind)}\s+labeled\s+{re.escape(label)})\b",
                flags=re.IGNORECASE,
            )
            rewritten = pattern.sub(
                lambda m: _reference_phrase_from_match(m.group(1), style, kind, label, display),
                rewritten,
            )
        return rewritten

    items = []
    for (kind, number), label in reverse_map.items():
        display = _display_for_kind(kind, number, label)
        if display:
            items.append((display, kind, label))
    items.sort(key=lambda item: len(item[0]), reverse=True)
    for display, kind, label in items:
        pattern = re.compile(
            rf"(?<![\w:])((?:the\s+)?{re.escape(display)})\b",
            flags=re.IGNORECASE,
        )
        rewritten = pattern.sub(
            lambda m: _reference_phrase_from_match(m.group(1), style, kind, label, display),
            rewritten,
        )
    return rewritten


def _rewrite_markdown_report_references(text: str, session: dict[str, Any], style: Optional[str] = None) -> str:
    style = _effective_report_reference_style(session, style=style)
    if style == "match_audit":
        return text
    parts = re.split(r"(```.*?```)", text, flags=re.DOTALL)
    out = []
    for part in parts:
        if part.startswith("```") and part.endswith("```"):
            out.append(part)
        else:
            out.append(_rewrite_reference_mentions_in_prose(part, session, style=style))
    return "".join(out)


def _rewrite_tex_report_references(text: str, session: dict[str, Any], style: Optional[str] = None) -> str:
    style = _effective_report_reference_style(session, style=style)
    if style == "match_audit":
        return text
    parts = re.split(r"(\\begin\{Verbatim\}.*?\\end\{Verbatim\})", text, flags=re.DOTALL)
    out = []
    for part in parts:
        if part.startswith(r"\begin{Verbatim}") and part.endswith(r"\end{Verbatim}"):
            out.append(part)
        else:
            out.append(_rewrite_reference_mentions_in_prose(part, session, style=style))
    return "".join(out)


def build_final_report_markdown(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    style = _effective_report_reference_style(session)
    text = _OLD_BUILD_FINAL_REPORT_MARKDOWN_WITH_REFERENCE_STYLE(session, report_title=report_title)
    text = _rewrite_markdown_report_references(text, session, style=style)
    return _inject_markdown_reference_status(text, _reference_report_status_markdown(session, style=style))


def build_final_report_tex(session: dict[str, Any], report_title: Optional[str] = None) -> str:
    style = _effective_report_reference_style(session)
    text = _OLD_BUILD_FINAL_REPORT_TEX_WITH_REFERENCE_STYLE(session, report_title=report_title)
    text = _rewrite_tex_report_references(text, session, style=style)
    return _inject_tex_reference_status(text, _reference_report_status_tex(session, style=style))


def build_final_report(
    session_or_pdf: dict[str, Any] | str | Path,
    report_title: Optional[str] = None,
    include_verification_summary_in_final_report: Optional[bool] = None,
    write_separate_verification_report: Optional[bool] = None,
    report_reference_style: Optional[str] = None,
) -> dict[str, str]:
    session = session_or_pdf if isinstance(session_or_pdf, dict) else load_session_from_pdf(session_or_pdf)
    if session is None:
        raise FileNotFoundError("No audit session found for this PDF.")
    if report_reference_style is not None:
        session["report_reference_style"] = _normalize_report_reference_style(report_reference_style)
        session["updated_at"] = utc_now()
        save_session(session)
    return _OLD_BUILD_FINAL_REPORT_WITH_REFERENCE_STYLE(
        session,
        report_title=report_title,
        include_verification_summary_in_final_report=include_verification_summary_in_final_report,
        write_separate_verification_report=write_separate_verification_report,
    )


def rebuild_reports_with_reference_style(
    session_or_pdf_path: dict[str, Any] | str | Path,
    report_reference_style: str = "compiled_pdf_numbers",
    report_title: Optional[str] = None,
    include_verification_summary_in_final_report: Optional[bool] = None,
    write_separate_verification_report: Optional[bool] = None,
) -> dict[str, str]:
    style = _normalize_report_reference_style(report_reference_style)
    return build_final_report(
        session_or_pdf_path,
        report_title=report_title,
        include_verification_summary_in_final_report=include_verification_summary_in_final_report,
        write_separate_verification_report=write_separate_verification_report,
        report_reference_style=style,
    )
````

</details>

In [20]:
# ============================================================
# Example run commands for paper audit (documentation cell)
# ============================================================
# This cell is intentionally "comment-only" guidance.
# Copy one example, uncomment it, then run.
#
# Quick option guide:
# - continue_existing:
#     True  -> resume existing *_audit session if found
#     False -> start new session for this PDF
#
# - verification_mode:
#     "local_python_only" (default): model suggests local Python checks only
#     "code_interpreter": enable Responses API Code Interpreter tool for selected chunks
#     "none": ask model to skip optional numeric/symbolic verification
#
# - use_code_interpreter:
#     Must be True when verification_mode="code_interpreter"
#
# - verification_chunk_indices:
#     Optional list of 1-based chunk numbers. If provided, only those chunks use
#     verification_mode; all other chunks fall back to local_python_only.
#
# - code_interpreter_memory_limit:
#     Optional string, default "4g" (e.g., "4g", "8g")
#
# - reuse_code_interpreter_container:
#     Optional bool. If True, notebook keeps discovered container id for reuse metadata.
#
# - reference_mention_style:
#     "auto" (default): prefer compiled numbering when confirmed; otherwise fall back safely
#     "compiled_pdf_numbers": prefer compiled/PDF-visible numbering whenever it is known
#     "source_labels": prefer LaTeX source labels when the chunk has them
#
# - report_reference_style:
#     "match_audit" (default): preserve the saved chunk wording in rebuilt reports
#     "compiled_pdf_numbers": best-effort prose rewrite to compiled numbering when AUX mapping exists
#     "source_labels": best-effort prose rewrite back to source labels when mapping is unique

# ---------- Preflight (recommended) ----------
# import inspect
# print(inspect.signature(audit_the_paper))


# ---------- Set your paper path ----------
# PDF_PATH = "/absolute/path/to/your_paper.pdf"


# ---------- Example 1: Fresh full audit, default local verification ----------
# result = start_fresh_audit(
#     PDF_PATH,
#     model="gpt-5.5",
#     reasoning_effort="medium",
# )


# ---------- Example 2: Resume existing audit session ----------
# result = audit_the_paper(
#     PDF_PATH,
#     model="gpt-5.5",
#     reasoning_effort="xhigh",
#     continue_existing=True,
# )


# ---------- Example 3: Fresh audit with Code Interpreter on all chunks ----------
# result = start_fresh_audit(
#     PDF_PATH,
#     model="gpt-5.5",
#     reasoning_effort="xhigh",
#     use_code_interpreter=True,
#     code_interpreter_memory_limit="4g",
#     reuse_code_interpreter_container=False,
#     verification_mode="code_interpreter",
# )


# ---------- Example 4: Mixed mode (Code Interpreter only on selected chunks) ----------
# result = start_fresh_audit(
#     PDF_PATH,
#     model="gpt-5.5",
#     reasoning_effort="xhigh",
#     use_code_interpreter=True,
#     verification_mode="code_interpreter",
#     verification_chunk_indices=[3, 7, 12],  # 1-based chunk indices
# )


# ---------- Example 5: Continue audit but disable optional verification ----------
# result = audit_the_paper(
#     PDF_PATH,
#     continue_existing=True,
#     verification_mode="none",
# )


# ---------- Example 6: Run a single chosen chunk with Code Interpreter ----------
# session = load_session_from_pdf(PDF_PATH)
# manifest = load_manifest(session)
# chunk = manifest["chunks"][2]  # chunk #3 (0-based indexing here)
# one_chunk = process_one_chunk(
#     session,
#     chunk,
#     verification_mode="code_interpreter",
# )

# ---------- Example 7: Fresh audit, then run generated verification scripts locally ----------
# result = start_fresh_audit(
#     PDF_PATH,
#     model="gpt-5.5",
#     reasoning_effort="xhigh",
#     run_generated_verification=True,
#     verification_timeout_seconds=120,
# )


# ---------- Example 8: Run verification later on an existing audit ----------
# verification = run_verification_suite(
#     PDF_PATH,
#     timeout=120,
#     safe_only=True,
# )
# verification


# ---------- Example 9: Build the separate verification report ----------
# verification_report_paths = build_verification_report(PDF_PATH)
# verification_report_paths


# ---------- Example 10: Rerun only failed or timed-out verification scripts ----------
# rerun = rerun_failed_verification_scripts(
#     PDF_PATH,
#     timeout=120,
#     safe_only=True,
# )
# rerun

# ---------- Example 11: Backfill descriptions for Python checks in an older audit ----------
# backfill = backfill_python_check_descriptions(
#     PDF_PATH,
#     rebuild_reports=True,
#     rewrite_chunk_markdown=True,
# )
# backfill


# ---------- Example 12: Ask a follow-up question about the paper ----------
# paper_answer = ask_about_paper(
#     PDF_PATH,
#     "What is the main claim of the paper, and how is the argument structured?",
# )
# paper_answer


# ---------- Example 13: Ask a follow-up question about the audit findings ----------
# audit_answer = ask_about_audit(
#     PDF_PATH,
#     "What are the most serious issues found in Theorem 2.1?",
# )
# audit_answer


# ---------- Example 14: Show saved Q&A history ----------
# qa_history = show_qa_history(PDF_PATH)
# qa_history


# ---------- Example 15: Build a separate Q&A appendix report ----------
# qa_report_paths = rebuild_qa_report(PDF_PATH)
# qa_report_paths


# ---------- Example 16: Fresh audit with compiled/PDF numbering preferred ----------
# result = start_fresh_audit(
#     PDF_PATH,
#     model="gpt-5.5",
#     reasoning_effort="xhigh",
#     reference_mention_style="compiled_pdf_numbers",
# )
# result


# ---------- Example 17: Fresh audit with source labels preferred ----------
# result = start_fresh_audit(
#     PDF_PATH,
#     model="gpt-5.5",
#     reasoning_effort="xhigh",
#     reference_mention_style="source_labels",
# )
# result


# ---------- Example 18: Rebuild the final report using compiled-number references ----------
# report_paths = rebuild_reports_with_reference_style(
#     PDF_PATH,
#     report_reference_style="compiled_pdf_numbers",
# )
# report_paths


### Archived legacy notebook cell 25 (bootstrap moved)

The active maintenance bootstrap now lives near the top of the notebook. This old bootstrap is preserved as non-executing reference only.

<details>
<summary>Archived original code cell</summary>

````python
# Runtime live-audit facade bootstrap: keep the notebook as a thin client.

from audit_hooks import install_runtime_frontend
from audit_policy_hooks import build_final_report, build_user_message_for_chunk

install_runtime_frontend(
    globals(),
    client=client,
    prompt_builder=build_user_message_for_chunk,
    final_report_builder=build_final_report,
    display_audit=display_audit,
)
````

</details>

In [22]:
from audit_runtime import get_audit_status

PDF_PATH = "/Users/vytaszacharovas/CodexProjects/MATH_AUDIT/examples/Draft__Asymptotics_of_the_Stirling_numbers/elem-stir2-2026-hk-version.pdf"

info = get_audit_status(PDF_PATH, include_manifest=True)
info["status"], info["session"]["pending"], info["usage"]["totals"]

({'status': 'running',
  'progress_pct': 42.0,
  'current_chunk_id': 'chunk_048',
  'chunks_completed': 47,
  'chunks_total': 84,
  'estimated_pages_completed': 22,
  'estimated_pages_total': 51,
  'cost_usd': 14.001818999999998,
  'updated_at': '2026-04-27T09:08:33.348011+00:00',
  'total_audit_seconds': 13053.123606000001,
  'current_chunk_elapsed_seconds': 0.0,
  'audit_started_at': '2026-04-27T05:30:55.537298+00:00',
  'audit_finished_at': None,
  'pause_requested': False,
  'pause_requested_at': None},
 {'chunk_id': 'chunk_048',
  'response_id': 'resp_0a3e4d0362962ba60069ef278ed08c81949009d2c7fbb05431',
  'created_at': '2026-04-27T09:08:30.670583+00:00',
  'started_at': '2026-04-27T09:08:30.670583+00:00',
  'verification_mode': 'local_python_only',
  'used_code_interpreter_tool': False,
  'request_path': '/Users/vytaszacharovas/CodexProjects/MATH_AUDIT/examples/Draft__Asymptotics_of_the_Stirling_numbers/elem-stir2-2026-hk-version_audit/requests/chunk_048_20260427T0908306706340000_

In [23]:
from audit_runtime import get_audit_status

PDF_PATH = "/Users/vytaszacharovas/CodexProjects/MATH_AUDIT/examples/Draft__Asymptotics_of_the_Stirling_numbers/elem-stir2-2026-hk-version.pdf"
info = get_audit_status(PDF_PATH, include_manifest=True)

print("status:", info["status"]["status"])
print("current chunk:", info["status"]["current_chunk_id"])
print("chunks:", info["status"]["chunks_completed"], "/", info["status"]["chunks_total"])
print("pending:", info["session"]["pending"])
print("cost:", info["usage"]["totals"]["cost_usd"])

status: running
current chunk: chunk_048
chunks: 47 / 84
pending: {'chunk_id': 'chunk_048', 'response_id': 'resp_0a3e4d0362962ba60069ef278ed08c81949009d2c7fbb05431', 'created_at': '2026-04-27T09:08:30.670583+00:00', 'started_at': '2026-04-27T09:08:30.670583+00:00', 'verification_mode': 'local_python_only', 'used_code_interpreter_tool': False, 'request_path': '/Users/vytaszacharovas/CodexProjects/MATH_AUDIT/examples/Draft__Asymptotics_of_the_Stirling_numbers/elem-stir2-2026-hk-version_audit/requests/chunk_048_20260427T0908306706340000_initial.request.json'}
cost: 14.001818999999998


In [24]:



resp = client.responses.retrieve("resp_0a3e4d0362962ba60069ef278ed08c81949009d2c7fbb05431")

print("status:", getattr(resp, "status", None))
print("error:", getattr(resp, "error", None))
print("incomplete_details:", getattr(resp, "incomplete_details", None))

status: queued
error: None
incomplete_details: None


In [25]:
from audit_runtime import get_audit_status

PDF_PATH = "/Users/vytaszacharovas/CodexProjects/MATH_AUDIT/examples/Draft__Asymptotics_of_the_Stirling_numbers/elem-stir2-2026-hk-version.pdf"
info = get_audit_status(PDF_PATH, include_manifest=True)
info["status"], info["session"]["pending"], info["usage"]["totals"]

({'status': 'running',
  'progress_pct': 42.0,
  'current_chunk_id': 'chunk_048',
  'chunks_completed': 47,
  'chunks_total': 84,
  'estimated_pages_completed': 22,
  'estimated_pages_total': 51,
  'cost_usd': 14.001818999999998,
  'updated_at': '2026-04-27T09:08:33.348011+00:00',
  'total_audit_seconds': 13053.123606000001,
  'current_chunk_elapsed_seconds': 0.0,
  'audit_started_at': '2026-04-27T05:30:55.537298+00:00',
  'audit_finished_at': None,
  'pause_requested': False,
  'pause_requested_at': None},
 {'chunk_id': 'chunk_048',
  'response_id': 'resp_0a3e4d0362962ba60069ef278ed08c81949009d2c7fbb05431',
  'created_at': '2026-04-27T09:08:30.670583+00:00',
  'started_at': '2026-04-27T09:08:30.670583+00:00',
  'verification_mode': 'local_python_only',
  'used_code_interpreter_tool': False,
  'request_path': '/Users/vytaszacharovas/CodexProjects/MATH_AUDIT/examples/Draft__Asymptotics_of_the_Stirling_numbers/elem-stir2-2026-hk-version_audit/requests/chunk_048_20260427T0908306706340000_

In [26]:

resp = client.responses.retrieve("resp_0a3e4d0362962ba60069ef278ed08c81949009d2c7fbb05431")

print("status:", getattr(resp, "status", None))
print("error:", getattr(resp, "error", None))
print("incomplete_details:", getattr(resp, "incomplete_details", None))

status: queued
error: None
incomplete_details: None


In [27]:
info = get_audit_status(PDF_PATH, include_manifest=True)

info["status"], info["session"]["pending"]

({'status': 'running',
  'progress_pct': 42.0,
  'current_chunk_id': 'chunk_048',
  'chunks_completed': 47,
  'chunks_total': 84,
  'estimated_pages_completed': 22,
  'estimated_pages_total': 51,
  'cost_usd': 14.001818999999998,
  'updated_at': '2026-04-27T09:08:33.348011+00:00',
  'total_audit_seconds': 13053.123606000001,
  'current_chunk_elapsed_seconds': 0.0,
  'audit_started_at': '2026-04-27T05:30:55.537298+00:00',
  'audit_finished_at': None,
  'pause_requested': False,
  'pause_requested_at': None},
 {'chunk_id': 'chunk_048',
  'response_id': 'resp_0a3e4d0362962ba60069ef278ed08c81949009d2c7fbb05431',
  'created_at': '2026-04-27T09:08:30.670583+00:00',
  'started_at': '2026-04-27T09:08:30.670583+00:00',
  'verification_mode': 'local_python_only',
  'used_code_interpreter_tool': False,
  'request_path': '/Users/vytaszacharovas/CodexProjects/MATH_AUDIT/examples/Draft__Asymptotics_of_the_Stirling_numbers/elem-stir2-2026-hk-version_audit/requests/chunk_048_20260427T0908306706340000_